# Data Extraction Pipeline (Stages 1–4)
This notebook extracts workflow metadata, run-level metrics, step telemetry (TTFTS), and workload signatures (including artifact-based executed-test evidence).

## Stage 1 — Verify workflows and label execution-style evidence

In [10]:
# ============================================================
# Stage 1 (REWIRED): Fetch workflows from GitHub for URL_List repos,
# follow called scripts/local actions, and emit verified_workflows_v16.csv
#
# CHANGES ONLY (per request):
# 1) FIX missed Gradle connected*AndroidTest when tasks include GHA expressions
#    => sanitize ${{ ... }} expressions before regex matching.
# 2) ADD a column for step name(s) that include the test invocation
#    => test_invocation_step_names
# 3) Flutter integration tests are labeled "Flutter Integration Test" instead of "3P-CLI"
#    => flutter_project_hint dropped
# 4) ADD Detox Android E2E detection
#    => invocation_types includes "Detox"
#    => looks_like_instru=yes only when Detox + Android runtime/style evidence is present
#    => test_invocation_step_names captures the step with yarn/npx detox test
# 5) FIX Python DeprecationWarning ("Flags not at the start...")
#    => no inline flags inside shared regex fragments
# 6) ADD workflow-structure anchor position proxies (NEW)
#    => anchor_job_ordinal
#    => anchor_step_ordinal_in_job
#    (based on declared YAML order of the first detected invocation step)
#
# 7) Add STRICT emulator.wtf "indirect invocation" detection (already in your latest)
# 8) NEW (ONLY): Add STRICT BrowserStack "indirect invocation" detection
#    - Requires BrowserStack signal AND credible execution trigger (API/script/CLI/gradle task)
#    - Prevents FP from mere env/setup mentions
#
# 9) NEW (ONLY): Persist called-file instrumentation evidence for downstream stages
#    => called_instru_signal
#    => called_instru_file_paths
#    => called_instru_origin_refs
#    => called_instru_origin_step_names
#    => called_instru_file_types
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union
from urllib.parse import urlparse

import requests

try:
    import yaml  # PyYAML (not required)
except Exception:
    yaml = None

# =========================
# CONFIG (KEEP THESE AS YOUR STAGE-1 CONTRACT)
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_URL_LIST_CSV = ROOT_DIR / "URL_List.csv"               # input list of repos
OUT_STAGE1_CSV  = ROOT_DIR / "verified_workflows_v16.csv" # Stage-1 output name (original)

MAX_TOKENS_TO_USE = 7

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

# follow local files referenced by workflow:
FOLLOW_CALLED_FILES = True
MAX_FOLLOW_DEPTH = 2
MAX_FOLLOW_BYTES = 1_500_000  # skip huge files

# =========================
# Helpers
# =========================
BOM = "\ufeff"
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")  # sanitize expressions like ${{ matrix.flavor }}

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def _clean_key(k: str) -> str:
    return (k or "").replace(BOM, "").strip()

def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Input CSV not found: {path}")
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            clean_row = {}
            for k, v in r.items():
                ck = _clean_key(k)
                clean_row[ck] = (v or "")
            rows.append(clean_row)
    return rows, fields

def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        x = (x or "").strip()
        if not x:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def safe_join(items: List[str], max_len: int = 1500) -> str:
    s = ",".join(unique_preserve(items))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."

def safe_join_pipe(items: List[str], max_len: int = 3000) -> str:
    s = "|".join(unique_preserve(items))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."

def parse_repo_full_name(url: str) -> str:
    u = (url or "").strip()
    if not u:
        return ""
    if re.match(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$", u):
        return u
    if u.startswith("git@github.com:"):
        u2 = u.split("git@github.com:", 1)[1]
        u2 = u2[:-4] if u2.endswith(".git") else u2
        return u2.strip("/")
    if "github.com" in u:
        try:
            p = urlparse(u)
            parts = [x for x in (p.path or "").split("/") if x]
            if len(parts) >= 2:
                return f"{parts[0]}/{parts[1].replace('.git','')}"
        except Exception:
            return ""
    return ""

def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

def sanitize_gha_expr(text: str) -> str:
    # "connected${{ matrix.flavor }}DebugAndroidTest" -> "connectedDebugAndroidTest"
    return GHA_EXPR_RE.sub("", text or "")

def normalize_repo_rel_path(ref: str) -> str:
    rr = (ref or "").replace("\\", "/").strip()
    rr = rr[2:] if rr.startswith("./") else rr
    rr = rr.lstrip("/")
    while "//" in rr:
        rr = rr.replace("//", "/")
    return rr

# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "stage1-v16-workflow-scan/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1

# =========================
# GitHub endpoints
# =========================
def get_repo_meta(gh: GitHubClient, full_name: str) -> Dict[str, str]:
    url = f"https://api.github.com/repos/{full_name}"
    data = gh.request_json("GET", url, params={})
    if not isinstance(data, dict):
        return {}
    return {
        "default_branch": (data.get("default_branch") or "").strip(),
        "archived": str(bool(data.get("archived"))).lower(),
        "private": str(bool(data.get("private"))).lower(),
    }

def list_workflows(gh: GitHubClient, full_name: str) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows"
    data = gh.request_json("GET", url, params={"per_page": 100})
    if not isinstance(data, dict):
        return []
    wfs = data.get("workflows", [])
    return wfs if isinstance(wfs, list) else []

def fetch_file_text_at_ref(gh: GitHubClient, full_name: str, path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref} if ref else {})
    if not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        try:
            r = gh.session.get(dl, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            if r.status_code == 200:
                return r.text or ""
        except requests.exceptions.RequestException:
            return ""
    return ""

def file_size_at_ref(gh: GitHubClient, full_name: str, path: str, ref: str) -> Optional[int]:
    url = f"https://api.github.com/repos/{full_name}/contents/{path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref} if ref else {})
    if not isinstance(data, dict):
        return None
    sz = data.get("size")
    try:
        return int(sz)
    except Exception:
        return None

# =========================
# Signal detection patterns (Stage-1)
# =========================
GRADLE_INVOKE_PREFIX = r"(?:^|[ \t\r\n;&|()\"'`])"

GRADLE_CMD_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradlew\.bat\b|gradle\s+)",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# Flutter integration tests (hint + android-targeted)
FLUTTER_IT_RE = re.compile(r"\bflutter\s+(?:test|drive)\b", flags=re.IGNORECASE | re.DOTALL)
FLUTTER_IT_ANDROID_HINT_RE = re.compile(r"\b(integration_test|--driver\b|test_driver)\b", flags=re.IGNORECASE | re.DOTALL)
FLUTTER_DEVICE_FLAG_RE = re.compile(r"\s+-d\s+(?P<dev>\"[^\"]+\"|'[^']+'|\S+)", flags=re.IGNORECASE | re.DOTALL)
FLUTTER_DEVICE_IS_ANDROID_RE = re.compile(
    r"\b(android|emulator-\d+|sdk\s+gphone|android\s+sdk\s+built\s+for|pixel)\b",
    flags=re.IGNORECASE | re.DOTALL,
)

# Detox Android E2E invocation (CLI via yarn/npm/pnpm/npx)
DETOX_INVOKE_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}("
    r"(?:yarn|npm|pnpm)\s+[^\n\r]*\bdetox(?::[a-z0-9:_-]+)?\b|"
    r"(?:npx\s+detox\s+test\b)|"
    r"(?:detox\s+test\b)"
    r")",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# --- GMD tasks ---
GMD_TASK_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"manageddevice[\w:-]*(check|androidtest|test|setup)\b|"
    r":[\w:-]*manageddevice[\w:-]*(check|androidtest|test|setup)\b"
    r")",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

GMD_MANAGEDDEV_PROP_RE = re.compile(
    r"\B-Pandroid\.(?:testoptions\.manageddevices|experimental\.testOptions\.managedDevices)\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

GENERIC_ANDROIDTEST_TASK_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b(?!connected)\w+androidtest\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

CONNECTED_ANDROIDTEST_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"connected\w*androidtest|connectedcheck|devicecheck|alldevicescheck|"
    r"device\w*androidtest"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

BASELINE_PROFILE_TASK_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"generate\w*baselineprofile|collect\w*baselineprofile|baselineprofile"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

ADB_INSTR_RE = re.compile(r"\badb\s+shell\s+am\s+instrument\b|\bam\s+instrument\b", flags=re.IGNORECASE | re.DOTALL)

EMU_COMMUNITY_ACTION_RE = re.compile(
    r"\b("
    r"reactivecircus/android-emulator-runner|"
    r"malinskiy/action-android/emulator-run-cmd|"
    r"android-emulator-runner"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

EMU_CUSTOM_RUNTIME_RE = re.compile(
    r"\b("
    r"\bemulator\b.*\b-avd\b|"
    r"\bavdmanager\b|"
    r"adb\s+wait[- ]?for[- ]?device"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

REAL_DEVICE_ADB_RE = re.compile(
    r"\badb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b",
    flags=re.IGNORECASE | re.MULTILINE,
)

THIRD_PARTY_PROVIDER_NAME_RE = re.compile(
    r"\b("
    r"firebase\s+test\s+lab|gcloud\s+firebase|"
    r"browserstack|bstack|hub\.browserstack\.com|"
    r"sauce(labs)?|saucectl|"
    r"appcenter|microsoft/appcenter|"
    r"emulator\.wtf|"
    r"maestro\s+cloud"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

THIRD_PARTY_INVOKE_RE = re.compile(
    r"\b("
    r"(gcloud\s+firebase\s+test\s+android\s+run\b)|"
    r"(firebase\s+test\s+android\s+run\b)|"
    r"(flank\s+android\s+run\b)|"
    r"(appcenter\s+test\s+run\s+android\b)|"
    r"(appcenter\s+test\s+run\s+espresso\b)|"
    r"(saucectl\s+(run|test)\b)|"
    r"(emulator-wtf/run-tests@)|"
    r"(maestro\s+cloud\b)"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

THIRD_PARTY_SETUP_ONLY_RE = re.compile(
    r"\b("
    r"google-github-actions/(auth|setup-gcloud)|"
    r"gcloud\s+auth|"
    r"gcloud\s+config\s+set"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# =========================
# STRICT indirect 3P invocation detectors
# =========================

# --- emulator.wtf (already added previously) ---
EMULATOR_WTF_SIGNAL_RE = re.compile(
    r"\b("
    r"emulator\.wtf|"
    r"emulator_wtf|"
    r"ew_api_token|"
    r"emulatorwtf_token|"
    r"emulator_wtf_token"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

EMULATOR_WTF_GRADLE_TASK_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\b("
    r"(?:[:\w.-]+)?emulatorwtf(?:[:\w.-]+)?"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

def _is_emulator_wtf_indirect_invoke(text: str) -> bool:
    low = sanitize_gha_expr(text or "").lower()
    return bool(EMULATOR_WTF_SIGNAL_RE.search(low)) and bool(EMULATOR_WTF_GRADLE_TASK_RE.search(low))

# --- NEW: BrowserStack strict indirect invoke ---
# Signal: domain, bs:// app ids, common secrets/envs, local tunnel, sdk token
BROWSERSTACK_SIGNAL_RE = re.compile(
    r"\b("
    r"api-cloud\.browserstack\.com|"
    r"hub\.browserstack\.com|"
    r"browserstack(local)?|"
    r"\bbs://|"
    r"browserstack_username|browserstack_access(_)?key|"
    r"bstack(_)?(username|access(_)?key)|"
    r"browserstack-sdk"
    r")\b",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

# Execution trigger: API call, bstack CLI, explicit scripts, or gradle tasks containing browserstack
BROWSERSTACK_EXEC_TRIGGER_RE = re.compile(
    rf"{GRADLE_INVOKE_PREFIX}("
    r"curl\s+[^\n\r]*(api-cloud\.browserstack\.com|hub\.browserstack\.com)|"
    r"(?:python|python3)\s+[^\n\r]*browserstack[^\s]*\.(?:py)\b|"
    r"node\s+[^\n\r]*browserstack[^\s]*\.(?:js|mjs|cjs)\b|"
    r"(?:yarn|npm|pnpm)\s+[^\n\r]*\bbrowserstack\b|"
    r"\bbstack\b\s+[^\n\r]*(?:run|execute|test|app-automate|appautomate|espresso|xcuitest|appium)\b|"
    r"(\./gradlew\b|gradle\s+|gradlew\.bat\b)[^\n\r]*\bbrowserstack\b"
    r")",
    flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
)

def _is_browserstack_indirect_invoke(text: str) -> bool:
    low = sanitize_gha_expr(text or "").lower()
    return bool(BROWSERSTACK_SIGNAL_RE.search(low)) and bool(BROWSERSTACK_EXEC_TRIGGER_RE.search(low))

# =========================
# Follow-called-files extraction (UNCHANGED)
# =========================
LOCAL_USES_RE = re.compile(r'(?mi)^\s*uses\s*:\s*(?P<ref>\./\S+?)(?:\s+#.*)?$')
WORKDIR_RE = re.compile(r'(?mi)^\s*working-directory\s*:\s*(?P<wd>[^\n#]+)')

SCRIPT_CALL_RE = re.compile(r'''(?mix)
(?:^|[;&|()\s"'`])
(?:(?:bash|sh|pwsh|powershell|python|python3|node|ruby)\s+)?
(?P<path>(?:\./|\.\\)?[\w./\\-]+\.(?:sh|ps1|bat|cmd|py|js|rb|pl))
(?:\s|$)
''')

GENERIC_REL_EXEC_RE = re.compile(r'(?m)(?:^|[;&|()\s"\'`])(?P<path>\./[A-Za-z0-9_./\\-]+)(?:\s|$)')
CONFIG_ARG_RE = re.compile(r'(?mi)\b--config(?:=|\s+)(?P<path>[^\s"\']+)')

NO_FOLLOW_BASENAMES = {"gradlew", "gradlew.bat", "gradle", "adb", "flutter", "gcloud", "java", "python", "python3"}

def _strip_quotes(s: str) -> str:
    return (s or "").strip().strip('"').strip("'").strip("`")

def is_dynamic_ref(ref: str) -> bool:
    r = ref or ""
    return ("${{" in r) or ("${" in r) or ("$(" in r) or ("%{" in r)

def extract_workdirs(text: str) -> List[str]:
    wds = []
    for m in WORKDIR_RE.finditer(text or ""):
        wd = _strip_quotes(m.group("wd"))
        if wd:
            wd = wd.replace("\\", "/").lstrip("./")
            wds.append(wd)
    return unique_preserve(wds)

def extract_references(text: str) -> List[str]:
    refs: List[str] = []

    for m in LOCAL_USES_RE.finditer(text or ""):
        ref = _strip_quotes(m.group("ref"))
        if "@" in ref:
            ref = ref.split("@", 1)[0]
        refs.append(ref)

    for m in SCRIPT_CALL_RE.finditer(text or ""):
        refs.append(_strip_quotes(m.group("path")))

    for m in CONFIG_ARG_RE.finditer(text or ""):
        refs.append(_strip_quotes(m.group("path")))

    for m in GENERIC_REL_EXEC_RE.finditer(text or ""):
        p = _strip_quotes(m.group("path"))
        base = Path(p.replace("\\", "/")).name.lower()
        if base in NO_FOLLOW_BASENAMES:
            continue
        refs.append(p)

    out = []
    for r in refs:
        if not r:
            continue
        out.append(r.replace("\\", "/").strip())
    return unique_preserve(out)

def normalize_ref_path(ref: str) -> str:
    rr = (ref or "").replace("\\", "/").strip()
    rr = rr[2:] if rr.startswith("./") else rr
    rr = rr.lstrip("/")
    return rr

def candidate_paths_for_ref(ref: str, workdirs: List[str]) -> List[str]:
    rr = normalize_ref_path(ref)
    prefixes = [""] + [wd.strip("/").replace("\\", "/") for wd in (workdirs or []) if wd.strip()]
    out = []
    for pref in prefixes:
        p = f"{pref}/{rr}" if pref else rr
        out.append(p.strip("/"))
    return unique_preserve(out)

def possible_action_ymls(path: str) -> List[str]:
    p = path.strip("/")
    return unique_preserve([f"{p}/action.yml", f"{p}/action.yaml"])

def classify_called_file_type(path: str) -> str:
    low = (path or "").lower().replace("\\", "/")
    if low.endswith("action.yml") or low.endswith("action.yaml"):
        return "local_action"
    if "/.github/workflows/" in f"/{low}":
        return "local_workflow"
    return "script"

# =========================
# Step parsing for invocation step names + anchor ordinals (UNCHANGED)
# =========================
STEP_NAME_LINE_RE = re.compile(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", re.MULTILINE)

def _count_leading_spaces(s: str) -> int:
    return len(s) - len(s.lstrip(" "))

def parse_workflow_step_records(yaml_text: str) -> List[Dict[str, Union[str, int]]]:
    if not yaml_text:
        return []

    lines = yaml_text.splitlines()
    n = len(lines)
    out: List[Dict[str, Union[str, int]]] = []

    jobs_idx = None
    for i, line in enumerate(lines):
        if re.match(r"^\s*jobs\s*:\s*$", line):
            jobs_idx = i
            break
    if jobs_idx is None:
        return out

    jobs_indent = _count_leading_spaces(lines[jobs_idx])
    i = jobs_idx + 1
    job_ordinal = 0

    while i < n:
        line = lines[i]
        if line.strip() == "":
            i += 1
            continue

        indent = _count_leading_spaces(line)
        if indent <= jobs_indent:
            break

        m_job = re.match(r"^\s*([A-Za-z0-9_.-]+)\s*:\s*$", line)
        if not m_job or indent != jobs_indent + 2:
            i += 1
            continue

        job_id = m_job.group(1).strip()
        job_ordinal += 1

        block_start = i + 1
        j = block_start
        while j < n:
            nxt = lines[j]
            if nxt.strip() == "":
                j += 1
                continue
            nxt_indent = _count_leading_spaces(nxt)
            if nxt_indent <= indent:
                break
            j += 1
        job_block_lines = lines[block_start:j]
        job_block = "\n".join(job_block_lines)

        m_name = re.search(r"(?mi)^\s*name\s*:\s*(.+?)\s*$", job_block)
        job_name = m_name.group(1).strip().strip('"').strip("'") if m_name else ""

        job_lines = job_block_lines
        steps_idx = None
        steps_indent = None
        for k, jl in enumerate(job_lines):
            if re.match(r"^\s*steps\s*:\s*$", jl):
                steps_idx = k
                steps_indent = _count_leading_spaces(jl)
                break

        if steps_idx is not None and steps_indent is not None:
            step_ordinal = 0
            k = steps_idx + 1
            while k < len(job_lines):
                cur = job_lines[k]
                if cur.strip() == "":
                    k += 1
                    continue
                cur_indent = _count_leading_spaces(cur)
                if cur_indent <= steps_indent:
                    break

                m_step = re.match(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", cur)
                if not m_step:
                    k += 1
                    continue

                base_indent = len(m_step.group(1))
                step_name = m_step.group(2).strip().strip('"').strip("'")
                block = [cur]
                kk = k + 1
                while kk < len(job_lines):
                    nxt = job_lines[kk]
                    m2 = re.match(r"^(\s*)-\s*name\s*:\s*(.+?)\s*$", nxt)
                    if m2 and len(m2.group(1)) == base_indent:
                        break
                    if nxt.strip() and _count_leading_spaces(nxt) <= steps_indent:
                        break
                    block.append(nxt)
                    kk += 1

                step_ordinal += 1
                out.append({
                    "job_id": job_id,
                    "job_name": job_name,
                    "job_ordinal": job_ordinal,
                    "step_name": step_name,
                    "step_ordinal_in_job": step_ordinal,
                    "step_block": "\n".join(block),
                })

                k = kk
        i = j

    return out

def build_origin_ref_to_step_names(workflow_yaml_text: str) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    for rec in parse_workflow_step_records(workflow_yaml_text):
        step_name = str(rec.get("step_name") or "").strip()
        step_block = str(rec.get("step_block") or "")
        refs = extract_references(step_block)
        for r in refs:
            rr = normalize_repo_rel_path(r)
            if not rr:
                continue
            out.setdefault(rr, [])
            out[rr] = unique_preserve(out[rr] + [step_name])
    return out

def _flutter_androidish_from_text(text: str, runtime_ev: Dict[str, bool]) -> bool:
    t = text or ""
    t2 = sanitize_gha_expr(t)
    low = t2.lower()

    if not (FLUTTER_IT_RE.search(low) and FLUTTER_IT_ANDROID_HINT_RE.search(low)):
        return False

    targeted = False
    for m in FLUTTER_DEVICE_FLAG_RE.finditer(low):
        dev = (m.group("dev") or "").strip().strip('"').strip("'").lower()
        if FLUTTER_DEVICE_IS_ANDROID_RE.search(dev):
            targeted = True
            break
    if FLUTTER_DEVICE_IS_ANDROID_RE.search(low):
        targeted = True

    if targeted:
        return True

    if runtime_ev.get("emu_comm") or runtime_ev.get("emu_custom") or runtime_ev.get("real_device") or runtime_ev.get("third_party_invoke"):
        return True

    return False

def _detox_androidish_from_text(text: str, runtime_ev: Dict[str, bool]) -> bool:
    t = text or ""
    t2 = sanitize_gha_expr(t)
    low = t2.lower()

    if not DETOX_INVOKE_RE.search(low):
        return False

    if runtime_ev.get("emu_comm") or runtime_ev.get("emu_custom") or runtime_ev.get("real_device") or runtime_ev.get("third_party_invoke"):
        return True

    return False

def _is_third_party_invoke_non_flutter(text: str) -> bool:
    low = sanitize_gha_expr(text or "").lower()
    return bool(THIRD_PARTY_INVOKE_RE.search(low)) and not bool(
        THIRD_PARTY_SETUP_ONLY_RE.search(low) and not THIRD_PARTY_INVOKE_RE.search(low)
    )

def extract_test_invocation_step_names_and_anchor(
    gh: GitHubClient,
    full_name: str,
    base_ref: str,
    workflow_yaml_text: str,
) -> Tuple[List[str], Optional[int], Optional[int]]:
    if not workflow_yaml_text:
        return [], None, None

    step_records = parse_workflow_step_records(workflow_yaml_text)
    if not step_records:
        return [], None, None

    step_names: List[str] = []
    anchor_job_ordinal: Optional[int] = None
    anchor_step_ordinal_in_job: Optional[int] = None

    ev_full = scan_text_for_evidence(workflow_yaml_text)
    runtime_ev = {
        "emu_comm": bool(ev_full.get("emu_comm")),
        "emu_custom": bool(ev_full.get("emu_custom")),
        "real_device": bool(ev_full.get("real_device")),
        "third_party_invoke": bool(ev_full.get("third_party_invoke")),
    }

    for rec in step_records:
        step_name = str(rec.get("step_name") or "")
        blk = str(rec.get("step_block") or "")
        blk_s = sanitize_gha_expr(blk)
        low = blk_s.lower()

        direct_connected = bool(CONNECTED_ANDROIDTEST_RE.search(low))
        direct_gmd_task = bool(GMD_TASK_RE.search(low))
        direct_gmd_prop = bool(GMD_MANAGEDDEV_PROP_RE.search(low))
        direct_generic_androidtest = bool(GENERIC_ANDROIDTEST_TASK_RE.search(low))
        direct_baseline = bool(BASELINE_PROFILE_TASK_RE.search(low))
        direct_adb = bool(ADB_INSTR_RE.search(low))

        direct_3p = _is_third_party_invoke_non_flutter(low)
        direct_emu_wtf_indirect = _is_emulator_wtf_indirect_invoke(low)
        direct_bs_indirect = _is_browserstack_indirect_invoke(low)

        direct_flutter_androidish = _flutter_androidish_from_text(low, runtime_ev)
        direct_detox_androidish = _detox_androidish_from_text(low, runtime_ev)

        direct_gmd = bool(direct_gmd_task or (direct_gmd_prop and (direct_baseline or direct_generic_androidtest)))

        matched_here = False
        if (
            direct_connected
            or direct_gmd
            or direct_baseline
            or direct_adb
            or direct_3p
            or direct_emu_wtf_indirect
            or direct_bs_indirect
            or direct_flutter_androidish
            or direct_detox_androidish
        ):
            matched_here = True
        else:
            refs = extract_references(blk)
            wds = extract_workdirs(blk)

            for r in refs:
                if not r or is_dynamic_ref(r):
                    continue
                candidates = candidate_paths_for_ref(r, wds)
                is_prob_action = bool(r.strip().startswith("./")) and (r.endswith("/") or "/" in r)

                found_invoke_in_called = False

                for c in candidates:
                    if is_prob_action and (not c.lower().endswith((".yml", ".yaml", ".sh", ".ps1", ".py", ".js", ".rb", ".pl", ".bat", ".cmd"))):
                        for ay in possible_action_ymls(c):
                            txt = fetch_file_text_at_ref(gh, full_name, ay, base_ref)
                            if not txt:
                                continue
                            ev = scan_text_for_evidence(txt)
                            if compute_looks_like_instru(ev) == "yes":
                                found_invoke_in_called = True
                                break
                        if found_invoke_in_called:
                            break

                    txt = fetch_file_text_at_ref(gh, full_name, c, base_ref)
                    if not txt:
                        continue
                    ev = scan_text_for_evidence(txt)
                    if compute_looks_like_instru(ev) == "yes":
                        found_invoke_in_called = True
                        break

                if found_invoke_in_called:
                    matched_here = True
                    break

        if matched_here:
            step_names.append(step_name)
            if anchor_job_ordinal is None:
                try:
                    anchor_job_ordinal = int(rec.get("job_ordinal"))  # type: ignore[arg-type]
                except Exception:
                    anchor_job_ordinal = None
                try:
                    anchor_step_ordinal_in_job = int(rec.get("step_ordinal_in_job"))  # type: ignore[arg-type]
                except Exception:
                    anchor_step_ordinal_in_job = None

    return unique_preserve(step_names), anchor_job_ordinal, anchor_step_ordinal_in_job

# =========================
# Scan logic
# =========================
def detect_provider_names(text: str) -> List[str]:
    t = (text or "").lower()
    names = []
    if re.search(r"\b(gcloud\s+firebase|firebase\s+test\s+lab|firebase\s+test\s+android\s+run)\b", t):
        names.append("Firebase Test Lab")
    if re.search(r"\bbrowserstack|bstack|hub\.browserstack\.com\b", t):
        names.append("BrowserStack")
    if re.search(r"\bsauce(labs)?|saucectl\b", t):
        names.append("Sauce Labs")
    if re.search(r"\b(appcenter|microsoft/appcenter)\b", t):
        names.append("App Center")
    if re.search(r"\bemulator\.wtf|emulator-wtf/run-tests@\b", t):
        names.append("emulator.wtf")
    if re.search(r"\bmaestro\s+cloud\b", t):
        names.append("Maestro Cloud")
    return unique_preserve(names)

def scan_text_for_evidence(text: str) -> Dict[str, Union[bool, List[str]]]:
    txt = sanitize_gha_expr(text or "")
    low = txt.lower()

    has_gradle = bool(GRADLE_CMD_RE.search(low))

    baseline = bool(BASELINE_PROFILE_TASK_RE.search(low))
    adb = bool(ADB_INSTR_RE.search(low))

    gmd_prop = bool(GMD_MANAGEDDEV_PROP_RE.search(low))
    generic_androidtest = bool(GENERIC_ANDROIDTEST_TASK_RE.search(low))

    gmd_task = bool(GMD_TASK_RE.search(low))
    gmd = bool(gmd_task or (gmd_prop and (baseline or generic_androidtest)))

    connected = bool(CONNECTED_ANDROIDTEST_RE.search(low))

    emu_comm = bool(EMU_COMMUNITY_ACTION_RE.search(low))
    emu_custom = bool(EMU_CUSTOM_RUNTIME_RE.search(low))
    real_device = bool(REAL_DEVICE_ADB_RE.search(low))

    # Direct 3P invoke
    tp_invoke_direct = bool(THIRD_PARTY_INVOKE_RE.search(low)) and not bool(
        THIRD_PARTY_SETUP_ONLY_RE.search(low) and not THIRD_PARTY_INVOKE_RE.search(low)
    )

    # Strict indirect invokes
    tp_invoke_emu_wtf_indirect = _is_emulator_wtf_indirect_invoke(low)
    tp_invoke_bs_indirect = _is_browserstack_indirect_invoke(low)

    tp_invoke = bool(tp_invoke_direct or tp_invoke_emu_wtf_indirect or tp_invoke_bs_indirect)

    tp_providers = detect_provider_names(low) if THIRD_PARTY_PROVIDER_NAME_RE.search(low) else []

    runtime_ev = {
        "emu_comm": emu_comm,
        "emu_custom": emu_custom,
        "real_device": real_device,
        "third_party_invoke": tp_invoke,
    }

    flutter_androidish = _flutter_androidish_from_text(txt, runtime_ev)
    detox_androidish = _detox_androidish_from_text(txt, runtime_ev)

    return {
        "has_gradle": has_gradle,
        "gmd": gmd,
        "connected": connected,
        "baseline": baseline,
        "adb": adb,
        "emu_comm": emu_comm,
        "emu_custom": emu_custom,
        "real_device": real_device,
        "third_party_invoke": tp_invoke,
        "third_party_providers": tp_providers,
        "flutter_androidish_invoke": flutter_androidish,
        "detox_androidish_invoke": detox_androidish,
    }

def merge_evidence(a: Dict, b: Dict) -> Dict:
    out = dict(a)
    for k, v in b.items():
        if isinstance(v, bool):
            out[k] = bool(out.get(k, False) or v)
        elif isinstance(v, list):
            out[k] = unique_preserve((out.get(k, []) or []) + v)
        else:
            out[k] = v
    return out

def compute_invocation_types(ev: Dict) -> List[str]:
    inv: List[str] = []

    if ev.get("detox_androidish_invoke"):
        inv.append("Detox")

    if ev.get("flutter_androidish_invoke"):
        inv.append("Flutter Integration Test")
    else:
        if ev.get("third_party_invoke"):
            inv.append("3P-CLI")

    if ev.get("adb"):
        inv.append("ADB")
    if ev.get("gmd"):
        inv.append("Gradle_GMD")
    if ev.get("connected"):
        inv.append("Gradle_Connected")
    if ev.get("baseline"):
        inv.append("Gradle_BaselineProfile")
    if ev.get("has_gradle") and (ev.get("gmd") or ev.get("connected") or ev.get("baseline")):
        inv.append("Gradle")

    return sorted(set(inv))

def compute_styles(ev: Dict) -> List[str]:
    styles: List[str] = []
    if ev.get("third_party_invoke"):
        styles.append("Third-Party")
    if ev.get("gmd"):
        styles.append("GMD")
    if ev.get("real_device"):
        styles.append("Real-Device")

    if ev.get("emu_comm"):
        styles.append("Community")
    else:
        if ev.get("emu_custom"):
            styles.append("Custom")

    return sorted(set(styles))

def compute_looks_like_instru(ev: Dict) -> str:
    if (
        ev.get("gmd")
        or ev.get("connected")
        or ev.get("baseline")
        or ev.get("adb")
        or ev.get("third_party_invoke")
        or ev.get("flutter_androidish_invoke")
        or ev.get("detox_androidish_invoke")
    ):
        return "yes"
    return "no"

def infer_instru_detect_method(styles: List[str], inv: List[str]) -> str:
    if "Third-Party" in styles:
        return "third_party_cli"
    if "GMD" in styles:
        return "gradle_gmd"
    if "Community" in styles or "Custom" in styles:
        if any(x in inv for x in ["Gradle_Connected", "Gradle"]):
            return "gradle_connected"
        if "Detox" in inv:
            return "invocation_signal"
    if "Real-Device" in styles:
        return "real_device_adb"
    if inv:
        return "invocation_signal"
    return "none"

# =========================
# Called-file following via GitHub API (ADJUSTED ONLY TO PERSIST CALLED-FILE INSTRU EVIDENCE)
# =========================
def follow_called_files(
    gh: GitHubClient,
    full_name: str,
    base_ref: str,
    root_text: str,
    origin_ref_to_step_names: Optional[Dict[str, List[str]]] = None,
    max_depth: int = MAX_FOLLOW_DEPTH,
) -> Tuple[Dict, int, int, List[str], bool, List[str], List[str], List[str], List[str]]:
    if not FOLLOW_CALLED_FILES:
        return scan_text_for_evidence(root_text), 0, 0, [], False, [], [], [], []

    agg_evidence = scan_text_for_evidence(root_text)
    unresolved_dynamic = 0
    followed_paths: List[str] = []
    visited: Set[str] = set()

    called_instru_signal = False
    called_instru_file_paths: List[str] = []
    called_instru_origin_refs: List[str] = []
    called_instru_origin_step_names: List[str] = []
    called_instru_file_types: List[str] = []

    origin_ref_to_step_names = origin_ref_to_step_names or {}

    def fetch_and_scan(path: str) -> Optional[Tuple[str, Dict]]:
        sz = file_size_at_ref(gh, full_name, path, base_ref)
        if sz is not None and sz > MAX_FOLLOW_BYTES:
            return None
        txt = fetch_file_text_at_ref(gh, full_name, path, base_ref)
        if not txt:
            return None
        ev = scan_text_for_evidence(txt)
        return txt, ev

    def register_called_instru(path: str, origin_ref: str) -> None:
        nonlocal called_instru_signal, called_instru_file_paths, called_instru_origin_refs, called_instru_origin_step_names, called_instru_file_types
        called_instru_signal = True
        norm_path = normalize_repo_rel_path(path)
        norm_origin = normalize_repo_rel_path(origin_ref)
        called_instru_file_paths.append(norm_path)
        called_instru_origin_refs.append(norm_origin)
        called_instru_file_types.append(classify_called_file_type(norm_path))
        called_instru_origin_step_names.extend(origin_ref_to_step_names.get(norm_origin, []))

    def walk(text: str, depth: int) -> None:
        nonlocal agg_evidence, unresolved_dynamic, followed_paths, visited
        if depth > max_depth:
            return

        refs = extract_references(text)
        wds = extract_workdirs(text)

        for r in refs:
            if not r:
                continue
            if is_dynamic_ref(r):
                unresolved_dynamic += 1
                continue

            norm_origin_ref = normalize_repo_rel_path(r)
            candidates = candidate_paths_for_ref(r, wds)
            is_prob_action = bool(r.strip().startswith("./")) and (r.endswith("/") or "/" in r)

            for c in candidates:
                if c in visited:
                    continue

                if is_prob_action and (not c.lower().endswith((".yml", ".yaml", ".sh", ".ps1", ".py", ".js", ".rb", ".pl", ".bat", ".cmd"))):
                    for ay in possible_action_ymls(c):
                        if ay in visited:
                            continue
                        got = fetch_and_scan(ay)
                        if got:
                            visited.add(ay)
                            followed_paths.append(ay)
                            txt2, ev2 = got
                            agg_evidence = merge_evidence(agg_evidence, ev2)
                            if compute_looks_like_instru(ev2) == "yes":
                                register_called_instru(ay, norm_origin_ref)
                            walk(txt2, depth + 1)

                got = fetch_and_scan(c)
                if got:
                    visited.add(c)
                    followed_paths.append(c)
                    txt2, ev2 = got
                    agg_evidence = merge_evidence(agg_evidence, ev2)
                    if compute_looks_like_instru(ev2) == "yes":
                        register_called_instru(c, norm_origin_ref)
                    walk(txt2, depth + 1)

    walk(root_text, 0)
    return (
        agg_evidence,
        len(unique_preserve(followed_paths)),
        int(unresolved_dynamic),
        unique_preserve(followed_paths),
        bool(called_instru_signal),
        unique_preserve(called_instru_file_paths),
        unique_preserve(called_instru_origin_refs),
        unique_preserve(called_instru_origin_step_names),
        unique_preserve(called_instru_file_types),
    )

# =========================
# Stage-1 processing
# =========================
def build_stage1_rows_for_repo(gh: GitHubClient, full_name: str, repo_url: str) -> List[Dict[str, str]]:
    meta = get_repo_meta(gh, full_name)
    default_branch = meta.get("default_branch") or "main"

    workflows = list_workflows(gh, full_name)
    out_rows: List[Dict[str, str]] = []

    for wf in workflows:
        wf_name = (wf.get("name") or "").strip()
        wf_path = (wf.get("path") or "").strip()
        wf_state = (wf.get("state") or "").strip()
        wf_id = str(wf.get("id") or "")

        if not wf_path:
            continue

        yaml_text = fetch_file_text_at_ref(gh, full_name, wf_path, default_branch)
        if not yaml_text:
            continue

        origin_ref_to_step_names = build_origin_ref_to_step_names(yaml_text)

        (
            ev0,
            followed_count,
            unresolved_dyn,
            followed_paths,
            called_instru_signal,
            called_instru_file_paths,
            called_instru_origin_refs,
            called_instru_origin_step_names,
            called_instru_file_types,
        ) = follow_called_files(
            gh=gh,
            full_name=full_name,
            base_ref=default_branch,
            root_text=yaml_text,
            origin_ref_to_step_names=origin_ref_to_step_names,
            max_depth=MAX_FOLLOW_DEPTH,
        )

        invocation_types = compute_invocation_types(ev0)
        styles = compute_styles(ev0)
        looks_like = compute_looks_like_instru(ev0)

        tp_names = ev0.get("third_party_providers", []) if isinstance(ev0.get("third_party_providers"), list) else []
        tp_name_str = safe_join(tp_names) if ("Third-Party" in styles) else ""

        step_inv_names, anchor_job_ordinal, anchor_step_ordinal_in_job = extract_test_invocation_step_names_and_anchor(
            gh=gh,
            full_name=full_name,
            base_ref=default_branch,
            workflow_yaml_text=yaml_text,
        )

        row = {
            "repo_url": repo_url,
            "full_name": full_name,
            "workflow_id": wf_id,
            "workflow_identifier": wf_name,
            "workflow_path": wf_path,
            "workflow_state": wf_state,
            "styles": ",".join(styles),
            "invocation_types": ",".join(invocation_types),
            "looks_like_instru": looks_like,
            "instru_detect_method": infer_instru_detect_method(styles, invocation_types),
            "third_party_provider_name": tp_name_str,
            "test_invocation_step_names": safe_join(step_inv_names, max_len=1200),
            "anchor_job_ordinal": "" if anchor_job_ordinal is None else str(anchor_job_ordinal),
            "anchor_step_ordinal_in_job": "" if anchor_step_ordinal_in_job is None else str(anchor_step_ordinal_in_job),
            "followed_files_count": str(followed_count),
            "unresolved_dynamic_refs_count": str(unresolved_dyn),
            "followed_paths": safe_join(followed_paths, max_len=1500),

            "called_instru_signal": "True" if called_instru_signal else "False",
            "called_instru_file_paths": safe_join_pipe(called_instru_file_paths, max_len=3000),
            "called_instru_origin_refs": safe_join_pipe(called_instru_origin_refs, max_len=3000),
            "called_instru_origin_step_names": safe_join_pipe(called_instru_origin_step_names, max_len=3000),
            "called_instru_file_types": safe_join_pipe(called_instru_file_types, max_len=1000),

            "stage1_extracted_at_utc": now_utc_iso(),
        }
        out_rows.append(row)

    return out_rows

def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    url_rows, url_fields = read_csv_rows(IN_URL_LIST_CSV)
    if not url_rows:
        raise RuntimeError("URL_List.csv is empty.")

    candidates = ["repo_urls", "repo_url", "url", "repo"]

    def get_url(r: Dict[str, str]) -> str:
        for c in candidates:
            if (r.get(c) or "").strip():
                return (r.get(c) or "").strip()
        if url_fields:
            return (r.get(url_fields[0]) or "").strip()
        return ""

    repo_urls = unique_preserve([get_url(r) for r in url_rows])

    stage1_rows: List[Dict[str, str]] = []

    for u in repo_urls:
        full_name = parse_repo_full_name(u)
        if not full_name:
            continue
        try:
            rows = build_stage1_rows_for_repo(gh, full_name, u)
            stage1_rows.extend(rows)
        except Exception as e:
            stage1_rows.append({
                "repo_url": u,
                "full_name": full_name,
                "workflow_id": "",
                "workflow_identifier": "",
                "workflow_path": "",
                "workflow_state": "",
                "styles": "",
                "invocation_types": "",
                "looks_like_instru": "no",
                "instru_detect_method": "error",
                "third_party_provider_name": "",
                "test_invocation_step_names": "",
                "anchor_job_ordinal": "",
                "anchor_step_ordinal_in_job": "",
                "followed_files_count": "0",
                "unresolved_dynamic_refs_count": "0",
                "followed_paths": "",
                "called_instru_signal": "False",
                "called_instru_file_paths": "",
                "called_instru_origin_refs": "",
                "called_instru_origin_step_names": "",
                "called_instru_file_types": "",
                "stage1_extracted_at_utc": now_utc_iso(),
            })
            print(f"[warn] {full_name}: {e}")

    out_fields = [
        "repo_url",
        "full_name",
        "workflow_id",
        "workflow_identifier",
        "workflow_path",
        "workflow_state",
        "styles",
        "invocation_types",
        "looks_like_instru",
        "instru_detect_method",
        "third_party_provider_name",
        "test_invocation_step_names",
        "anchor_job_ordinal",
        "anchor_step_ordinal_in_job",
        "followed_files_count",
        "unresolved_dynamic_refs_count",
        "followed_paths",
        "called_instru_signal",
        "called_instru_file_paths",
        "called_instru_origin_refs",
        "called_instru_origin_step_names",
        "called_instru_file_types",
        "stage1_extracted_at_utc",
    ]

    write_csv(OUT_STAGE1_CSV, out_fields, stage1_rows)
    print("[done] Stage 1:", OUT_STAGE1_CSV, f"(rows={len(stage1_rows)})")

if __name__ == "__main__":
    main()

[done] Stage 1: C:\Android Mobile App\ICST2026_Ext\verified_workflows_v16.csv (rows=62)


## Stage 2 — Extract run-level metrics and attach style labels

In [11]:
# ============================================================
# Stage 2 V18 (CURRENT STUDY ALIGNED, AUXILIARY COMPLEXITY ENHANCED)
# Durable Layer-1 instrumentation-envelope inventory + run×style inventory
#
# V18 purpose
# ------------------------------------------------------------
# Keep the current study unit unchanged:
#   - Stage 2 = durable Layer-1 run inventory + run×style inventory
#   - Stage 3 = precise Layer-2 measured from step telemetry
#
# New in V18
# ------------------------------------------------------------
# Adds auxiliary fields to better expose and control for:
#   - repeated same-style execution within a run
#   - matrix-expanded jobs
#   - parallel same-style execution
#   - invocation-like multiplicity proxies at Stage 2
#
# These fields are descriptive/supportive only.
# They do NOT change the primary Stage-2 Layer-1 decomposition.
#
# Core study structure
# 1) Layer 1 decomposition  (Stage 2 durable, job-level)
# 2) Layer 2 decomposition  (Stage 3 precise, step-level; revised)
# 3) Auxiliary labeling / complexity fields
#
# ------------------------------------------------------------
# Layer 1 (Stage 2, durable, broad, job-level)
#
#   Run Duration
# = Time to Instrumentation Envelope
# + Instrumentation Job Envelope
# + Post-Instrumentation Tail
#
# where:
#   Time to Instrumentation Envelope
#     = run start -> first instrumentation-related job start
#
#   Instrumentation Job Envelope
#     = first instrumentation-related job start -> last instrumentation-related job end
#
#   Post-Instrumentation Tail
#     = last instrumentation-related job end -> run end
#
# ------------------------------------------------------------
# Layer 2 (Stage 3 onward, revised, precise, step-level)
#
#   Run Duration
# = Pre-Invocation
# + Invocation Execution Window
# + Post-Invocation
#
# where:
#   Pre-Invocation
#     = run start -> matched invocation step start
#
#   Invocation Execution Window
#     = style-relevant execution interval centered on the invocation anchor
#
#   Post-Invocation
#     = instrumentation execution end -> run completion
#
# IMPORTANT ALIGNMENT NOTE
# - Stage 2 does NOT directly measure Layer 2.
# - Stage 2 remains a durable Layer-1 inventory only.
# - However, Stage 2 exposes broad job-level proxy aliases aligned to the
#   revised Layer 2 naming for downstream compatibility:
#
#     pre_invocation_seconds_proxy
#     invocation_execution_window_seconds_proxy
#     post_invocation_seconds_proxy
#
# - These proxies are BROAD JOB-LEVEL approximations, not precise step-level values.
#
# Anchor concept
# - "Anchor job" is retained as the earliest strongest invocation-carrying job
# - It remains a semantic reference point only
# - The Layer-1 middle component is the broader instrumentation envelope,
#   not necessarily the anchor job alone
#
# Outputs
# - run_inventory.csv                (run-level durable inventory)
# - run_inventory_per_style.csv      (run × style durable Layer-1 inventory)
# ============================================================

import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_VERIFIED_WORKFLOWS_CSV = ROOT_DIR / "verified_workflows_v16.csv"

OUT_RUN_INVENTORY_CSV = ROOT_DIR / "run_inventory.csv"
OUT_RUN_PER_STYLE_CSV = ROOT_DIR / "run_inventory_per_style.csv"

DEFAULT_BRANCH_ONLY = True
PROCESS_ONLY_LOOKS_LIKE_INSTRU = True
FETCH_JOBS_FOR_EACH_RUN = True

MAX_RUNS_PER_WORKFLOW: Optional[int] = None
RUN_CREATED_AT_AFTER: Optional[str] = None

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

MAX_TOKENS_TO_USE = 7
SLEEP_BETWEEN_WORKFLOWS_SEC = 0.05


# =========================
# Helpers
# =========================
STYLE_CANONICAL = ["Community", "Custom", "GMD", "Third-Party", "Real-Device"]

STYLE_ALIASES = {
    "community": "Community",
    "custom": "Custom",
    "gmd": "GMD",
    "third party": "Third-Party",
    "third-party": "Third-Party",
    "third_party": "Third-Party",
    "thirdparty": "Third-Party",
    "3p": "Third-Party",
    "real device": "Real-Device",
    "real-device": "Real-Device",
    "real_device": "Real-Device",
    "realdevice": "Real-Device",

    "emu community": "Community",
    "emulator community": "Community",
    "emu_custom": "Custom",
    "emu custom": "Custom",
    "emulator custom": "Custom",
    "real devices": "Real-Device",
    "real-devices": "Real-Device",
    "real_devices": "Real-Device",
    "realdevices": "Real-Device",
}


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def norm(s: Optional[str]) -> str:
    return (s or "").strip()


def low(s: Optional[str]) -> str:
    return norm(s).lower()


def canon_key(s: Optional[str]) -> str:
    x = low(s).replace("_", " ").replace("-", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x


def normalize_style_label(s: Optional[str]) -> str:
    return STYLE_ALIASES.get(canon_key(s), norm(s))


def split_styles(s: Optional[str]) -> List[str]:
    raw = norm(s)
    if not raw:
        return []
    vals = [normalize_style_label(x) for x in re.split(r"[|,;/]+", raw) if norm(x)]
    return unique_preserve([v for v in vals if v in STYLE_CANONICAL])


def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(str(iso).replace("Z", "+00:00"))
    except Exception:
        return None


def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None


def ensure_csv_header(csv_path: Path, fieldnames: List[str]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()


def append_row(csv_path: Path, fieldnames: List[str], row: Dict) -> None:
    with csv_path.open("a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writerow({k: row.get(k, "") for k in fieldnames})


def load_existing_keys(csv_path: Path, key_field: str) -> Set[str]:
    keys: Set[str] = set()
    if not csv_path.exists():
        return keys
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for row in rdr:
            k = norm(row.get(key_field))
            if k:
                keys.add(k)
    return keys


def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        s = norm(str(x))
        if not s:
            continue
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out


def safe_join_names(names: List[str], max_len: int = 800) -> str:
    s = ",".join(unique_preserve(names))
    return s if len(s) <= max_len else s[: max_len - 3] + "..."


def read_env_tokens(path: Path) -> List[str]:
    if not path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {path}")
    toks: List[str] = []
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN") and v:
            toks.append(v)
            if len(toks) >= MAX_TOKENS_TO_USE:
                break
    if not toks:
        raise ValueError(f"No GitHub tokens found in {path}")
    return toks


# =========================
# Name normalization / matching
# =========================
_NORM_WS_RE = re.compile(r"\s+")
_NORM_PUNCT_RE = re.compile(r"[\[\]\(\)\{\}:;|]+")


def normalize_name(s: str) -> str:
    x = (s or "").strip().strip('"').strip("'").lower()
    x = _NORM_PUNCT_RE.sub(" ", x)
    x = _NORM_WS_RE.sub(" ", x).strip()
    return x


def parse_anchor_step_names(csv_value: str) -> List[str]:
    if not csv_value:
        return []
    raw = [p.strip() for p in str(csv_value).split(",")]
    return unique_preserve([r for r in raw if r])


def anchored_step_match(runtime_step_name: str, anchor_names: List[str]) -> bool:
    rn = normalize_name(runtime_step_name)
    if not rn:
        return False
    for a in anchor_names:
        an = normalize_name(a)
        if not an:
            continue
        if rn == an or an in rn or rn in an:
            return True
    return False


# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None


class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "run-inventory-stage2-v18-layer1-envelope-auxiliary/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"

            try:
                resp = self.session.request(method, url, params=params, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass

            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            retry_after = resp.headers.get("Retry-After")
            if resp.status_code in (403, 429) and retry_after:
                try:
                    ra = int(retry_after)
                    time.sleep(min(BACKOFF_CAP_S, max(1, ra)) + random.random())
                    continue
                except Exception:
                    pass

            text_l = low(resp.text or "")
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            try:
                return resp.json()
            except Exception:
                return None

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# =========================
# GitHub endpoints
# =========================
def get_repo_default_branch(gh: GitHubClient, full_name: str) -> str:
    data = gh.request_json("GET", f"https://api.github.com/repos/{full_name}")
    if not data or not isinstance(data, dict):
        return ""
    return norm(data.get("default_branch"))


def list_workflow_runs(gh: GitHubClient, full_name: str, workflow_id_or_file: str, branch: Optional[str]) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/workflows/{workflow_id_or_file}/runs"
    params = {"branch": branch} if branch else {}
    return list(gh.paginate(url, params=params, item_key="workflow_runs"))


def list_run_jobs(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/jobs"
    return list(gh.paginate(url, params={}, item_key="jobs"))


# =========================
# Job / step heuristics
# =========================
THIRD_PARTY_PROVIDER_RE = re.compile(
    r"(browserstack|sauce\s*labs|saucelabs|kobiton|headspin|bitbar|perfecto|lambdatest|genymotion\s*cloud|firebase\s*test\s*lab)",
    re.I,
)
THIRD_PARTY_LIFECYCLE_RE = re.compile(
    r"(start|stop|upload|download|results?|report|reports|session|app url|build id|device logs?|summary|finali[sz]e)",
    re.I,
)
THIRD_PARTY_INVOKE_RE = re.compile(
    r"(firebase\s+test\s+android\s+run|gcloud\s+firebase\s+test\s+android\s+run|flank\s+android\s+run|appcenter\s+test\s+run|saucectl\s+(run|test)|browserstack|maestro\s+cloud)",
    re.I,
)
GMD_RE = re.compile(r"(manageddevice|gmd|gradle managed device|managed device)", re.I)
COMMUNITY_RE = re.compile(
    r"(android-emulator-runner|emulator runner|create avd|avd|start emulator|emulator|connectedcheck|connectedandroidtest|androidtest)",
    re.I,
)
CUSTOM_RE = re.compile(
    r"(detox|flutter.*integration|integration test|baseline.?profile|macrobenchmark|uiautomator|espresso|instrumentation)",
    re.I,
)
GENERIC_INSTRU_RE = re.compile(
    r"(connectedcheck|connectedandroidtest|androidtest|instrumentation|manageddevice|gmd|detox|flutter.*integration|integration test|baseline.?profile|macrobenchmark|uiautomator|espresso|emulator|avd|firebase test|test lab|device farm)",
    re.I,
)
ARTIFACT_STEP_RE = re.compile(
    r"(upload-artifact|download-artifact|artifact|test-results|test results|results|report|reports|logs?|summary)",
    re.I,
)


def get_job_runtime_text(job: Dict) -> str:
    vals: List[str] = []
    vals.append(norm(job.get("name")))
    for st in (job.get("steps") or []):
        vals.append(norm(st.get("name")))
    return " | ".join([v for v in vals if v])


def job_base_name(job_name: str) -> str:
    """
    Collapse common matrix-expanded job suffixes:
      build (Foss) -> build
      test [api35] -> test
    """
    x = norm(job_name)
    if not x:
        return ""
    x = re.sub(r"\s+\([^)]*\)\s*$", "", x).strip()
    x = re.sub(r"\s+\[[^\]]*\]\s*$", "", x).strip()
    return x


def is_matrix_like_job_name(job_name: str) -> bool:
    x = norm(job_name)
    if not x:
        return False
    base = job_base_name(x)
    return bool(base and base != x)


def detect_job_style_tags(job: Dict, declared_styles: List[str], anchor_step_names: List[str]) -> Tuple[bool, List[str], str]:
    """
    Returns:
      is_instru_related_job, matched_styles, detect_method

    V18 framing:
    - Stage 2 is still a broad Layer-1 detector of instrumentation-related jobs
    - continuation/finalization jobs may still belong to the broad instrumentation envelope
    - auxiliary fields now expose multiplicity/complexity for repeated same-style cases
    """
    text = get_job_runtime_text(job)
    steps = job.get("steps") if isinstance(job.get("steps"), list) else []

    anchor_hit = False
    for st in steps:
        step_name = norm(st.get("name"))
        if step_name and anchored_step_match(step_name, anchor_step_names):
            anchor_hit = True
            break

    generic_instru = bool(GENERIC_INSTRU_RE.search(text))
    tp_hit = bool(THIRD_PARTY_PROVIDER_RE.search(text))
    tp_lifecycle = bool(THIRD_PARTY_LIFECYCLE_RE.search(text))
    gmd_hit = bool(GMD_RE.search(text))
    community_hit = bool(COMMUNITY_RE.search(text))
    custom_hit = bool(CUSTOM_RE.search(text))
    artifact_hit = bool(ARTIFACT_STEP_RE.search(text))
    real_device_hit = bool(re.search(r"(firebase test lab|device farm|real device)", text, re.I))

    matched_styles: List[str] = []

    if "Third-Party" in declared_styles and (tp_hit or (artifact_hit and tp_lifecycle)):
        matched_styles.append("Third-Party")
    if "GMD" in declared_styles and gmd_hit:
        matched_styles.append("GMD")
    if "Community" in declared_styles and community_hit:
        matched_styles.append("Community")
    if "Custom" in declared_styles and custom_hit:
        matched_styles.append("Custom")
    if "Real-Device" in declared_styles and real_device_hit:
        matched_styles.append("Real-Device")

    matched_styles = unique_preserve(matched_styles)

    is_instru_related_job = False
    detect_method = "none"

    if anchor_hit:
        is_instru_related_job = True
        detect_method = "step_name_anchor"
    elif generic_instru or tp_hit:
        is_instru_related_job = True
        detect_method = "job_or_step_text_third_party" if tp_hit else "job_or_step_text_regex"
    elif artifact_hit and tp_hit and tp_lifecycle:
        is_instru_related_job = True
        detect_method = "third_party_lifecycle_artifact"
    elif artifact_hit and len(declared_styles) == 1 and generic_instru:
        is_instru_related_job = True
        detect_method = "artifact_plus_instru_text"

    if is_instru_related_job and not matched_styles and len(declared_styles) == 1:
        matched_styles = declared_styles[:]
        detect_method = detect_method + "_single_declared_style"

    return is_instru_related_job, matched_styles, detect_method


def step_is_invocation_candidate(step_name: str, target_style: str, anchor_step_names: List[str]) -> bool:
    s = norm(step_name)
    if not s:
        return False
    if anchored_step_match(s, anchor_step_names):
        return True

    style = normalize_style_label(target_style)
    if style == "Third-Party":
        return bool(THIRD_PARTY_INVOKE_RE.search(s) or THIRD_PARTY_PROVIDER_RE.search(s))
    if style == "GMD":
        return bool(GMD_RE.search(s))
    if style == "Community":
        return bool(COMMUNITY_RE.search(s))
    if style == "Custom":
        return bool(CUSTOM_RE.search(s))

    return bool(GENERIC_INSTRU_RE.search(s))


def compute_parallel_overlap_stats(windows: List[Tuple[Optional[datetime], Optional[datetime]]]) -> Tuple[bool, int]:
    """
    Returns:
      has_overlap, max_parallel
    """
    events: List[Tuple[datetime, int]] = []
    for sdt, edt in windows:
        if not sdt or not edt:
            continue
        if edt < sdt:
            continue
        events.append((sdt, +1))
        events.append((edt, -1))

    if not events:
        return False, 0

    # Start before end on ties
    events.sort(key=lambda x: (x[0], -x[1]))

    current = 0
    max_parallel = 0
    has_overlap = False
    for _t, delta in events:
        current += delta
        if current > max_parallel:
            max_parallel = current
        if current >= 2:
            has_overlap = True

    return has_overlap, max_parallel


# =========================
# Timing helpers
# =========================
def compute_run_window_from_jobs(
    jobs: List[Dict],
    run_started_at: str,
    run_updated_at: str,
) -> Tuple[str, str, Optional[int], str]:
    starts: List[datetime] = []
    ends: List[datetime] = []

    for j in jobs:
        sdt = iso_to_dt(j.get("started_at"))
        edt = iso_to_dt(j.get("completed_at"))
        if sdt:
            starts.append(sdt)
        if edt:
            ends.append(edt)

    run_start_dt = iso_to_dt(run_started_at)
    run_end_dt = iso_to_dt(run_updated_at)

    if not run_start_dt and starts:
        run_start_dt = min(starts)
    if not run_end_dt and ends:
        run_end_dt = max(ends)

    source = "run_api"
    if run_start_dt and run_end_dt and not (iso_to_dt(run_started_at) and iso_to_dt(run_updated_at)):
        source = "hybrid_run_api_jobs"
    elif not (iso_to_dt(run_started_at) and iso_to_dt(run_updated_at)) and starts and ends:
        source = "jobs_window"

    return (
        run_start_dt.isoformat().replace("+00:00", "Z") if run_start_dt else "",
        run_end_dt.isoformat().replace("+00:00", "Z") if run_end_dt else "",
        dt_to_seconds(run_start_dt, run_end_dt),
        source if run_start_dt and run_end_dt else "missing",
    )


def pick_anchor_job(
    jobs: List[Dict],
    declared_styles: List[str],
    anchor_step_names: List[str],
) -> Tuple[Optional[Dict], str, List[str]]:
    """
    Pick the earliest instrumentation-related job in the run.
    Returns:
      anchor_job, detect_method, all_instru_related_job_names
    """
    candidates: List[Tuple[datetime, Dict, str]] = []
    all_instru_related_job_names: List[str] = []

    for j in jobs:
        is_instru_related_job, _matched_styles, detect_method = detect_job_style_tags(
            job=j,
            declared_styles=declared_styles,
            anchor_step_names=anchor_step_names,
        )
        if not is_instru_related_job:
            continue

        jname = norm(j.get("name"))
        if jname:
            all_instru_related_job_names.append(jname)

        js = iso_to_dt(j.get("started_at"))
        if js:
            candidates.append((js, j, detect_method))

    if not candidates:
        return None, "none", unique_preserve(all_instru_related_job_names)

    candidates.sort(key=lambda x: x[0])
    _dt, anchor_job, detect_method = candidates[0]
    return anchor_job, detect_method, unique_preserve(all_instru_related_job_names)


# =========================
# Inventory builders
# =========================
def build_run_level_metrics(
    jobs: List[Dict],
    run_created_at: str,
    run_started_at: str,
    run_updated_at: str,
    anchor_step_names: List[str],
    declared_styles: List[str],
) -> Dict[str, Union[str, int, None]]:
    """
    Run-level durable Layer-1 metrics for V18:
    - broad instrumentation job envelope
    - anchor job retained as semantic reference
    - Layer-2-aligned proxy aliases exposed for downstream convenience,
      but still based on Layer-1 broad job telemetry
    - auxiliary multiplicity/matrix/parallel complexity fields added
    """
    out: Dict[str, Union[str, int, None]] = {
        "queue_seconds": None,
        "run_duration_seconds": None,
        "runner_labels_union": "",
        "instru_job_count": 0,
        "instru_job_names": "",
        "instru_detect_method": "none",

        # broad instrumentation envelope
        "instru_first_started_at": "",
        "instru_last_completed_at": "",
        "instru_window_seconds": None,

        "time_to_instrumentation_envelope_seconds": None,
        "instrumentation_job_envelope_seconds": None,
        "post_instrumentation_tail_seconds": None,
        "layer1_model": "instrumentation_job_envelope",
        "layer1_proxy_quality": "missing",

        # anchor job retained as reference point
        "anchor_job_name": "",
        "anchor_job_started_at": "",
        "anchor_job_completed_at": "",
        "anchor_job_source": "missing",

        # revised broad proxy aliases
        "pre_invocation_seconds_proxy": None,
        "invocation_execution_window_seconds_proxy": None,
        "post_invocation_seconds_proxy": None,

        # backward-friendly legacy aliases
        "time_to_invocation_seconds_proxy": None,
        "invocation_tail_seconds_proxy": None,
        "time_to_first_instru_seconds": None,
        "anchor_job_start_source": "missing",
        "time_to_first_instru_from_anchor_job_seconds": None,
        "time_to_first_instru_from_anchor_job_quality": "missing",

        # new V18 run-level auxiliary fields
        "instru_distinct_job_count": 0,
        "instru_distinct_job_base_name_count": 0,
        "instru_matrix_like_job_count": 0,
        "instru_matrix_expanded_flag": "false",
        "instru_parallel_jobs_flag": "false",
        "instru_max_parallel_jobs": 0,
    }

    out["queue_seconds"] = dt_to_seconds(iso_to_dt(run_created_at), iso_to_dt(run_started_at))

    run_start_eff, run_end_eff, run_dur_eff, _ = compute_run_window_from_jobs(
        jobs=jobs,
        run_started_at=run_started_at,
        run_updated_at=run_updated_at,
    )
    out["run_duration_seconds"] = run_dur_eff

    labels_union: Set[str] = set()
    instru_related_job_names: List[str] = []
    instru_related_job_base_names: List[str] = []
    instru_starts: List[Tuple[datetime, str, str]] = []
    instru_ends: List[Tuple[datetime, str, str]] = []
    instru_windows: List[Tuple[Optional[datetime], Optional[datetime]]] = []

    for j in jobs:
        for lab in (j.get("labels") or []):
            if isinstance(lab, str) and norm(lab):
                labels_union.add(norm(lab))

        is_instru_related_job, _styles, detect_method = detect_job_style_tags(
            job=j,
            declared_styles=declared_styles,
            anchor_step_names=anchor_step_names,
        )
        if not is_instru_related_job:
            continue

        jname = norm(j.get("name"))
        jbase = job_base_name(jname)
        js = iso_to_dt(j.get("started_at"))
        je = iso_to_dt(j.get("completed_at"))

        if jname:
            instru_related_job_names.append(jname)
            instru_related_job_base_names.append(jbase or jname)
        if js:
            instru_starts.append((js, jname, detect_method))
        if je:
            instru_ends.append((je, jname, detect_method))
        instru_windows.append((js, je))

    out["runner_labels_union"] = ",".join(sorted(labels_union))
    out["instru_job_names"] = safe_join_names(instru_related_job_names)
    out["instru_job_count"] = len(unique_preserve(instru_related_job_names))
    out["instru_distinct_job_count"] = len(unique_preserve(instru_related_job_names))
    out["instru_distinct_job_base_name_count"] = len(unique_preserve(instru_related_job_base_names))
    out["instru_matrix_like_job_count"] = sum(1 for n in unique_preserve(instru_related_job_names) if is_matrix_like_job_name(n))

    if (
        out["instru_distinct_job_count"] is not None
        and out["instru_distinct_job_base_name_count"] is not None
        and int(out["instru_distinct_job_count"]) > int(out["instru_distinct_job_base_name_count"])
    ):
        out["instru_matrix_expanded_flag"] = "true"

    has_parallel, max_parallel = compute_parallel_overlap_stats(instru_windows)
    out["instru_parallel_jobs_flag"] = "true" if has_parallel else "false"
    out["instru_max_parallel_jobs"] = max_parallel

    run_start_dt = iso_to_dt(run_start_eff)
    run_end_dt = iso_to_dt(run_end_eff)

    if instru_starts:
        instru_starts.sort(key=lambda x: x[0])
        first_dt, _first_name, first_method = instru_starts[0]
        out["instru_first_started_at"] = first_dt.isoformat().replace("+00:00", "Z")
        out["time_to_instrumentation_envelope_seconds"] = dt_to_seconds(run_start_dt, first_dt)
        out["instru_detect_method"] = first_method or "job_text"

    if instru_ends:
        instru_ends.sort(key=lambda x: x[0])
        last_dt, _last_name, _last_method = instru_ends[-1]
        out["instru_last_completed_at"] = last_dt.isoformat().replace("+00:00", "Z")
        out["post_instrumentation_tail_seconds"] = dt_to_seconds(last_dt, run_end_dt)

    if out["instru_first_started_at"] and out["instru_last_completed_at"]:
        first_dt = iso_to_dt(str(out["instru_first_started_at"]))
        last_dt = iso_to_dt(str(out["instru_last_completed_at"]))
        out["instru_window_seconds"] = dt_to_seconds(first_dt, last_dt)
        out["instrumentation_job_envelope_seconds"] = out["instru_window_seconds"]

    anchor_job, detect_method, all_instru_job_names = pick_anchor_job(
        jobs=jobs,
        declared_styles=declared_styles,
        anchor_step_names=anchor_step_names,
    )
    if all_instru_job_names:
        out["instru_job_names"] = safe_join_names(all_instru_job_names)
        out["instru_job_count"] = len(all_instru_job_names)
        out["instru_distinct_job_count"] = len(unique_preserve(all_instru_job_names))
        out["instru_matrix_like_job_count"] = sum(1 for n in unique_preserve(all_instru_job_names) if is_matrix_like_job_name(n))
    if detect_method and detect_method != "none" and out["instru_detect_method"] == "none":
        out["instru_detect_method"] = detect_method

    if anchor_job is not None:
        anchor_name = norm(anchor_job.get("name"))
        anchor_start_dt = iso_to_dt(anchor_job.get("started_at"))
        anchor_end_dt = iso_to_dt(anchor_job.get("completed_at"))

        out["anchor_job_name"] = anchor_name
        out["anchor_job_started_at"] = anchor_start_dt.isoformat().replace("+00:00", "Z") if anchor_start_dt else ""
        out["anchor_job_completed_at"] = anchor_end_dt.isoformat().replace("+00:00", "Z") if anchor_end_dt else ""
        out["anchor_job_source"] = detect_method or "job_text"

        out["anchor_job_start_source"] = out["anchor_job_source"]
        out["time_to_first_instru_from_anchor_job_seconds"] = 0 if anchor_start_dt else None
        out["time_to_first_instru_from_anchor_job_quality"] = "anchor_job_reference" if anchor_start_dt else "missing"

    out["pre_invocation_seconds_proxy"] = out["time_to_instrumentation_envelope_seconds"]
    out["invocation_execution_window_seconds_proxy"] = out["instrumentation_job_envelope_seconds"]
    out["post_invocation_seconds_proxy"] = out["post_instrumentation_tail_seconds"]

    out["time_to_invocation_seconds_proxy"] = out["time_to_instrumentation_envelope_seconds"]
    out["invocation_tail_seconds_proxy"] = out["post_instrumentation_tail_seconds"]
    out["time_to_first_instru_seconds"] = out["time_to_instrumentation_envelope_seconds"]

    if (
        out["time_to_instrumentation_envelope_seconds"] is not None
        and out["instrumentation_job_envelope_seconds"] is not None
        and out["post_instrumentation_tail_seconds"] is not None
    ):
        out["layer1_proxy_quality"] = "broad_job_envelope"
    elif out["instrumentation_job_envelope_seconds"] is not None:
        out["layer1_proxy_quality"] = "partial_broad_job_envelope"

    return out


def build_run_per_style_rows(
    full_name: str,
    workflow_identifier: str,
    workflow_id: str,
    workflow_path: str,
    run: Dict,
    declared_styles: List[str],
    anchor_step_names: List[str],
    jobs: List[Dict],
    run_start_eff: str,
    run_end_eff: str,
    run_duration_eff: Optional[int],
    run_timing_source: str,
) -> List[Dict[str, object]]:
    """
    Build durable Layer-1 run × style rows for V18.
    Core Stage-2 middle component remains the broad instrumentation envelope.
    Revised Layer-2-aligned proxy names are exposed as broad job-level proxies only.
    New auxiliary fields expose same-style multiplicity / matrix / parallel complexity.
    """
    run_start_dt = iso_to_dt(run_start_eff)
    run_end_dt = iso_to_dt(run_end_eff)

    style_to_jobs: Dict[str, List[Dict]] = {s: [] for s in declared_styles}
    ambiguous_job_names: List[str] = []
    all_instru_job_names_any: List[str] = []

    for j in jobs:
        is_instru_related_job, matched_styles, detect_method = detect_job_style_tags(
            job=j,
            declared_styles=declared_styles,
            anchor_step_names=anchor_step_names,
        )
        if not is_instru_related_job:
            continue

        jname = norm(j.get("name"))
        if jname:
            all_instru_job_names_any.append(jname)

        payload = {
            "job": j,
            "detect_method": detect_method,
        }

        if len(matched_styles) == 1:
            style_to_jobs[matched_styles[0]].append(payload)
        elif len(matched_styles) > 1:
            if jname:
                ambiguous_job_names.append(jname)
        else:
            if len(declared_styles) == 1:
                style_to_jobs[declared_styles[0]].append({
                    "job": j,
                    "detect_method": detect_method + "_fallback_single_style",
                })
            else:
                if jname:
                    ambiguous_job_names.append(jname)

    rows: List[Dict[str, object]] = []

    multi_style_run_flag = len(declared_styles) > 1
    all_styles_in_run = safe_join_names(declared_styles)
    all_instru_job_names_any_s = safe_join_names(all_instru_job_names_any)
    ambiguous_jobs_s = safe_join_names(ambiguous_job_names)

    for style in declared_styles:
        assigned = style_to_jobs.get(style, [])

        assigned_job_names: List[str] = []
        assigned_job_base_names: List[str] = []
        detect_methods: List[str] = []

        anchor_job = None
        anchor_job_name = ""
        anchor_job_started_at = ""
        anchor_job_completed_at = ""
        anchor_job_source = "missing"

        anchor_job_start_dt: Optional[datetime] = None
        anchor_job_end_dt: Optional[datetime] = None

        style_first_instru_job_name = ""
        style_first_instru_job_started_at = ""
        style_first_instru_job_source = "missing"
        style_last_instru_job_name = ""
        style_last_instru_job_completed_at = ""
        style_last_instru_job_source = "missing"

        first_any_dt: Optional[datetime] = None
        last_any_dt: Optional[datetime] = None

        candidates: List[Tuple[datetime, Dict, str]] = []
        end_candidates: List[Tuple[datetime, Dict, str]] = []

        # New V18 auxiliary trackers
        style_windows: List[Tuple[Optional[datetime], Optional[datetime]]] = []
        style_invocation_candidate_step_names: List[str] = []
        style_invocation_candidate_step_count = 0

        for item in assigned:
            j = item["job"]
            detect_method = norm(item["detect_method"])
            detect_methods.append(detect_method)

            jname = norm(j.get("name"))
            jbase = job_base_name(jname)
            js = iso_to_dt(j.get("started_at"))
            je = iso_to_dt(j.get("completed_at"))

            if jname:
                assigned_job_names.append(jname)
                assigned_job_base_names.append(jbase or jname)

            style_windows.append((js, je))

            # job-level timing
            if js:
                candidates.append((js, j, detect_method))
                if first_any_dt is None or js < first_any_dt:
                    first_any_dt = js
                    style_first_instru_job_name = jname
                    style_first_instru_job_started_at = js.isoformat().replace("+00:00", "Z")
                    style_first_instru_job_source = detect_method or "job_text"

            if je:
                end_candidates.append((je, j, detect_method))
                if last_any_dt is None or je > last_any_dt:
                    last_any_dt = je
                    style_last_instru_job_name = jname
                    style_last_instru_job_completed_at = je.isoformat().replace("+00:00", "Z")
                    style_last_instru_job_source = detect_method or "job_text"

            # step-level invocation candidate proxy counting
            steps = j.get("steps") if isinstance(j.get("steps"), list) else []
            for st in steps:
                step_name = norm(st.get("name"))
                if step_is_invocation_candidate(step_name, style, anchor_step_names):
                    style_invocation_candidate_step_count += 1
                    style_invocation_candidate_step_names.append(step_name)

        if candidates:
            candidates.sort(key=lambda x: x[0])
            _dt, anchor_job, anchor_method = candidates[0]
            anchor_job_name = norm(anchor_job.get("name"))
            anchor_job_start_dt = iso_to_dt(anchor_job.get("started_at"))
            anchor_job_end_dt = iso_to_dt(anchor_job.get("completed_at"))
            anchor_job_started_at = anchor_job_start_dt.isoformat().replace("+00:00", "Z") if anchor_job_start_dt else ""
            anchor_job_completed_at = anchor_job_end_dt.isoformat().replace("+00:00", "Z") if anchor_job_end_dt else ""
            anchor_job_source = anchor_method or "job_text"

        time_to_instrumentation_envelope = dt_to_seconds(run_start_dt, first_any_dt)
        instrumentation_job_envelope = dt_to_seconds(first_any_dt, last_any_dt)
        post_instrumentation_tail = dt_to_seconds(last_any_dt, run_end_dt)

        segmentation_confidence = (
            "high" if first_any_dt is not None and last_any_dt is not None and not ambiguous_job_names else
            "medium" if first_any_dt is not None else
            "missing"
        )

        style_distinct_job_count = len(unique_preserve(assigned_job_names))
        style_distinct_job_base_name_count = len(unique_preserve(assigned_job_base_names))
        style_matrix_like_job_count = sum(1 for n in unique_preserve(assigned_job_names) if is_matrix_like_job_name(n))
        style_matrix_expanded_flag = "true" if style_distinct_job_count > style_distinct_job_base_name_count else "false"

        style_parallel_same_style, style_max_parallel_jobs = compute_parallel_overlap_stats(style_windows)
        style_parallel_same_style_flag = "true" if style_parallel_same_style else "false"

        style_repeated_same_style_flag = "true" if (
            style_distinct_job_count > 1
            or style_invocation_candidate_step_count > 1
            or style_matrix_expanded_flag == "true"
        ) else "false"

        if style_distinct_job_count <= 1 and style_invocation_candidate_step_count <= 1:
            style_same_style_complexity_class = "single_path"
        elif style_matrix_expanded_flag == "true" and style_parallel_same_style_flag == "true":
            style_same_style_complexity_class = "matrix_parallel_repeated"
        elif style_matrix_expanded_flag == "true":
            style_same_style_complexity_class = "matrix_repeated"
        elif style_distinct_job_count > 1 and style_parallel_same_style_flag == "true":
            style_same_style_complexity_class = "parallel_repeated"
        elif style_distinct_job_count > 1 or style_invocation_candidate_step_count > 1:
            style_same_style_complexity_class = "serial_or_multi_candidate_repeated"
        else:
            style_same_style_complexity_class = "single_path"

        row = {
            "full_name": full_name,
            "workflow_identifier": workflow_identifier,
            "workflow_id": workflow_id,
            "workflow_path": workflow_path,

            "run_id": norm(str(run.get("id") or "")),
            "run_number": norm(str(run.get("run_number") or "")),
            "run_attempt": norm(str(run.get("run_attempt") or "")),
            "created_at": norm(run.get("created_at")),
            "run_started_at": norm(run.get("run_started_at")),
            "run_updated_at": norm(run.get("updated_at")),
            "status": norm(run.get("status")),
            "run_conclusion": norm(run.get("conclusion")),
            "event": norm(run.get("event")),
            "head_branch": norm(run.get("head_branch")),
            "head_sha": norm(run.get("head_sha")),
            "html_url": norm(run.get("html_url")),

            "target_style": style,
            "styles_in_run_all": all_styles_in_run,
            "multi_style_run_flag": "true" if multi_style_run_flag else "false",

            "layer1_run_started_at_effective": run_start_eff,
            "layer1_run_ended_at_effective": run_end_eff,
            "layer1_run_duration_seconds_effective": "" if run_duration_eff is None else str(run_duration_eff),
            "layer1_run_timing_source": run_timing_source,

            "style_instru_job_count": str(style_distinct_job_count),
            "style_instru_job_names": safe_join_names(assigned_job_names),

            "style_first_instru_job_name": style_first_instru_job_name,
            "style_first_instru_job_started_at": style_first_instru_job_started_at,
            "style_first_instru_job_source": style_first_instru_job_source,
            "style_last_instru_job_name": style_last_instru_job_name,
            "style_last_instru_job_completed_at": style_last_instru_job_completed_at,
            "style_last_instru_job_source": style_last_instru_job_source,

            # V18 new auxiliary complexity fields
            "style_distinct_job_count": str(style_distinct_job_count),
            "style_distinct_job_base_name_count": str(style_distinct_job_base_name_count),
            "style_matrix_like_job_count": str(style_matrix_like_job_count),
            "style_matrix_expanded_flag": style_matrix_expanded_flag,
            "style_parallel_same_style_flag": style_parallel_same_style_flag,
            "style_max_parallel_jobs": str(style_max_parallel_jobs),
            "style_repeated_same_style_flag": style_repeated_same_style_flag,
            "style_invocation_candidate_step_count_proxy": str(style_invocation_candidate_step_count),
            "style_distinct_invocation_step_name_count_proxy": str(len(unique_preserve(style_invocation_candidate_step_names))),
            "style_invocation_candidate_step_names_proxy": safe_join_names(style_invocation_candidate_step_names),
            "style_same_style_complexity_class": style_same_style_complexity_class,

            # preferred Layer-1 fields
            "style_time_to_instrumentation_envelope_seconds": "" if time_to_instrumentation_envelope is None else str(time_to_instrumentation_envelope),
            "style_instrumentation_job_envelope_seconds": "" if instrumentation_job_envelope is None else str(instrumentation_job_envelope),
            "style_post_instrumentation_tail_seconds": "" if post_instrumentation_tail is None else str(post_instrumentation_tail),
            "style_layer1_model": "instrumentation_job_envelope",

            # anchor reference fields
            "style_anchor_job_name": anchor_job_name,
            "style_anchor_job_started_at": anchor_job_started_at,
            "style_anchor_job_completed_at": anchor_job_completed_at,
            "style_anchor_job_source": anchor_job_source,

            # broad proxy aliases
            "style_pre_invocation_seconds_proxy": "" if time_to_instrumentation_envelope is None else str(time_to_instrumentation_envelope),
            "style_invocation_execution_window_seconds_proxy": "" if instrumentation_job_envelope is None else str(instrumentation_job_envelope),
            "style_post_invocation_seconds_proxy": "" if post_instrumentation_tail is None else str(post_instrumentation_tail),

            # backward-friendly legacy aliases
            "style_time_to_invocation_seconds_proxy": "" if time_to_instrumentation_envelope is None else str(time_to_instrumentation_envelope),
            "style_invocation_tail_seconds_proxy": "" if post_instrumentation_tail is None else str(post_instrumentation_tail),
            "style_time_to_instru_job_start_seconds": "" if time_to_instrumentation_envelope is None else str(time_to_instrumentation_envelope),
            "style_instru_job_envelope_seconds": "" if instrumentation_job_envelope is None else str(instrumentation_job_envelope),
            "style_post_instru_job_tail_seconds": "" if post_instrumentation_tail is None else str(post_instrumentation_tail),

            "style_job_detection_methods": safe_join_names(detect_methods),
            "style_job_segmentation_confidence": segmentation_confidence,
            "style_overlap_with_other_styles_flag": "true" if ambiguous_job_names else "false",
            "ambiguous_instru_job_names_in_run": ambiguous_jobs_s,
            "all_instru_job_names_in_run": all_instru_job_names_any_s,

            "extracted_at_utc": now_utc_iso(),
        }
        rows.append(row)

    return rows


# =========================
# Read verified workflows
# =========================
def load_verified_workflows(path: Path) -> List[Dict[str, str]]:
    if not path.exists():
        raise FileNotFoundError(f"Verified workflows CSV not found: {path}")
    rows: List[Dict[str, str]] = []
    with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        for r in rdr:
            rows.append({(k or ""): (v or "") for k, v in r.items()})
    return rows


# =========================
# MAIN
# =========================
def main() -> None:
    if not IN_VERIFIED_WORKFLOWS_CSV.exists():
        raise FileNotFoundError(f"Missing input: {IN_VERIFIED_WORKFLOWS_CSV}")

    if OUT_RUN_INVENTORY_CSV.exists():
        OUT_RUN_INVENTORY_CSV.unlink()
    if OUT_RUN_PER_STYLE_CSV.exists():
        OUT_RUN_PER_STYLE_CSV.unlink()

    tokens = read_env_tokens(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens)

    rows = load_verified_workflows(IN_VERIFIED_WORKFLOWS_CSV)
    if PROCESS_ONLY_LOOKS_LIKE_INSTRU:
        rows = [r for r in rows if low(r.get("looks_like_instru")) == "yes"]

    if not rows:
        raise RuntimeError("No workflows found to process (check verified CSV or filter).")

    after_dt = iso_to_dt(RUN_CREATED_AT_AFTER) if RUN_CREATED_AT_AFTER else None

    out_run_fields = [
        # workflow identity
        "full_name",
        "default_branch",
        "workflow_identifier",
        "workflow_id",
        "workflow_path",

        # Stage 1 labels / pass-through
        "looks_like_instru",
        "styles",
        "invocation_types",
        "third_party_provider_name",
        "test_invocation_step_names",
        "anchor_job_ordinal",
        "anchor_step_ordinal_in_job",
        "called_instru_signal",
        "called_instru_file_paths",
        "called_instru_origin_refs",
        "called_instru_origin_step_names",
        "called_instru_file_types",

        # run metadata
        "run_id",
        "run_number",
        "run_attempt",
        "head_sha",
        "created_at",
        "run_started_at",
        "run_updated_at",
        "status",
        "run_conclusion",
        "event",
        "head_branch",
        "html_url",
        "extracted_at_utc",

        # durable run timing
        "L1_run_started_at_effective",
        "L1_run_ended_at_effective",
        "L1_run_duration_seconds_effective",
        "L1_run_timing_source",

        # broad inventory fields
        "queue_seconds",
        "instru_detect_method",
        "instru_job_count",
        "instru_job_names",
        "instru_first_started_at",
        "instru_last_completed_at",
        "instru_window_seconds",

        # new V18 run-level auxiliary complexity fields
        "instru_distinct_job_count",
        "instru_distinct_job_base_name_count",
        "instru_matrix_like_job_count",
        "instru_matrix_expanded_flag",
        "instru_parallel_jobs_flag",
        "instru_max_parallel_jobs",

        # preferred Layer-1 fields
        "time_to_instrumentation_envelope_seconds",
        "instrumentation_job_envelope_seconds",
        "post_instrumentation_tail_seconds",
        "layer1_model",
        "layer1_proxy_quality",

        # anchor reference fields
        "anchor_job_name",
        "anchor_job_started_at",
        "anchor_job_completed_at",
        "anchor_job_source",

        # broad proxy aliases
        "pre_invocation_seconds_proxy",
        "invocation_execution_window_seconds_proxy",
        "post_invocation_seconds_proxy",

        # backward-friendly legacy aliases
        "time_to_invocation_seconds_proxy",
        "invocation_tail_seconds_proxy",
        "time_to_first_instru_seconds",
        "anchor_job_start_source",
        "time_to_first_instru_from_anchor_job_seconds",
        "time_to_first_instru_from_anchor_job_quality",
    ]

    out_style_fields = [
        "full_name",
        "workflow_identifier",
        "workflow_id",
        "workflow_path",

        "run_id",
        "run_number",
        "run_attempt",
        "created_at",
        "run_started_at",
        "run_updated_at",
        "status",
        "run_conclusion",
        "event",
        "head_branch",
        "head_sha",
        "html_url",

        "target_style",
        "styles_in_run_all",
        "multi_style_run_flag",

        "layer1_run_started_at_effective",
        "layer1_run_ended_at_effective",
        "layer1_run_duration_seconds_effective",
        "layer1_run_timing_source",

        "style_instru_job_count",
        "style_instru_job_names",
        "style_first_instru_job_name",
        "style_first_instru_job_started_at",
        "style_first_instru_job_source",
        "style_last_instru_job_name",
        "style_last_instru_job_completed_at",
        "style_last_instru_job_source",

        # new V18 auxiliary complexity fields
        "style_distinct_job_count",
        "style_distinct_job_base_name_count",
        "style_matrix_like_job_count",
        "style_matrix_expanded_flag",
        "style_parallel_same_style_flag",
        "style_max_parallel_jobs",
        "style_repeated_same_style_flag",
        "style_invocation_candidate_step_count_proxy",
        "style_distinct_invocation_step_name_count_proxy",
        "style_invocation_candidate_step_names_proxy",
        "style_same_style_complexity_class",

        # preferred Layer-1 fields
        "style_time_to_instrumentation_envelope_seconds",
        "style_instrumentation_job_envelope_seconds",
        "style_post_instrumentation_tail_seconds",
        "style_layer1_model",

        # anchor reference fields
        "style_anchor_job_name",
        "style_anchor_job_started_at",
        "style_anchor_job_completed_at",
        "style_anchor_job_source",

        # broad proxy aliases
        "style_pre_invocation_seconds_proxy",
        "style_invocation_execution_window_seconds_proxy",
        "style_post_invocation_seconds_proxy",

        # backward-friendly legacy aliases
        "style_time_to_invocation_seconds_proxy",
        "style_invocation_tail_seconds_proxy",
        "style_time_to_instru_job_start_seconds",
        "style_instru_job_envelope_seconds",
        "style_post_instru_job_tail_seconds",

        "style_job_detection_methods",
        "style_job_segmentation_confidence",
        "style_overlap_with_other_styles_flag",
        "ambiguous_instru_job_names_in_run",
        "all_instru_job_names_in_run",

        "extracted_at_utc",
    ]

    ensure_csv_header(OUT_RUN_INVENTORY_CSV, out_run_fields)
    ensure_csv_header(OUT_RUN_PER_STYLE_CSV, out_style_fields)

    existing_run_ids = load_existing_keys(OUT_RUN_INVENTORY_CSV, "run_id")
    default_branch_cache: Dict[str, str] = {}

    wf_iter = rows
    if tqdm is not None:
        wf_iter = tqdm(rows, desc="Stage2 V18: workflows -> runs")

    for wf in wf_iter:
        full_name = norm(wf.get("full_name"))
        workflow_identifier = norm(wf.get("workflow_identifier"))
        workflow_id = norm(wf.get("workflow_id"))
        workflow_path = norm(wf.get("workflow_path"))

        if not full_name:
            continue

        workflow_key_for_runs = workflow_id or workflow_identifier or workflow_path
        if not workflow_key_for_runs:
            continue

        if DEFAULT_BRANCH_ONLY:
            if full_name not in default_branch_cache:
                default_branch_cache[full_name] = get_repo_default_branch(gh, full_name)
            default_branch = default_branch_cache[full_name]
            if not default_branch:
                continue
        else:
            default_branch = ""

        branch = default_branch if DEFAULT_BRANCH_ONLY else None

        runs = list_workflow_runs(gh, full_name, workflow_key_for_runs, branch=branch) or []
        if MAX_RUNS_PER_WORKFLOW is not None:
            runs = runs[:MAX_RUNS_PER_WORKFLOW]

        declared_styles = split_styles(wf.get("styles") or wf.get("inferred_styles") or "")
        if not declared_styles:
            one_style = normalize_style_label(wf.get("inferred_style"))
            declared_styles = [one_style] if one_style in STYLE_CANONICAL else []

        anchor_step_names = parse_anchor_step_names(wf.get("test_invocation_step_names") or "")

        for run in runs:
            run_id = norm(str(run.get("id") or ""))
            if not run_id:
                continue
            if run_id in existing_run_ids:
                continue

            created_at = norm(run.get("created_at"))
            if after_dt:
                cdt = iso_to_dt(created_at)
                if cdt and cdt < after_dt:
                    continue

            head_branch = norm(run.get("head_branch"))
            if DEFAULT_BRANCH_ONLY and head_branch and head_branch != default_branch:
                continue

            jobs = list_run_jobs(gh, full_name, int(run_id)) if FETCH_JOBS_FOR_EACH_RUN else []

            run_started_at = norm(run.get("run_started_at"))
            run_updated_at = norm(run.get("updated_at"))

            run_start_eff, run_end_eff, run_dur_eff, run_timing_source = compute_run_window_from_jobs(
                jobs=jobs,
                run_started_at=run_started_at,
                run_updated_at=run_updated_at,
            )

            run_metrics = build_run_level_metrics(
                jobs=jobs,
                run_created_at=created_at,
                run_started_at=run_started_at,
                run_updated_at=run_updated_at,
                anchor_step_names=anchor_step_names,
                declared_styles=declared_styles,
            )

            append_row(OUT_RUN_INVENTORY_CSV, out_run_fields, {
                "full_name": full_name,
                "default_branch": default_branch,
                "workflow_identifier": workflow_identifier,
                "workflow_id": workflow_id,
                "workflow_path": workflow_path,

                "looks_like_instru": norm(wf.get("looks_like_instru")),
                "styles": norm(wf.get("styles")),
                "invocation_types": norm(wf.get("invocation_types")),
                "third_party_provider_name": norm(wf.get("third_party_provider_name")),
                "test_invocation_step_names": norm(wf.get("test_invocation_step_names")),
                "anchor_job_ordinal": norm(wf.get("anchor_job_ordinal")),
                "anchor_step_ordinal_in_job": norm(wf.get("anchor_step_ordinal_in_job")),
                "called_instru_signal": norm(wf.get("called_instru_signal")),
                "called_instru_file_paths": norm(wf.get("called_instru_file_paths")),
                "called_instru_origin_refs": norm(wf.get("called_instru_origin_refs")),
                "called_instru_origin_step_names": norm(wf.get("called_instru_origin_step_names")),
                "called_instru_file_types": norm(wf.get("called_instru_file_types")),

                "run_id": run_id,
                "run_number": norm(str(run.get("run_number") or "")),
                "run_attempt": norm(str(run.get("run_attempt") or "")),
                "head_sha": norm(run.get("head_sha")),
                "created_at": created_at,
                "run_started_at": run_started_at,
                "run_updated_at": run_updated_at,
                "status": norm(run.get("status")),
                "run_conclusion": norm(run.get("conclusion")),
                "event": norm(run.get("event")),
                "head_branch": head_branch,
                "html_url": norm(run.get("html_url")),
                "extracted_at_utc": now_utc_iso(),

                "L1_run_started_at_effective": run_start_eff,
                "L1_run_ended_at_effective": run_end_eff,
                "L1_run_duration_seconds_effective": "" if run_dur_eff is None else str(run_dur_eff),
                "L1_run_timing_source": run_timing_source,

                "queue_seconds": "" if run_metrics["queue_seconds"] is None else str(run_metrics["queue_seconds"]),
                "instru_detect_method": run_metrics["instru_detect_method"],
                "instru_job_count": "" if run_metrics["instru_job_count"] is None else str(run_metrics["instru_job_count"]),
                "instru_job_names": run_metrics["instru_job_names"],
                "instru_first_started_at": run_metrics["instru_first_started_at"],
                "instru_last_completed_at": run_metrics["instru_last_completed_at"],
                "instru_window_seconds": "" if run_metrics["instru_window_seconds"] is None else str(run_metrics["instru_window_seconds"]),

                "instru_distinct_job_count": "" if run_metrics["instru_distinct_job_count"] is None else str(run_metrics["instru_distinct_job_count"]),
                "instru_distinct_job_base_name_count": "" if run_metrics["instru_distinct_job_base_name_count"] is None else str(run_metrics["instru_distinct_job_base_name_count"]),
                "instru_matrix_like_job_count": "" if run_metrics["instru_matrix_like_job_count"] is None else str(run_metrics["instru_matrix_like_job_count"]),
                "instru_matrix_expanded_flag": run_metrics["instru_matrix_expanded_flag"],
                "instru_parallel_jobs_flag": run_metrics["instru_parallel_jobs_flag"],
                "instru_max_parallel_jobs": "" if run_metrics["instru_max_parallel_jobs"] is None else str(run_metrics["instru_max_parallel_jobs"]),

                "time_to_instrumentation_envelope_seconds": "" if run_metrics["time_to_instrumentation_envelope_seconds"] is None else str(run_metrics["time_to_instrumentation_envelope_seconds"]),
                "instrumentation_job_envelope_seconds": "" if run_metrics["instrumentation_job_envelope_seconds"] is None else str(run_metrics["instrumentation_job_envelope_seconds"]),
                "post_instrumentation_tail_seconds": "" if run_metrics["post_instrumentation_tail_seconds"] is None else str(run_metrics["post_instrumentation_tail_seconds"]),
                "layer1_model": run_metrics["layer1_model"],
                "layer1_proxy_quality": run_metrics["layer1_proxy_quality"],

                "anchor_job_name": run_metrics["anchor_job_name"],
                "anchor_job_started_at": run_metrics["anchor_job_started_at"],
                "anchor_job_completed_at": run_metrics["anchor_job_completed_at"],
                "anchor_job_source": run_metrics["anchor_job_source"],

                "pre_invocation_seconds_proxy": "" if run_metrics["pre_invocation_seconds_proxy"] is None else str(run_metrics["pre_invocation_seconds_proxy"]),
                "invocation_execution_window_seconds_proxy": "" if run_metrics["invocation_execution_window_seconds_proxy"] is None else str(run_metrics["invocation_execution_window_seconds_proxy"]),
                "post_invocation_seconds_proxy": "" if run_metrics["post_invocation_seconds_proxy"] is None else str(run_metrics["post_invocation_seconds_proxy"]),

                "time_to_invocation_seconds_proxy": "" if run_metrics["time_to_invocation_seconds_proxy"] is None else str(run_metrics["time_to_invocation_seconds_proxy"]),
                "invocation_tail_seconds_proxy": "" if run_metrics["invocation_tail_seconds_proxy"] is None else str(run_metrics["invocation_tail_seconds_proxy"]),
                "time_to_first_instru_seconds": "" if run_metrics["time_to_first_instru_seconds"] is None else str(run_metrics["time_to_first_instru_seconds"]),
                "anchor_job_start_source": run_metrics["anchor_job_start_source"],
                "time_to_first_instru_from_anchor_job_seconds": "" if run_metrics["time_to_first_instru_from_anchor_job_seconds"] is None else str(run_metrics["time_to_first_instru_from_anchor_job_seconds"]),
                "time_to_first_instru_from_anchor_job_quality": run_metrics["time_to_first_instru_from_anchor_job_quality"],
            })

            style_rows = build_run_per_style_rows(
                full_name=full_name,
                workflow_identifier=workflow_identifier,
                workflow_id=workflow_id,
                workflow_path=workflow_path,
                run=run,
                declared_styles=declared_styles,
                anchor_step_names=anchor_step_names,
                jobs=jobs,
                run_start_eff=run_start_eff,
                run_end_eff=run_end_eff,
                run_duration_eff=run_dur_eff,
                run_timing_source=run_timing_source,
            )
            for sr in style_rows:
                append_row(OUT_RUN_PER_STYLE_CSV, out_style_fields, sr)

            existing_run_ids.add(run_id)

        time.sleep(SLEEP_BETWEEN_WORKFLOWS_SEC)

    print("Done.")
    print("Wrote run-level inventory:", OUT_RUN_INVENTORY_CSV)
    print("Wrote run×style Layer-1 envelope inventory:", OUT_RUN_PER_STYLE_CSV)


if __name__ == "__main__":
    main()

Stage2 V18: workflows -> runs: 100%|██████████| 8/8 [1:21:50<00:00, 613.80s/it]   

Done.
Wrote run-level inventory: C:\Android Mobile App\ICST2026_Ext\run_inventory.csv
Wrote run×style Layer-1 envelope inventory: C:\Android Mobile App\ICST2026_Ext\run_inventory_per_style.csv


## Stage 3 — Extract step telemetry, derive TTFTS, and enhance run metrics

In [11]:
# ============================================================
# Stage 3 V18 — Current Study Plan Aligned + Auxiliary Complexity Support
# ADJUSTED FOR MULTI-STYLE STYLE-SCOPED LAYER-2 ANCHORING
#
# Current study alignment
# ------------------------------------------------------------
# Layer 1 (broad, carried from Stage 2)
#   Run Duration
# = Time to Instrumentation Envelope
# + Instrumentation Job Envelope
# + Post-Instrumentation Tail
#
# Layer 2 (precise, measured here from step telemetry)
#   Run Duration
# = Pre-Invocation
# + Invocation Execution Window
# + Post-Invocation
#
# where:
#   Pre-Invocation
#     = run start -> matched invocation step start
#
#   Invocation Execution Window
#     = matched invocation step start
#       -> last execution-related step end in the invocation-centered path
#
#   Post-Invocation
#     = invocation execution end -> run completion
#
# Current study scope
# - Stage 2 = instrumentation-capable inventory
# - Stage 3 = instrumentation-executed subset only
# - authoritative Stage 3 gate:
#     style_instru_job_count > 0
# - study-facing styles only:
#     Community, Custom, GMD, Third-Party
# - Real-Device excluded from Stage 3 study outputs
#
# V18 additions
# ------------------------------------------------------------
# - Carries Stage 2 V18 auxiliary complexity fields into Stage 3 outputs
# - Adds Stage 3 auxiliary fields for:
#     * invocation candidate multiplicity
#     * selected invocation priority source
#     * execution-window candidate multiplicity
#     * cross-job same-style execution window visibility
#
# Multi-style adjustment in this version
# ------------------------------------------------------------
# - Invocation-step selection is style-scoped using Stage 2 style job inventory
# - Execution-related classification is style-scoped for multi-style runs
# - Workflow YAML step matching is made more robust for nested runtime job names
#   such as "caller / android" -> "android"
#
# Outputs
# - run_metrics_v16_stage3_enhanced.csv
# - run_steps_v16_stage3_breakdown.csv
# - run_per_style_v1_stage3.csv
#
# Notes
# - No TTFTS
# - No fallback timing fields
# - No legacy measured aliases
# - run_per_style explicitly includes the exact cutpoints used
#   for Layer 2 so validation can be performed directly
# ============================================================

import base64
import csv
import random
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union

import requests

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None


# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_STAGE2_RUN_CSV = ROOT_DIR / "run_inventory.csv"
IN_STAGE2_PER_STYLE_CSV = ROOT_DIR / "run_inventory_per_style.csv"

OUT_STAGE3A_RUNS_CSV = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"
OUT_STAGE3B_STEPS_CSV = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"
OUT_STAGE3C_RUN_PER_STYLE_CSV = ROOT_DIR / "run_per_style_v1_stage3.csv"

MAX_TOKENS_TO_USE = 7
PROCESS_ONLY_RELEVANT_ROWS = True

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 60
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 7000


# =========================
# Helpers
# =========================
BOM = "\ufeff"
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")


STYLE_CANONICAL = ["Community", "Custom", "GMD", "Third-Party"]
STYLE_ALIASES = {
    "community": "Community",
    "emu community": "Community",
    "emulator community": "Community",

    "custom": "Custom",
    "emu custom": "Custom",
    "emulator custom": "Custom",

    "gmd": "GMD",

    "third party": "Third-Party",
    "third-party": "Third-Party",
    "third_party": "Third-Party",
    "thirdparty": "Third-Party",
    "3p": "Third-Party",

    "real device": "Real-Device",
    "real-device": "Real-Device",
    "real_device": "Real-Device",
    "realdevice": "Real-Device",
    "real devices": "Real-Device",
    "real-devices": "Real-Device",
    "real_devices": "Real-Device",
    "realdevices": "Real-Device",
}


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def norm(s: Optional[str]) -> str:
    return (s or "").strip()


def low(s: Optional[str]) -> str:
    return norm(s).lower()


def canon_key(s: Optional[str]) -> str:
    s2 = low(s)
    s2 = s2.replace("_", " ").replace("-", " ")
    s2 = re.sub(r"\s+", " ", s2).strip()
    return s2


def normalize_style_label(s: Optional[str]) -> str:
    return STYLE_ALIASES.get(canon_key(s), norm(s))


def split_styles(s: Optional[str]) -> List[str]:
    raw = norm(s)
    if not raw:
        return []
    parts = [normalize_style_label(x) for x in re.split(r"[|,;/]+", raw) if norm(x)]
    return unique_preserve([p for p in parts if p])


def first_nonempty_value(row: Dict[str, str], keys: List[str]) -> str:
    for k in keys:
        v = norm(row.get(k))
        if v:
            return v
    return ""


def first_nonempty_value_no_fallback(style_row: Dict[str, str], run_row: Dict[str, str], keys: List[str]) -> str:
    """
    Prefer style-row value. If absent, use run-row value only for the SAME field family.
    No semantic cross-field fallback.
    """
    for k in keys:
        v = norm(style_row.get(k))
        if v:
            return v
    for k in keys:
        v = norm(run_row.get(k))
        if v:
            return v
    return ""


def sanitize_gha_expr(text: str) -> str:
    return GHA_EXPR_RE.sub("", text or "")


def iso_to_dt(iso: Optional[str]) -> Optional[datetime]:
    if not iso:
        return None
    try:
        return datetime.fromisoformat(str(iso).replace("Z", "+00:00"))
    except Exception:
        return None


def dt_to_iso_z(dt: Optional[datetime]) -> str:
    if not dt:
        return ""
    return dt.isoformat().replace("+00:00", "Z")


def dt_to_seconds(a: Optional[datetime], b: Optional[datetime]) -> Optional[int]:
    if not a or not b:
        return None
    try:
        sec = int((b - a).total_seconds())
        return sec if sec >= 0 else None
    except Exception:
        return None


def safe_int_from_str(x: Optional[str]) -> Optional[int]:
    try:
        if x is None or str(x).strip() == "":
            return None
        return int(float(str(x).strip()))
    except Exception:
        return None


def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen: Set[str] = set()
    out: List[str] = []
    for x in seq:
        sx = norm(str(x))
        if not sx:
            continue
        if sx not in seen:
            seen.add(sx)
            out.append(sx)
    return out


def safe_join_names(names: List[str], max_len: int = 1000) -> str:
    s = ",".join(unique_preserve(names))
    if len(s) <= max_len:
        return s
    return s[: max_len - 3] + "..."


def read_env_tokens(path: Path) -> List[str]:
    toks: List[str] = []
    if not path.exists():
        return toks
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        if k.strip().startswith("GITHUB_TOKEN"):
            tok = v.strip().strip('"').strip("'")
            if tok:
                toks.append(tok)
    return toks[:MAX_TOKENS_TO_USE]


def read_csv_rows(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        rdr = csv.DictReader(f)
        out = []
        for r in rdr:
            clean = {}
            for k, v in r.items():
                kk = (k or "").replace(BOM, "").strip()
                clean[kk] = v or ""
            out.append(clean)
        return out


def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in fieldnames})


def parse_repo(repo_full_name: str) -> Tuple[str, str]:
    parts = repo_full_name.split("/")
    if len(parts) != 2:
        raise ValueError(f"Bad full_name: {repo_full_name}")
    return parts[0], parts[1]


def resolve_run_attempt(style_row: Dict[str, str], run_row: Dict[str, str]) -> str:
    return first_nonempty_value_no_fallback(style_row, run_row, ["run_attempt"])


def resolve_run_status(style_row: Dict[str, str], run_row: Dict[str, str]) -> str:
    return first_nonempty_value_no_fallback(style_row, run_row, ["status"])


def resolve_run_conclusion(style_row: Dict[str, str], run_row: Dict[str, str]) -> str:
    return first_nonempty_value_no_fallback(
        style_row,
        run_row,
        ["run_conclusion", "conclusion", "workflow_conclusion"],
    )


def resolve_event(style_row: Dict[str, str], run_row: Dict[str, str]) -> str:
    return first_nonempty_value_no_fallback(style_row, run_row, ["event"])


def resolve_trigger(style_row: Dict[str, str], run_row: Dict[str, str]) -> str:
    return first_nonempty_value_no_fallback(style_row, run_row, ["trigger"])


# =========================
# Executed-run gating
# =========================
def row_is_instru_executed(row: Dict[str, str]) -> bool:
    c = safe_int_from_str(row.get("style_instru_job_count"))
    return c is not None and c > 0


# =========================
# Stage 4 compatibility support
# =========================
def detect_runner_os_from_job(job: dict) -> str:
    labels = job.get("labels") or []
    if isinstance(labels, list):
        labels_joined = " ".join(str(x).strip() for x in labels if str(x).strip())
        if re.search(r"\bubuntu\b|\blinux\b", labels_joined, re.I):
            return "ubuntu"
        if re.search(r"\bmacos\b|\bosx\b|\bmac\b", labels_joined, re.I):
            return "macos"
        if re.search(r"\bwindows\b|\bwin(dows)?\b", labels_joined, re.I):
            return "windows"

    runner_name = norm(job.get("runner_name"))
    if re.search(r"\bubuntu\b|\blinux\b", runner_name, re.I):
        return "ubuntu"
    if re.search(r"\bmacos\b|\bosx\b|\bmac\b", runner_name, re.I):
        return "macos"
    if re.search(r"\bwindows\b|\bwin(dows)?\b", runner_name, re.I):
        return "windows"
    return ""


def stage4_compatible_run_support_fields(row: Dict[str, str], jobs: List[dict]) -> Dict[str, object]:
    out: Dict[str, object] = {}

    workflow_identifier = first_nonempty_value(row, ["workflow_identifier", "workflow_id", "workflow_path"])
    head_sha = first_nonempty_value(row, ["head_sha"])
    effective_ref = head_sha

    style_instru_job_count = first_nonempty_value(row, ["style_instru_job_count"])
    run_style_instru_job_count = first_nonempty_value(row, ["instru_job_count"])

    runner_os = first_nonempty_value(row, ["runner_os", "runs_on", "os"])
    runs_on = first_nonempty_value(row, ["runs_on"])
    os_val = first_nonempty_value(row, ["os"])
    runner_labels = first_nonempty_value(row, ["runner_labels"])

    if not runner_os:
        derived_runner_os_vals = unique_preserve([detect_runner_os_from_job(j) for j in jobs if detect_runner_os_from_job(j)])
        if len(derived_runner_os_vals) == 1:
            runner_os = derived_runner_os_vals[0]
        elif len(derived_runner_os_vals) > 1:
            runner_os = "mixed_or_unknown"

    if not runner_labels:
        all_labels: List[str] = []
        for j in jobs:
            labels = j.get("labels") or []
            if isinstance(labels, list):
                all_labels.extend([str(x).strip() for x in labels if str(x).strip()])
        runner_labels = safe_join_names(unique_preserve(all_labels), max_len=800)

    derived_job_count = len(jobs) if jobs else ""

    job_count_total = first_nonempty_value(row, ["job_count_total", "jobs_total", "total_jobs", "jobs_count"])
    if not job_count_total and derived_job_count != "":
        job_count_total = str(derived_job_count)

    out["workflow_identifier"] = workflow_identifier
    out["head_sha"] = head_sha
    out["effective_ref_for_stage4"] = effective_ref
    out["instru_job_count"] = run_style_instru_job_count or style_instru_job_count

    out["runner_os"] = runner_os
    out["runs_on"] = runs_on
    out["os"] = os_val
    out["runner_labels"] = runner_labels

    out["job_count_total"] = job_count_total
    out["jobs_total"] = first_nonempty_value(row, ["jobs_total"]) or job_count_total
    out["total_jobs"] = first_nonempty_value(row, ["total_jobs"]) or job_count_total
    out["jobs_count"] = first_nonempty_value(row, ["jobs_count"]) or job_count_total

    return out


# =========================
# GitHub client
# =========================
@dataclass
class GitHubClient:
    tokens: List[str]
    idx: int = 0

    def __post_init__(self):
        if not self.tokens:
            raise RuntimeError("No GitHub tokens found.")

    def _headers(self) -> Dict[str, str]:
        return {
            "Authorization": f"token {self.tokens[self.idx]}",
            "Accept": "application/vnd.github+json",
            "User-Agent": "ICST2026-Stage3-V18",
        }

    def _rotate(self):
        self.idx = (self.idx + 1) % len(self.tokens)

    def get(self, url: str, stream: bool = False) -> requests.Response:
        last_exc = None
        for attempt in range(MAX_RETRIES_PER_REQUEST):
            try:
                r = requests.get(
                    url,
                    headers=self._headers(),
                    timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                    stream=stream,
                )
                if r.status_code in (403, 429):
                    self._rotate()
                    time.sleep(min(BACKOFF_CAP_S, BACKOFF_BASE_S ** attempt) + random.random())
                    continue
                if r.status_code >= 500:
                    time.sleep(min(BACKOFF_CAP_S, BACKOFF_BASE_S ** attempt) + random.random())
                    continue
                return r
            except requests.RequestException as e:
                last_exc = e
                time.sleep(min(BACKOFF_CAP_S, BACKOFF_BASE_S ** attempt) + random.random())
        if last_exc:
            raise last_exc
        raise RuntimeError(f"Failed GET: {url}")


# =========================
# YAML / API caches
# =========================
WORKFLOW_YAML_CACHE: Dict[Tuple[str, str, str], str] = {}
JOBS_CACHE: Dict[Tuple[str, str], List[dict]] = {}


def gh_contents_raw(gh: GitHubClient, owner: str, repo: str, path: str, ref: str) -> Optional[str]:
    key = (f"{owner}/{repo}", path, ref)
    if key in WORKFLOW_YAML_CACHE:
        return WORKFLOW_YAML_CACHE[key]

    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    r = gh.get(url)
    if r.status_code == 404:
        return None
    if r.status_code != 200:
        return None

    js = r.json()
    if not isinstance(js, dict):
        return None
    if js.get("type") != "file":
        return None

    content = js.get("content", "")
    encoding = js.get("encoding", "")
    if encoding == "base64":
        try:
            txt = base64.b64decode(content).decode("utf-8", errors="ignore")
        except Exception:
            return None
    else:
        dl = js.get("download_url")
        if not dl:
            return None
        rr = gh.get(dl)
        if rr.status_code != 200:
            return None
        txt = rr.text

    if len(WORKFLOW_YAML_CACHE) < WORKFLOW_YAML_CACHE_MAX:
        WORKFLOW_YAML_CACHE[key] = txt
    return txt


# =========================
# Stage 1 / called-file parsing helpers
# =========================
def split_multi_value_cell(s: Optional[str]) -> List[str]:
    raw = norm(s)
    if not raw:
        return []
    if "||" in raw:
        parts = [x.strip() for x in raw.split("||")]
    elif "|" in raw:
        parts = [x.strip() for x in raw.split("|")]
    elif ";" in raw:
        parts = [x.strip() for x in raw.split(";")]
    else:
        parts = [x.strip() for x in raw.split(",")]
    return [p for p in parts if p]


def parse_stage1_confirmed_called_file_paths(row: Dict[str, str]) -> List[str]:
    signal = low(row.get("called_instru_signal"))
    if signal not in {"true", "1", "yes", "y"}:
        return []
    paths = split_multi_value_cell(row.get("called_instru_file_paths"))
    cleaned: List[str] = []
    for p in paths:
        p2 = p.strip().replace("\\", "/")
        if p2.startswith("./"):
            p2 = p2[2:]
        if p2:
            cleaned.append(p2)
    return unique_preserve(cleaned)


def parse_stage1_confirmed_called_origins(row: Dict[str, str]) -> Set[str]:
    vals = split_multi_value_cell(row.get("called_instru_origin_step_names"))
    return set(low(v) for v in vals if norm(v))


# =========================
# Workflow YAML extraction
# =========================
STEP_NAME_RE = re.compile(r"^\s*-\s*name\s*:\s*(.+?)\s*$")
STEP_USES_RE = re.compile(r"^\s*uses\s*:\s*(.+?)\s*$")
STEP_RUN_RE = re.compile(r"^\s*run\s*:\s*(.*)$")
JOB_ID_RE = re.compile(r"^\s{2}([A-Za-z0-9_.-]+)\s*:\s*$")


def strip_quotes(s: str) -> str:
    s = s.strip()
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        return s[1:-1]
    return s


def extract_steps_from_workflow_yaml(text: str) -> List[Dict[str, str]]:
    lines = text.splitlines()
    out: List[Dict[str, str]] = []
    current_job = ""
    i = 0
    in_steps_block = False

    while i < len(lines):
        line = lines[i]
        m_job = JOB_ID_RE.match(line)
        if m_job:
            current_job = m_job.group(1).strip()
            in_steps_block = False

        if re.match(r"^\s*steps\s*:\s*$", line):
            in_steps_block = True
            i += 1
            continue

        if in_steps_block:
            m_name = STEP_NAME_RE.match(line)
            if m_name:
                current_step_name = strip_quotes(m_name.group(1).strip())
                out.append({"job_name": current_job, "step_name": current_step_name, "uses": "", "run": ""})
                i += 1
                continue

            if out:
                m_uses = STEP_USES_RE.match(line.strip())
                if m_uses:
                    out[-1]["uses"] = strip_quotes(m_uses.group(1).strip())
                m_run = STEP_RUN_RE.match(line.strip())
                if m_run and out[-1].get("run", "") == "":
                    out[-1]["run"] = m_run.group(1)
        i += 1

    return out


# =========================
# Heuristics / classification
# =========================
_NORM_WS_RE = re.compile(r"\s+")
_NORM_PUNCT_RE = re.compile(r"[\[\]\(\)\{\}:;|]+")
_JOB_MATRIX_SUFFIX_RE = re.compile(r"\s*\([^)]*\)\s*$")


def normalize_name(s: str) -> str:
    x = (s or "").strip().strip('"').strip("'").lower()
    x = _NORM_PUNCT_RE.sub(" ", x)
    x = _NORM_WS_RE.sub(" ", x).strip()
    return x


def job_base_name(s: str) -> str:
    return _JOB_MATRIX_SUFFIX_RE.sub("", norm(s)).strip()


def runtime_job_matches_yaml_job(runtime_job_name: str, yaml_job_name: str) -> bool:
    rt = low(runtime_job_name)
    y = low(yaml_job_name)
    if not rt or not y:
        return False
    if rt == y:
        return True
    rt_parts = [p.strip() for p in rt.split("/") if p.strip()]
    if rt_parts and rt_parts[-1] == y:
        return True
    return False


def step_in_style_job_scope(step_job_name: str, style_job_names: Set[str], invocation_job_name: str = "") -> bool:
    sj = norm(step_job_name)
    if not style_job_names:
        return True

    if sj and sj in style_job_names:
        return True

    sj_base = job_base_name(sj)
    style_bases = {job_base_name(x) for x in style_job_names if norm(x)}
    if sj_base and sj_base in style_bases:
        return True

    if invocation_job_name:
        ij = norm(invocation_job_name)
        if sj and ij and (sj == ij or job_base_name(sj) == job_base_name(ij)):
            return True

    return False


def parse_anchor_step_names(csv_value: str) -> List[str]:
    if not csv_value:
        return []
    raw = [p.strip() for p in str(csv_value).split(",")]
    return unique_preserve([r for r in raw if r])


def anchored_step_match(runtime_step_name: str, anchor_names: List[str]) -> bool:
    rn = normalize_name(runtime_step_name)
    if not rn:
        return False
    for a in anchor_names:
        an = normalize_name(a)
        if not an:
            continue
        if rn == an or an in rn or rn in an:
            return True
    return False


ANDROIDISH_RE = re.compile(
    r"(adb|am instrument|avd|emulator|uiautomator|espresso|androidtest|connectedcheck|connectedandroidtest|manageddevice|gmd|"
    r"detox|baseline profile|baselineprofile|macrobenchmark|instrumentation|integration[\s_-]*test)",
    re.I,
)

THIRD_PARTY_PROVIDER_RE = re.compile(
    r"(browserstack|bstack|sauce\s*labs|saucelabs|kobiton|headspin|bitbar|perfecto|lambdatest|genymotion\s*cloud|firebase\s*test\s*lab|gcloud\s+firebase|emulator\.wtf|maestro\s+cloud)",
    re.I,
)

THIRD_PARTY_INVOKE_RE = re.compile(
    r"(firebase\s+test\s+android\s+run|gcloud\s+firebase\s+test\s+android\s+run|flank\s+android\s+run|"
    r"appcenter\s+test\s+run|saucectl\s+(run|test)|maestro\s+cloud|detox\s+test|browserstack)",
    re.I,
)

SETUP_HINT_RE = re.compile(
    r"(checkout|setup-java|setup java|setup-jdk|setup android|sdkmanager|cache|restore cache|install dependencies|npm ci|yarn install|bundle install|gradle dependencies)",
    re.I,
)

PROVISION_HINT_RE = re.compile(
    r"(create avd|avdmanager|start emulator|launch emulator|boot emulator|wait[- ]for[- ]device|provision|device farm|test lab|browserstack local|sauce connect)",
    re.I,
)

ARTIFACT_HINT_RE = re.compile(
    r"(upload-artifact|download-artifact|artifact|test-results|results|report|reports|junit|coverage|logs?)",
    re.I,
)

CLEANUP_HINT_RE = re.compile(
    r"(cleanup|tear\s*down|teardown|stop emulator|kill emulator|shutdown emulator|remove avd|delete avd|close session|stop session)",
    re.I,
)

TEST_HINT_RE = re.compile(
    r"(connectedcheck|connectedandroidtest|androidtest|am instrument|instrumentation|manageddevice|gmd|"
    r"detox|baselineprofile|baseline profile|macrobenchmark|uiautomator|espresso|integration[\s_-]*test|firebase\s+test\s+android\s+run|gcloud\s+firebase)",
    re.I,
)

FILE_HINT_INSTRU_RE = re.compile(
    r"(androidtest|connectedcheck|connectedandroidtest|detox|integration[\s_-]*test|managed[\s_-]*device|gmd|baseline[\s_-]*profile|macrobenchmark|espresso|uiautomator|instrumentation)",
    re.I,
)

CUSTOM_SCRIPT_HINT_RE = re.compile(
    r"(./gradlew|gradlew|python|bash|sh |pwsh|powershell|node |npm |yarn |ruby |bundle exec)",
    re.I,
)

INVOCATION_START_STRONG_RE = re.compile(
    r"(adb.*am instrument|am instrument|connectedcheck|connectedandroidtest|androidtest|"
    r"run flutter integration tests|run android emulator|integration[\s_-]*tests?|"
    r"detox\s+test|instrumentation|manageddevice|gmd|baseline[\s_-]*profile|"
    r"macrobenchmark|uiautomator|espresso|firebase\s+test\s+android\s+run|"
    r"gcloud\s+firebase\s+test\s+android\s+run)",
    re.I,
)

INVOCATION_WRAPPER_RE = re.compile(
    r"(cache|restore cache|save cache|avd cache|download artifacts?|download artifact|"
    r"prepare test apk|prepare apk|prepare|setup ssh|wait[- ]for[- ]device|"
    r"boot emulator|start emulator|launch emulator|create avd|provision)",
    re.I,
)

POST_PHASE_HINT_RE = re.compile(
    r"(^post\b|post |complete job|upload artifacts?|download artifacts?|upload|download|artifact|report|reports|teardown|tear down|cleanup|close session)",
    re.I,
)

JOB_TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

JOB_PATH_GENERIC_TOKENS = {
    "call",
    "run",
    "test",
    "tests",
    "job",
    "jobs",
    "setup",
    "build",
    "debug",
    "release",
    "ci",
    "workflow",
    "workflows",
    "min",
    "max",
}


def infer_flags_from_step(step_name: str, uses: str, run_cmd: str, target_style: str = "") -> Dict[str, Union[bool, str]]:
    sname = norm(step_name)
    suse = norm(uses)
    srun = sanitize_gha_expr(norm(run_cmd))
    combo = " | ".join([sname, suse, srun])
    style = normalize_style_label(target_style)

    flags: Dict[str, Union[bool, str]] = {
        "stage1_anchor_match": False,
        "stage1_anchor_match_reason": "",

        "explicit_instru": False,
        "explicit_instru_reason": "",

        "setup": False,
        "provision": False,
        "artifact_report": False,
        "cleanup_teardown": False,

        "third_party_provider": False,
        "third_party_provider_name": "",
        "third_party_invoke": False,

        "gmd": False,
        "community": False,
        "custom": False,

        "custom_followed_file_instru": False,
        "custom_stage1_supported_exec": False,
    }

    if SETUP_HINT_RE.search(combo):
        flags["setup"] = True
    if PROVISION_HINT_RE.search(combo):
        flags["provision"] = True
    if ARTIFACT_HINT_RE.search(combo):
        flags["artifact_report"] = True
    if CLEANUP_HINT_RE.search(combo):
        flags["cleanup_teardown"] = True

    if THIRD_PARTY_PROVIDER_RE.search(combo):
        flags["third_party_provider"] = True
        m = THIRD_PARTY_PROVIDER_RE.search(combo)
        flags["third_party_provider_name"] = m.group(1) if m else ""
    if THIRD_PARTY_INVOKE_RE.search(combo):
        flags["third_party_invoke"] = True

    if re.search(r"(manageddevice|gmd|gradle managed device)", combo, re.I):
        flags["gmd"] = True
    if re.search(r"(android-emulator-runner|emulator runner|create avd|start emulator|connectedcheck|connectedandroidtest)", combo, re.I):
        flags["community"] = True
    if re.search(r"(detox|integration[\s_-]*test|baseline profile|baselineprofile|macrobenchmark|uiautomator|espresso|instrumentation)", combo, re.I):
        flags["custom"] = True

    if TEST_HINT_RE.search(combo) or ANDROIDISH_RE.search(combo):
        flags["explicit_instru"] = True
        flags["explicit_instru_reason"] = "androidish_or_test_hint"

    if style == "Third-Party":
        if bool(flags.get("third_party_provider")) or bool(flags.get("third_party_invoke")):
            flags["explicit_instru"] = True
            if not flags["explicit_instru_reason"]:
                flags["explicit_instru_reason"] = "third_party"
    elif style == "GMD":
        if bool(flags.get("gmd")):
            flags["explicit_instru"] = True
            if not flags["explicit_instru_reason"]:
                flags["explicit_instru_reason"] = "gmd"
    elif style == "Community":
        if bool(flags.get("community")):
            flags["explicit_instru"] = True
            if not flags["explicit_instru_reason"]:
                flags["explicit_instru_reason"] = "community"
    elif style == "Custom":
        if bool(flags.get("custom")):
            flags["explicit_instru"] = True
            if not flags["explicit_instru_reason"]:
                flags["explicit_instru_reason"] = "custom"

    return flags


def build_step_combo(step_name: str, uses: str, run_cmd: str) -> str:
    return " | ".join([
        norm(step_name),
        norm(uses),
        sanitize_gha_expr(norm(run_cmd)),
    ])


def is_strong_invocation_candidate(
    flags: Dict[str, Union[bool, str]],
    step_name: str,
    uses: str,
    run_cmd: str,
) -> bool:
    combo = build_step_combo(step_name, uses, run_cmd)

    if bool(flags.get("stage1_anchor_match")):
        return True

    if bool(flags.get("custom_stage1_supported_exec")):
        return True

    if INVOCATION_START_STRONG_RE.search(combo):
        return True

    if INVOCATION_WRAPPER_RE.search(combo):
        return False

    if bool(flags.get("explicit_instru")) and not (
        bool(flags.get("setup"))
        or bool(flags.get("provision"))
        or bool(flags.get("artifact_report"))
        or bool(flags.get("cleanup_teardown"))
    ):
        return True

    return False


def tokenize_job_for_path(job_name: str) -> List[str]:
    toks: List[str] = []
    for tok in JOB_TOKEN_RE.findall(low(job_name)):
        if not tok or tok.isdigit():
            continue
        if tok in JOB_PATH_GENERIC_TOKENS:
            continue
        toks.append(tok)
    return unique_preserve(toks)


def job_path_overlap_count(job_a: str, job_b: str) -> int:
    a = set(tokenize_job_for_path(job_a))
    b = set(tokenize_job_for_path(job_b))
    if not a or not b:
        return 0
    return len(a & b)


def execution_candidate_path_linked(step_job_name: str, invocation_job_name: str) -> bool:
    sj = norm(step_job_name)
    ij = norm(invocation_job_name)
    if not sj or not ij:
        return False

    if sj == ij or job_base_name(sj) == job_base_name(ij):
        return True

    return job_path_overlap_count(sj, ij) >= 1


def normalize_platform_value(s: str) -> str:
    x = low(s)
    if not x:
        return ""

    # Prefer target/test family first
    if "android" in x:
        return "android"
    if "ios" in x:
        return "ios"
    if "web" in x:
        return "web"

    # Fallback host runner family
    if "ubuntu" in x or "linux" in x:
        return "linux"
    if "windows" in x or re.search(r"\bwin\b", x):
        return "windows"
    if "macos" in x or "osx" in x or re.search(r"\bmac\b", x):
        return "macos"

    return x


def infer_platform_from_job_name(job_name: str) -> str:
    j = low(job_name)
    if not j:
        return ""

    # Target platform first
    if "android" in j:
        return "android"
    if "ios" in j:
        return "ios"
    if "web" in j:
        return "web"

    # Host fallback only if target family is absent
    if "windows" in j or re.search(r"\bwin\b", j):
        return "windows"
    if "macos" in j or "osx" in j or re.search(r"\bmac\b", j):
        return "macos"
    if "linux" in j or "ubuntu" in j:
        return "linux"

    return ""


def get_step_platform(step_row: Dict[str, object]) -> str:
    """
    For Layer 2 continuity, prefer the TEST TARGET family
    (android / ios / web) over the host runner family.
    """
    job_platform = infer_platform_from_job_name(str(step_row.get("job_name", "")))
    if job_platform:
        return job_platform

    runner_os = normalize_platform_value(str(step_row.get("runner_os", "")))
    if runner_os:
        return runner_os

    runner_name = normalize_platform_value(str(step_row.get("runner_name", "")))
    if runner_name:
        return runner_name

    runner_labels = normalize_platform_value(str(step_row.get("runner_labels", "")))
    if runner_labels:
        return runner_labels

    return ""


def same_platform_as_invocation(step_row: Dict[str, object], invocation_platform: str) -> bool:
    if not invocation_platform:
        return False
    return get_step_platform(step_row) == invocation_platform


def mark_stage1_anchor_matches(
    steps: List[Dict[str, object]],
    stage1_anchor_names: List[str],
    called_origin_step_names: Set[str],
) -> None:
    for st in steps:
        step_low = low(str(st.get("step_name", "")))
        if stage1_anchor_names and anchored_step_match(str(st.get("step_name", "")), stage1_anchor_names):
            st["stage1_anchor_match"] = True
            st["stage1_anchor_match_reason"] = "stage1_test_invocation_step_names"
        elif step_low and step_low in called_origin_step_names:
            st["stage1_anchor_match"] = True
            st["stage1_anchor_match_reason"] = "matched_stage1_called_origin"
        else:
            st["stage1_anchor_match"] = False
            st["stage1_anchor_match_reason"] = ""


def enrich_with_called_file_support(
    gh: GitHubClient,
    owner: str,
    repo: str,
    ref: str,
    steps: List[Dict[str, object]],
    confirmed_called_paths: List[str],
    called_origin_step_names: Set[str],
    target_style: str,
) -> None:
    if not confirmed_called_paths:
        return

    file_evidence_map: Dict[str, bool] = {}
    for p in confirmed_called_paths:
        txt = gh_contents_raw(gh, owner, repo, p, ref)
        if not txt:
            continue
        file_evidence_map[p] = bool(FILE_HINT_INSTRU_RE.search(txt))

    if not file_evidence_map:
        return

    for st in steps:
        step_low = low(str(st.get("step_name", "")))
        uses_low = low(str(st.get("uses", "")))
        run_low = low(str(st.get("run", "")))
        combo = " | ".join([step_low, uses_low, run_low])

        matched_origin = step_low in called_origin_step_names if step_low else False
        matched_local_ref = any(p in combo for p in file_evidence_map.keys())

        if matched_origin or matched_local_ref:
            if any(file_evidence_map.values()):
                st["custom_followed_file_instru"] = True
                if normalize_style_label(target_style) == "Custom":
                    if CUSTOM_SCRIPT_HINT_RE.search(combo) or FILE_HINT_INSTRU_RE.search(combo):
                        st["custom_stage1_supported_exec"] = True


# =========================
# Jobs / steps API
# =========================
def list_jobs_for_run(gh: GitHubClient, owner: str, repo: str, run_id: str) -> List[dict]:
    cache_key = (f"{owner}/{repo}", run_id)
    if cache_key in JOBS_CACHE:
        return JOBS_CACHE[cache_key]

    jobs: List[dict] = []
    page = 1
    while page <= MAX_PAGES_PER_LIST:
        url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}/jobs?per_page=100&page={page}"
        r = gh.get(url)
        if r.status_code != 200:
            break
        js = r.json() if r.text else {}
        arr = js.get("jobs", [])
        if not arr:
            break
        jobs.extend(arr)
        if len(arr) < 100:
            break
        page += 1

    JOBS_CACHE[cache_key] = jobs
    return jobs


def step_rows_from_jobs(jobs: List[dict]) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for job_idx, job in enumerate(jobs, start=1):
        jname = norm(job.get("name"))
        job_id = str(job.get("id") or "")
        job_url = norm(job.get("url"))
        job_html_url = norm(job.get("html_url"))
        runner_name = norm(job.get("runner_name"))
        runner_os = detect_runner_os_from_job(job)

        runner_labels = ""
        if isinstance(job.get("labels"), list):
            runner_labels = ",".join(str(x).strip() for x in job.get("labels", []) if str(x).strip())

        job_started_at = norm(job.get("started_at"))
        job_completed_at = norm(job.get("completed_at"))

        for step_idx, st in enumerate(job.get("steps") or [], start=1):
            started = st.get("started_at") or ""
            completed = st.get("completed_at") or ""
            sdt = iso_to_dt(started)
            edt = iso_to_dt(completed)
            dur = dt_to_seconds(sdt, edt)

            rows.append({
                "job_name": jname,
                "job_id": job_id,
                "job_url": job_url,
                "job_html_url": job_html_url,
                "job_started_at": job_started_at,
                "job_completed_at": job_completed_at,
                "job_ordinal_in_run": str(job_idx),
                "step_ordinal_in_job": str(step_idx),
                "runner_os": runner_os,
                "runner_labels": runner_labels,
                "runner_name": runner_name,
                "step_name": norm(st.get("name")),
                "status": norm(st.get("status")),
                "conclusion": norm(st.get("conclusion")),
                "started_at": started,
                "completed_at": completed,
                "duration_seconds": dur,
                "uses": "",
                "run": "",
            })
    return rows


# =========================
# Layer 2 classification
# =========================
def is_execution_related(
    flags: Dict[str, Union[bool, str]],
    target_style: str,
    step_job_name: str = "",
    style_job_names: Optional[Set[str]] = None,
    invocation_job_name: str = "",
) -> bool:
    style_job_names = style_job_names or set()

    if not step_in_style_job_scope(step_job_name, style_job_names, invocation_job_name):
        return False

    if bool(flags.get("stage1_anchor_match")):
        return True
    if bool(flags.get("explicit_instru")):
        return True
    if normalize_style_label(target_style) == "Custom" and bool(flags.get("custom_stage1_supported_exec")):
        return True
    return False


def classify_step_activity_group(flags: Dict[str, Union[bool, str]]) -> str:
    if bool(flags.get("artifact_report")):
        return "Artifact/Report"
    if bool(flags.get("cleanup_teardown")):
        return "Cleanup/Teardown"
    if bool(flags.get("setup")):
        return "Setup"
    if bool(flags.get("provision")):
        return "Provision"
    if bool(flags.get("explicit_instru")) or bool(flags.get("stage1_anchor_match")) or bool(flags.get("custom_stage1_supported_exec")):
        return "Test"
    return "Other"


def classify_execution_role(
    flags: Dict[str, Union[bool, str]],
    target_style: str,
    step_job_name: str = "",
    style_job_names: Optional[Set[str]] = None,
    invocation_job_name: str = "",
) -> str:
    return (
        "Execution-related"
        if is_execution_related(
            flags,
            target_style,
            step_job_name=step_job_name,
            style_job_names=style_job_names,
            invocation_job_name=invocation_job_name,
        )
        else "Non-execution overhead"
    )


def classify_overhead_phase(
    step_start: Optional[datetime],
    invocation_start: Optional[datetime],
    execution_window_end: Optional[datetime],
) -> str:
    if not step_start or not invocation_start:
        return "Pre-test overhead"
    if step_start < invocation_start:
        return "Pre-test overhead"
    if execution_window_end and step_start <= execution_window_end:
        return "Active test"
    return "Post-test overhead"


def infer_phase_metadata(
    step_row: Dict[str, object],
    flags: Dict[str, Union[bool, str]],
    step_activity_group: str,
    invocation_job_name: str,
    invocation_start_dt: Optional[datetime],
) -> Dict[str, object]:
    step_name = str(step_row.get("step_name", ""))
    uses = str(step_row.get("uses", ""))
    run_cmd = str(step_row.get("run", ""))
    job_name = str(step_row.get("job_name", ""))
    step_start = iso_to_dt(str(step_row.get("started_at", "")))

    combo = build_step_combo(step_name, uses, run_cmd)

    path_linked = False
    if invocation_job_name:
        path_linked = execution_candidate_path_linked(job_name, invocation_job_name)
    else:
        path_linked = bool(flags.get("stage1_anchor_match")) or bool(flags.get("explicit_instru")) or bool(flags.get("custom_stage1_supported_exec"))

    if not path_linked:
        phase_guess = "outside_selected_path"
        phase_reason = "not_selected_path"
        phase_score = 0
    elif bool(flags.get("artifact_report")) or bool(flags.get("cleanup_teardown")) or POST_PHASE_HINT_RE.search(combo):
        if bool(flags.get("artifact_report")) and bool(flags.get("cleanup_teardown")):
            phase_reason = "artifact+cleanup"
        elif bool(flags.get("artifact_report")):
            phase_reason = "artifact"
        elif bool(flags.get("cleanup_teardown")):
            phase_reason = "cleanup_teardown"
        else:
            phase_reason = "post_hint"
        phase_guess = "post_invocation"
        phase_score = 4
    elif bool(flags.get("stage1_anchor_match")):
        phase_guess = "invocation_execution"
        phase_reason = "stage1_anchor_match"
        phase_score = 5
    elif bool(flags.get("custom_stage1_supported_exec")):
        phase_guess = "invocation_execution"
        phase_reason = "custom_supported_exec"
        phase_score = 5
    elif bool(flags.get("explicit_instru")) and not (
        bool(flags.get("artifact_report")) or bool(flags.get("cleanup_teardown"))
    ):
        phase_guess = "invocation_execution"
        phase_reason = "explicit_instru"
        phase_score = 4
    elif bool(flags.get("setup")) or bool(flags.get("provision")) or INVOCATION_WRAPPER_RE.search(combo):
        if bool(flags.get("setup")) and bool(flags.get("provision")):
            phase_reason = "setup+provision"
        elif bool(flags.get("setup")):
            phase_reason = "setup"
        elif bool(flags.get("provision")):
            phase_reason = "provision"
        else:
            phase_reason = "wrapper_hint"
        phase_guess = "pre_invocation"
        phase_score = 3
    elif invocation_start_dt is not None and step_start is not None and step_start < invocation_start_dt:
        phase_guess = "pre_invocation"
        phase_reason = "before_invocation_start"
        phase_score = 2
    else:
        phase_guess = "pre_invocation"
        phase_reason = "path_scoped_fallback"
        phase_score = 1

    if phase_guess == "outside_selected_path":
        phase_group_consistency = "outside_selected_path"
    elif phase_guess == "pre_invocation":
        if step_activity_group in {"Setup", "Provision"}:
            phase_group_consistency = "consistent"
        elif step_activity_group in {"Other", "Test"}:
            phase_group_consistency = "soft_conflict"
        else:
            phase_group_consistency = "hard_conflict"
    elif phase_guess == "invocation_execution":
        if step_activity_group == "Test":
            phase_group_consistency = "consistent"
        elif step_activity_group in {"Setup", "Provision", "Other"}:
            phase_group_consistency = "soft_conflict"
        else:
            phase_group_consistency = "hard_conflict"
    else:  # post_invocation
        if step_activity_group in {"Artifact/Report", "Cleanup/Teardown", "Other"}:
            phase_group_consistency = "consistent"
        elif step_activity_group == "Test":
            phase_group_consistency = "soft_conflict"
        else:
            phase_group_consistency = "hard_conflict"

    return {
        "phase_guess": phase_guess,
        "phase_reason": phase_reason,
        "phase_score": phase_score,
        "phase_group_consistency": phase_group_consistency,
        "path_linked_to_selected_invocation": "true" if path_linked else "false",
    }


# =========================
# Layer 2 precise boundaries
# =========================
def parse_style_job_names(style_row: Dict[str, str]) -> Set[str]:
    names = split_multi_value_cell(style_row.get("style_instru_job_names"))
    return set(norm(x) for x in names if norm(x))


def make_cutpoint_record(
    step_row: Dict[str, object],
    source_type: str,
) -> Dict[str, object]:
    return {
        "step_name": str(step_row.get("step_name", "")),
        "job_name": str(step_row.get("job_name", "")),
        "source": source_type,
        "step_started_at": str(step_row.get("started_at", "")),
        "step_completed_at": str(step_row.get("completed_at", "")),
        "job_ordinal_in_run": str(step_row.get("job_ordinal_in_run", "")),
        "step_ordinal_in_job": str(step_row.get("step_ordinal_in_job", "")),
    }


def pick_measured_invocation_step(
    merged_steps: List[Dict[str, object]],
    target_style: str,
    style_job_names: Set[str],
) -> Tuple[Optional[datetime], Optional[datetime], Dict[str, object], Dict[str, Union[bool, str]]]:
    candidates_stage1: List[Tuple[datetime, datetime, Dict[str, object], Dict[str, Union[bool, str]]]] = []
    candidates_explicit_strong: List[Tuple[datetime, datetime, Dict[str, object], Dict[str, Union[bool, str]]]] = []
    candidates_explicit_weak: List[Tuple[datetime, datetime, Dict[str, object], Dict[str, Union[bool, str]]]] = []
    candidates_custom_supported: List[Tuple[datetime, datetime, Dict[str, object], Dict[str, Union[bool, str]]]] = []

    candidate_pool = [
        st for st in merged_steps
        if step_in_style_job_scope(str(st.get("job_name", "")), style_job_names)
    ]
    if not style_job_names:
        candidate_pool = merged_steps

    for st in candidate_pool:
        st_start = iso_to_dt(str(st.get("started_at", "")))
        st_end = iso_to_dt(str(st.get("completed_at", "")))
        if not st_start:
            continue

        step_name = str(st.get("step_name", ""))
        uses = str(st.get("uses", ""))
        run_cmd = str(st.get("run", ""))

        flags = infer_flags_from_step(
            step_name=step_name,
            uses=uses,
            run_cmd=run_cmd,
            target_style=target_style,
        )

        if bool(st.get("stage1_anchor_match")):
            flags["stage1_anchor_match"] = True
            flags["stage1_anchor_match_reason"] = st.get("stage1_anchor_match_reason", "")
        if bool(st.get("custom_followed_file_instru")):
            flags["custom_followed_file_instru"] = True
        if bool(st.get("custom_stage1_supported_exec")):
            flags["custom_stage1_supported_exec"] = True

        cutpoint = make_cutpoint_record(st, "")
        rec = (
            st_start,
            st_end if st_end else st_start,
            cutpoint,
            flags,
        )

        if bool(flags.get("stage1_anchor_match")):
            cutpoint["source"] = "stage1_anchor_match"
            candidates_stage1.append(rec)
            continue

        if normalize_style_label(target_style) == "Custom" and bool(flags.get("custom_stage1_supported_exec")):
            cutpoint["source"] = "stage1_supported_custom_exec"
            candidates_custom_supported.append(rec)
            continue

        if bool(flags.get("explicit_instru")):
            if is_strong_invocation_candidate(flags, step_name, uses, run_cmd):
                cutpoint["source"] = "explicit_instru_execution_start"
                candidates_explicit_strong.append(rec)
            else:
                cutpoint["source"] = "explicit_instru_step_fallback"
                candidates_explicit_weak.append(rec)

    if candidates_stage1:
        candidates_stage1.sort(key=lambda x: x[0])
        sdt, edt, cutpoint, flags = candidates_stage1[0]
        return sdt, edt, cutpoint, flags

    if candidates_explicit_strong:
        candidates_explicit_strong.sort(key=lambda x: x[0])
        sdt, edt, cutpoint, flags = candidates_explicit_strong[0]
        return sdt, edt, cutpoint, flags

    if candidates_custom_supported:
        candidates_custom_supported.sort(key=lambda x: x[0])
        sdt, edt, cutpoint, flags = candidates_custom_supported[0]
        return sdt, edt, cutpoint, flags

    if candidates_explicit_weak:
        candidates_explicit_weak.sort(key=lambda x: x[0])
        sdt, edt, cutpoint, flags = candidates_explicit_weak[0]
        return sdt, edt, cutpoint, flags

    return None, None, {
        "step_name": "",
        "job_name": "",
        "source": "missing",
        "step_started_at": "",
        "step_completed_at": "",
        "job_ordinal_in_run": "",
        "step_ordinal_in_job": "",
    }, {}


def is_execution_window_candidate(
    flags: Dict[str, Union[bool, str]],
    target_style: str,
    step_job_name: str,
    invocation_job_name: str,
    style_job_names: Set[str],
) -> bool:
    if not is_execution_related(
        flags=flags,
        target_style=target_style,
        step_job_name=step_job_name,
        style_job_names=style_job_names,
        invocation_job_name=invocation_job_name,
    ):
        return False

    style = normalize_style_label(target_style)
    same_job = bool(invocation_job_name) and (
        norm(step_job_name) == norm(invocation_job_name)
        or job_base_name(step_job_name) == job_base_name(invocation_job_name)
    )
    in_style_job = step_in_style_job_scope(step_job_name, style_job_names, invocation_job_name)

    if same_job:
        return True
    if in_style_job:
        return True
    if style == "Third-Party" and (bool(flags.get("third_party_provider")) or bool(flags.get("third_party_invoke"))):
        return True
    if style == "Custom" and bool(flags.get("custom_stage1_supported_exec")):
        return True

    return False


def collect_execution_window_candidates(
    merged_steps: List[Dict[str, object]],
    target_style: str,
    invocation_start_dt: Optional[datetime],
    invocation_job_name: str,
    style_job_names: Set[str],
    invocation_platform: str = "",
) -> List[Dict[str, object]]:
    if invocation_start_dt is None:
        return []

    candidates: List[Dict[str, object]] = []

    for st in merged_steps:
        st_start = iso_to_dt(str(st.get("started_at", "")))
        st_end = iso_to_dt(str(st.get("completed_at", "")))
        if not st_start or not st_end:
            continue
        if st_start < invocation_start_dt:
            continue

        step_name = str(st.get("step_name", ""))
        job_name = str(st.get("job_name", ""))

        flags = infer_flags_from_step(
            step_name=step_name,
            uses=str(st.get("uses", "")),
            run_cmd=str(st.get("run", "")),
            target_style=target_style,
        )
        if bool(st.get("stage1_anchor_match")):
            flags["stage1_anchor_match"] = True
            flags["stage1_anchor_match_reason"] = st.get("stage1_anchor_match_reason", "")
        if bool(st.get("custom_followed_file_instru")):
            flags["custom_followed_file_instru"] = True
        if bool(st.get("custom_stage1_supported_exec")):
            flags["custom_stage1_supported_exec"] = True

        if not is_execution_window_candidate(
            flags=flags,
            target_style=target_style,
            step_job_name=job_name,
            invocation_job_name=invocation_job_name,
            style_job_names=style_job_names,
        ):
            continue

        step_activity_group = classify_step_activity_group(flags)
        phase_meta = infer_phase_metadata(
            step_row=st,
            flags=flags,
            step_activity_group=step_activity_group,
            invocation_job_name=invocation_job_name,
            invocation_start_dt=invocation_start_dt,
        )

        if phase_meta["phase_guess"] in {"post_invocation", "outside_selected_path"}:
            continue

        path_linked = execution_candidate_path_linked(job_name, invocation_job_name)
        same_platform = same_platform_as_invocation(st, invocation_platform)

        cutpoint = make_cutpoint_record(
            st,
            "last_execution_related_step_path_linked_same_platform"
            if (path_linked and same_platform)
            else (
                "last_execution_related_step_path_linked"
                if path_linked
                else "last_execution_related_step_broad_scope"
            ),
        )

        candidates.append({
            "end_dt": st_end,
            "job_name": job_name,
            "path_linked": path_linked,
            "same_platform": same_platform,
            "platform": get_step_platform(st),
            "phase_guess": phase_meta["phase_guess"],
            "phase_score": int(phase_meta["phase_score"]),
            "phase_group_consistency": phase_meta["phase_group_consistency"],
            "cutpoint": cutpoint,
        })

    return candidates


def select_preferred_execution_window_candidates(
    candidates: List[Dict[str, object]],
) -> List[Dict[str, object]]:
    if not candidates:
        return []

    same_platform_and_path_exec = [
        c for c in candidates
        if bool(c.get("same_platform")) and bool(c.get("path_linked")) and c.get("phase_guess") == "invocation_execution"
    ]
    if same_platform_and_path_exec:
        return same_platform_and_path_exec

    same_platform_exec = [
        c for c in candidates
        if bool(c.get("same_platform")) and c.get("phase_guess") == "invocation_execution"
    ]
    if same_platform_exec:
        return same_platform_exec

    path_linked_exec = [
        c for c in candidates
        if bool(c.get("path_linked")) and c.get("phase_guess") == "invocation_execution"
    ]
    if path_linked_exec:
        return path_linked_exec

    same_platform_and_path = [
        c for c in candidates
        if bool(c.get("same_platform")) and bool(c.get("path_linked"))
    ]
    if same_platform_and_path:
        return same_platform_and_path

    same_platform_only = [
        c for c in candidates
        if bool(c.get("same_platform"))
    ]
    if same_platform_only:
        return same_platform_only

    path_linked_only = [
        c for c in candidates
        if bool(c.get("path_linked"))
    ]
    if path_linked_only:
        return path_linked_only

    return candidates


def pick_invocation_execution_end(
    merged_steps: List[Dict[str, object]],
    target_style: str,
    invocation_start_dt: Optional[datetime],
    invocation_end_dt: Optional[datetime],
    invocation_job_name: str,
    style_job_names: Set[str],
) -> Tuple[Optional[datetime], Dict[str, object]]:
    if invocation_start_dt is None:
        return None, {
            "step_name": "",
            "job_name": "",
            "source": "missing",
            "step_started_at": "",
            "step_completed_at": "",
            "job_ordinal_in_run": "",
            "step_ordinal_in_job": "",
        }

    invocation_platform = ""
    for st in merged_steps:
        if (
            norm(str(st.get("job_name", ""))) == norm(invocation_job_name)
            or job_base_name(str(st.get("job_name", ""))) == job_base_name(invocation_job_name)
        ):
            invocation_platform = get_step_platform(st)
            if invocation_platform:
                break

    all_candidates = collect_execution_window_candidates(
        merged_steps=merged_steps,
        target_style=target_style,
        invocation_start_dt=invocation_start_dt,
        invocation_job_name=invocation_job_name,
        style_job_names=style_job_names,
        invocation_platform=invocation_platform,
    )
    preferred_candidates = select_preferred_execution_window_candidates(all_candidates)

    if preferred_candidates:
        preferred_candidates.sort(key=lambda x: (x["phase_score"], x["end_dt"]))
        chosen = preferred_candidates[-1]
        return chosen["end_dt"], chosen["cutpoint"]

    if invocation_end_dt is not None:
        return invocation_end_dt, {
            "step_name": "",
            "job_name": invocation_job_name,
            "source": "invocation_step_terminal",
            "step_started_at": "",
            "step_completed_at": dt_to_iso_z(invocation_end_dt),
            "job_ordinal_in_run": "",
            "step_ordinal_in_job": "",
        }

    return None, {
        "step_name": "",
        "job_name": "",
        "source": "missing",
        "step_started_at": "",
        "step_completed_at": "",
        "job_ordinal_in_run": "",
        "step_ordinal_in_job": "",
    }


# =========================
# V18 auxiliary summary helpers
# =========================
def summarize_invocation_candidates(
    merged_steps: List[Dict[str, object]],
    target_style: str,
    style_job_names: Set[str],
) -> Dict[str, object]:
    stage1_candidates: List[Dict[str, object]] = []
    explicit_candidates: List[Dict[str, object]] = []
    custom_candidates: List[Dict[str, object]] = []

    candidate_pool = [
        st for st in merged_steps
        if step_in_style_job_scope(str(st.get("job_name", "")), style_job_names)
    ]
    if not style_job_names:
        candidate_pool = merged_steps

    for st in candidate_pool:
        st_start = iso_to_dt(str(st.get("started_at", "")))
        if not st_start:
            continue

        step_name = str(st.get("step_name", ""))
        uses = str(st.get("uses", ""))
        run_cmd = str(st.get("run", ""))

        flags = infer_flags_from_step(
            step_name=step_name,
            uses=uses,
            run_cmd=run_cmd,
            target_style=target_style,
        )
        if bool(st.get("stage1_anchor_match")):
            flags["stage1_anchor_match"] = True
            flags["stage1_anchor_match_reason"] = st.get("stage1_anchor_match_reason", "")
        if bool(st.get("custom_followed_file_instru")):
            flags["custom_followed_file_instru"] = True
        if bool(st.get("custom_stage1_supported_exec")):
            flags["custom_stage1_supported_exec"] = True

        rec = {
            "job_name": str(st.get("job_name", "")),
            "step_name": str(st.get("step_name", "")),
            "started_at": str(st.get("started_at", "")),
        }

        if bool(flags.get("stage1_anchor_match")):
            stage1_candidates.append(rec)
        elif normalize_style_label(target_style) == "Custom" and bool(flags.get("custom_stage1_supported_exec")):
            custom_candidates.append(rec)
        elif bool(flags.get("explicit_instru")):
            explicit_candidates.append(rec)

    all_candidates = stage1_candidates + explicit_candidates + custom_candidates
    distinct_step_names = unique_preserve([str(x["step_name"]) for x in all_candidates if norm(str(x["step_name"]))])
    distinct_jobs = unique_preserve([str(x["job_name"]) for x in all_candidates if norm(str(x["job_name"]))])

    return {
        "invocation_candidate_count_total": len(all_candidates),
        "stage1_anchor_candidate_count": len(stage1_candidates),
        "explicit_instru_candidate_count": len(explicit_candidates),
        "custom_supported_candidate_count": len(custom_candidates),
        "distinct_invocation_candidate_step_name_count": len(distinct_step_names),
        "distinct_invocation_candidate_job_count": len(distinct_jobs),
        "invocation_candidate_step_names": safe_join_names(distinct_step_names),
        "invocation_candidate_job_names": safe_join_names(distinct_jobs),
    }


def summarize_execution_window_candidates(
    merged_steps: List[Dict[str, object]],
    target_style: str,
    invocation_start_dt: Optional[datetime],
    invocation_job_name: str,
    style_job_names: Set[str],
) -> Dict[str, object]:
    if invocation_start_dt is None:
        return {
            "execution_window_candidate_count": 0,
            "execution_window_distinct_job_count": 0,
            "execution_window_candidate_job_names": "",
            "cross_job_execution_window_flag": "false",
        }

    invocation_platform = ""
    for st in merged_steps:
        if (
            norm(str(st.get("job_name", ""))) == norm(invocation_job_name)
            or job_base_name(str(st.get("job_name", ""))) == job_base_name(invocation_job_name)
        ):
            invocation_platform = get_step_platform(st)
            if invocation_platform:
                break

    all_candidates = collect_execution_window_candidates(
        merged_steps=merged_steps,
        target_style=target_style,
        invocation_start_dt=invocation_start_dt,
        invocation_job_name=invocation_job_name,
        style_job_names=style_job_names,
        invocation_platform=invocation_platform,
    )
    preferred_candidates = select_preferred_execution_window_candidates(all_candidates)

    cand_jobs = unique_preserve([
        str(c.get("job_name", ""))
        for c in preferred_candidates
        if norm(str(c.get("job_name", "")))
    ])
    cross_job = "true" if len(cand_jobs) >= 2 else "false"

    return {
        "execution_window_candidate_count": len(preferred_candidates),
        "execution_window_distinct_job_count": len(cand_jobs),
        "execution_window_candidate_job_names": safe_join_names(cand_jobs),
        "cross_job_execution_window_flag": cross_job,
    }


# =========================
# Aggregation helpers
# =========================
def init_duration_buckets() -> Dict[str, int]:
    return {
        "Setup": 0,
        "Provision": 0,
        "Test": 0,
        "Artifact/Report": 0,
        "Cleanup/Teardown": 0,
        "Other": 0,
        "Execution-related": 0,
        "Non-execution overhead": 0,
        "Pre-test overhead": 0,
        "Active test": 0,
        "Post-test overhead": 0,
    }


def init_count_buckets() -> Dict[str, int]:
    return {
        "Setup": 0,
        "Provision": 0,
        "Test": 0,
        "Artifact/Report": 0,
        "Cleanup/Teardown": 0,
        "Other": 0,
        "Execution-related": 0,
        "Non-execution overhead": 0,
        "Pre-test overhead": 0,
        "Active test": 0,
        "Post-test overhead": 0,
    }


# =========================
# Metrics keys
# =========================
STYLE_METRIC_KEYS = [
    "layer2_measurement_mode",
    "layer2_measurement_quality",

    "run_boundary_start_at",
    "run_boundary_end_at",

    "matched_invocation_step_name",
    "matched_invocation_job_name",
    "matched_invocation_source",
    "matched_invocation_step_started_at",
    "matched_invocation_step_completed_at",
    "matched_invocation_job_ordinal_in_run",
    "matched_invocation_step_ordinal_in_job",

    "invocation_execution_end_step_name",
    "invocation_execution_end_job_name",
    "invocation_execution_end_source",
    "invocation_execution_end_step_started_at",
    "invocation_execution_end_step_completed_at",
    "invocation_execution_end_job_ordinal_in_run",
    "invocation_execution_end_step_ordinal_in_job",

    "invocation_execution_window_started_at",
    "invocation_execution_window_ended_at",
    "pre_invocation_seconds",
    "invocation_execution_window_seconds",
    "post_invocation_seconds",

    "setup_sum_seconds",
    "provision_sum_seconds",
    "test_sum_seconds",
    "artifact_report_sum_seconds",
    "cleanup_teardown_sum_seconds",
    "other_sum_seconds",

    "execution_related_sum_seconds",
    "non_execution_overhead_sum_seconds",

    "pre_test_overhead_sum_seconds",
    "active_test_sum_seconds",
    "post_test_overhead_sum_seconds",

    "setup_step_count",
    "provision_step_count",
    "test_step_count",
    "artifact_report_step_count",
    "cleanup_teardown_step_count",
    "other_step_count",

    "execution_related_step_count",
    "non_execution_overhead_step_count",

    "pre_test_overhead_step_count",
    "active_test_step_count",
    "post_test_overhead_step_count",

    # V18 new Stage 3 auxiliary fields
    "invocation_candidate_count_total",
    "stage1_anchor_candidate_count",
    "explicit_instru_candidate_count",
    "custom_supported_candidate_count",
    "distinct_invocation_candidate_step_name_count",
    "distinct_invocation_candidate_job_count",
    "invocation_candidate_step_names",
    "invocation_candidate_job_names",
    "selected_invocation_priority_source",
    "execution_window_candidate_count",
    "execution_window_distinct_job_count",
    "execution_window_candidate_job_names",
    "cross_job_execution_window_flag",

    # V18 carried Stage 2 style auxiliary fields
    "style_distinct_job_count",
    "style_distinct_job_base_name_count",
    "style_matrix_like_job_count",
    "style_matrix_expanded_flag",
    "style_parallel_same_style_flag",
    "style_max_parallel_jobs",
    "style_repeated_same_style_flag",
    "style_invocation_candidate_step_count_proxy",
    "style_distinct_invocation_step_name_count_proxy",
    "style_invocation_candidate_step_names_proxy",
    "style_same_style_complexity_class",
]


# =========================
# Stage 3 core builder
# =========================
def build_stage3_outputs_for_style(
    gh: GitHubClient,
    run_row: Dict[str, str],
    style_row: Dict[str, str],
) -> Tuple[Dict[str, object], List[Dict[str, object]], Dict[str, object]]:
    full_name = norm(style_row.get("full_name") or run_row.get("full_name"))
    workflow_path = norm(style_row.get("workflow_path") or run_row.get("workflow_path"))
    workflow_ref = first_nonempty_value(style_row, ["head_sha"]) or first_nonempty_value(run_row, ["head_sha"])
    run_id = norm(style_row.get("run_id"))
    target_style = normalize_style_label(style_row.get("target_style"))

    run_attempt = resolve_run_attempt(style_row, run_row)
    run_status = resolve_run_status(style_row, run_row)
    run_conclusion = resolve_run_conclusion(style_row, run_row)
    event = resolve_event(style_row, run_row)
    trigger = resolve_trigger(style_row, run_row)

    run_started_at = iso_to_dt(
        first_nonempty_value(style_row, ["layer1_run_started_at_effective", "run_started_at"])
        or first_nonempty_value(run_row, ["L1_run_started_at_effective", "run_started_at"])
    )
    run_ended_at = iso_to_dt(
        first_nonempty_value(style_row, ["layer1_run_ended_at_effective", "run_updated_at"])
        or first_nonempty_value(run_row, ["L1_run_ended_at_effective", "run_updated_at"])
    )

    owner, repo = parse_repo(full_name)
    jobs = list_jobs_for_run(gh, owner, repo, run_id)
    step_rows = step_rows_from_jobs(jobs)

    workflow_yaml = ""
    if FETCH_WORKFLOW_YAML and workflow_path and workflow_ref:
        workflow_yaml = gh_contents_raw(gh, owner, repo, workflow_path, workflow_ref) or ""

    yaml_steps = extract_steps_from_workflow_yaml(workflow_yaml) if workflow_yaml else []

    stage1_anchor_names = parse_anchor_step_names(first_nonempty_value(run_row, ["test_invocation_step_names"]))
    called_paths = parse_stage1_confirmed_called_file_paths(run_row)
    called_origin_step_names = parse_stage1_confirmed_called_origins(run_row)

    yaml_by_stepname: Dict[str, List[Dict[str, str]]] = {}
    for ys in yaml_steps:
        yaml_by_stepname.setdefault(low(ys.get("step_name")), []).append(ys)

    merged_steps: List[Dict[str, object]] = []
    for sr in step_rows:
        rt_job_raw = str(sr["job_name"])
        rt_step = low(str(sr["step_name"]))
        matched_yaml = None

        for ys in yaml_steps:
            if runtime_job_matches_yaml_job(rt_job_raw, ys.get("job_name", "")) and low(ys.get("step_name")) == rt_step:
                matched_yaml = ys
                break

        if matched_yaml is None:
            cands = yaml_by_stepname.get(rt_step, [])
            matched_yaml = cands[0] if cands else {"job_name": sr["job_name"], "step_name": sr["step_name"], "uses": "", "run": ""}

        merged = dict(sr)
        merged["uses"] = matched_yaml.get("uses", "")
        merged["run"] = matched_yaml.get("run", "")
        merged["stage1_anchor_match"] = False
        merged["stage1_anchor_match_reason"] = ""
        merged["custom_followed_file_instru"] = False
        merged["custom_stage1_supported_exec"] = False
        merged_steps.append(merged)

    if called_paths:
        enrich_with_called_file_support(
            gh=gh,
            owner=owner,
            repo=repo,
            ref=workflow_ref,
            steps=merged_steps,
            confirmed_called_paths=called_paths,
            called_origin_step_names=called_origin_step_names,
            target_style=target_style,
        )

    mark_stage1_anchor_matches(
        steps=merged_steps,
        stage1_anchor_names=stage1_anchor_names,
        called_origin_step_names=called_origin_step_names,
    )

    style_job_names = parse_style_job_names(style_row)

    invocation_start_dt, invocation_step_end_dt, invocation_cutpoint, _inv_flags = pick_measured_invocation_step(
        merged_steps=merged_steps,
        target_style=target_style,
        style_job_names=style_job_names,
    )

    execution_end_dt, execution_end_cutpoint = pick_invocation_execution_end(
        merged_steps=merged_steps,
        target_style=target_style,
        invocation_start_dt=invocation_start_dt,
        invocation_end_dt=invocation_step_end_dt,
        invocation_job_name=str(invocation_cutpoint.get("job_name", "")),
        style_job_names=style_job_names,
    )

    pre_invocation_seconds = dt_to_seconds(run_started_at, invocation_start_dt)
    invocation_execution_window_seconds = dt_to_seconds(invocation_start_dt, execution_end_dt)
    post_invocation_seconds = dt_to_seconds(execution_end_dt, run_ended_at)

    if invocation_start_dt and execution_end_dt:
        layer2_mode = "measured_step_based"
        layer2_quality = "direct_step_plus_last_execution_related_step"
    elif invocation_start_dt:
        layer2_mode = "partial_step_based"
        layer2_quality = "direct_step_missing_execution_end"
    else:
        layer2_mode = "missing"
        layer2_quality = "missing"

    candidate_summary = summarize_invocation_candidates(
        merged_steps=merged_steps,
        target_style=target_style,
        style_job_names=style_job_names,
    )
    execution_window_summary = summarize_execution_window_candidates(
        merged_steps=merged_steps,
        target_style=target_style,
        invocation_start_dt=invocation_start_dt,
        invocation_job_name=str(invocation_cutpoint.get("job_name", "")),
        style_job_names=style_job_names,
    )

    selected_invocation_priority_source = str(invocation_cutpoint.get("source", "missing"))
    selected_invocation_job_name = str(invocation_cutpoint.get("job_name", ""))

    duration_buckets = init_duration_buckets()
    count_buckets = init_count_buckets()

    step_breakdown_rows: List[Dict[str, object]] = []

    selected_inv_step = (
        normalize_name(str(invocation_cutpoint.get("step_name", ""))),
        normalize_name(str(invocation_cutpoint.get("job_name", ""))),
        norm(str(invocation_cutpoint.get("step_started_at", ""))),
    )
    selected_end_step = (
        normalize_name(str(execution_end_cutpoint.get("step_name", ""))),
        normalize_name(str(execution_end_cutpoint.get("job_name", ""))),
        norm(str(execution_end_cutpoint.get("step_completed_at", ""))),
    )

    for st in merged_steps:
        flags = infer_flags_from_step(
            step_name=str(st.get("step_name", "")),
            uses=str(st.get("uses", "")),
            run_cmd=str(st.get("run", "")),
            target_style=target_style,
        )

        if bool(st.get("stage1_anchor_match")):
            flags["stage1_anchor_match"] = True
            flags["stage1_anchor_match_reason"] = st.get("stage1_anchor_match_reason", "")
        if bool(st.get("custom_followed_file_instru")):
            flags["custom_followed_file_instru"] = True
        if bool(st.get("custom_stage1_supported_exec")):
            flags["custom_stage1_supported_exec"] = True

        st_start = iso_to_dt(str(st.get("started_at", "")))
        st_dur = st.get("duration_seconds")
        st_dur_i = st_dur if isinstance(st_dur, int) else safe_int_from_str(str(st_dur))

        step_activity_group = classify_step_activity_group(flags)
        execution_role = classify_execution_role(
            flags,
            target_style,
            step_job_name=str(st.get("job_name", "")),
            style_job_names=style_job_names,
            invocation_job_name=selected_invocation_job_name,
        )
        overhead_phase = classify_overhead_phase(
            step_start=st_start,
            invocation_start=invocation_start_dt,
            execution_window_end=execution_end_dt,
        )

        phase_meta = infer_phase_metadata(
            step_row=st,
            flags=flags,
            step_activity_group=step_activity_group,
            invocation_job_name=selected_invocation_job_name,
            invocation_start_dt=invocation_start_dt,
        )

        inside_execution_window = (
            "true"
            if (st_start is not None and invocation_start_dt is not None and execution_end_dt is not None and invocation_start_dt <= st_start <= execution_end_dt)
            else "false"
        )

        if st_dur_i is not None:
            duration_buckets[step_activity_group] += st_dur_i
            duration_buckets[execution_role] += st_dur_i
            duration_buckets[overhead_phase] += st_dur_i

        count_buckets[step_activity_group] += 1
        count_buckets[execution_role] += 1
        count_buckets[overhead_phase] += 1

        is_selected_invocation_cutpoint = (
            normalize_name(str(st.get("step_name", ""))) == selected_inv_step[0]
            and normalize_name(str(st.get("job_name", ""))) == selected_inv_step[1]
            and norm(str(st.get("started_at", ""))) == selected_inv_step[2]
        )

        is_selected_execution_end_cutpoint = (
            normalize_name(str(st.get("step_name", ""))) == selected_end_step[0]
            and normalize_name(str(st.get("job_name", ""))) == selected_end_step[1]
            and norm(str(st.get("completed_at", ""))) == selected_end_step[2]
        )

        step_breakdown_rows.append({
            "full_name": full_name,
            "workflow_path": workflow_path,
            "workflow_ref": workflow_ref,
            "run_id": run_id,
            "run_attempt": run_attempt,
            "status": run_status,
            "run_conclusion": run_conclusion,
            "event": event,
            "trigger": trigger,
            "target_style": target_style,

            "job_id": st.get("job_id", ""),
            "job_url": st.get("job_url", ""),
            "job_html_url": st.get("job_html_url", ""),
            "job_ordinal_in_run": st.get("job_ordinal_in_run", ""),
            "step_ordinal_in_job": st.get("step_ordinal_in_job", ""),
            "runner_os": st.get("runner_os", ""),
            "runner_labels": st.get("runner_labels", ""),
            "runner_name": st.get("runner_name", ""),
            "job_name": st.get("job_name", ""),
            "step_name": st.get("step_name", ""),
            "status_step": st.get("status", ""),
            "conclusion_step": st.get("conclusion", ""),
            "started_at": st.get("started_at", ""),
            "completed_at": st.get("completed_at", ""),
            "duration_seconds": st_dur_i if st_dur_i is not None else "",
            "uses": st.get("uses", ""),
            "run": st.get("run", ""),

            "step_activity_group": step_activity_group,
            "execution_role": execution_role,
            "overhead_phase": overhead_phase,
            "phase_guess": phase_meta["phase_guess"],
            "phase_reason": phase_meta["phase_reason"],
            "phase_score": phase_meta["phase_score"],
            "phase_group_consistency": phase_meta["phase_group_consistency"],
            "path_linked_to_selected_invocation": phase_meta["path_linked_to_selected_invocation"],
            "inside_invocation_execution_window": inside_execution_window,

            "selected_invocation_cutpoint": "true" if is_selected_invocation_cutpoint else "false",
            "selected_execution_end_cutpoint": "true" if is_selected_execution_end_cutpoint else "false",

            "stage1_anchor_match": str(bool(flags.get("stage1_anchor_match"))).lower(),
            "stage1_anchor_match_reason": flags.get("stage1_anchor_match_reason", ""),
            "explicit_instru": str(bool(flags.get("explicit_instru"))).lower(),
            "explicit_instru_reason": flags.get("explicit_instru_reason", ""),
            "custom_followed_file_instru": str(bool(flags.get("custom_followed_file_instru"))).lower(),
            "custom_stage1_supported_exec": str(bool(flags.get("custom_stage1_supported_exec"))).lower(),

            "setup_flag": str(bool(flags.get("setup"))).lower(),
            "provision_flag": str(bool(flags.get("provision"))).lower(),
            "artifact_report_flag": str(bool(flags.get("artifact_report"))).lower(),
            "cleanup_teardown_flag": str(bool(flags.get("cleanup_teardown"))).lower(),
            "third_party_provider_flag": str(bool(flags.get("third_party_provider"))).lower(),
            "third_party_provider_name": flags.get("third_party_provider_name", ""),
        })

    per_style_row: Dict[str, object] = {
        "full_name": full_name,
        "workflow_path": workflow_path,
        "workflow_ref": workflow_ref,
        "run_id": run_id,
        "run_attempt": run_attempt,
        "status": run_status,
        "run_conclusion": run_conclusion,
        "event": event,
        "trigger": trigger,
        "target_style": target_style,
        "inferred_styles_all": safe_join_names([s for s in split_styles(first_nonempty_value(run_row, ["styles"])) if s in STYLE_CANONICAL]),

        "layer2_measurement_mode": layer2_mode,
        "layer2_measurement_quality": layer2_quality,

        "run_boundary_start_at": dt_to_iso_z(run_started_at),
        "run_boundary_end_at": dt_to_iso_z(run_ended_at),

        "matched_invocation_step_name": invocation_cutpoint["step_name"],
        "matched_invocation_job_name": invocation_cutpoint["job_name"],
        "matched_invocation_source": invocation_cutpoint["source"],
        "matched_invocation_step_started_at": invocation_cutpoint["step_started_at"],
        "matched_invocation_step_completed_at": invocation_cutpoint["step_completed_at"],
        "matched_invocation_job_ordinal_in_run": invocation_cutpoint["job_ordinal_in_run"],
        "matched_invocation_step_ordinal_in_job": invocation_cutpoint["step_ordinal_in_job"],

        "invocation_execution_end_step_name": execution_end_cutpoint["step_name"],
        "invocation_execution_end_job_name": execution_end_cutpoint["job_name"],
        "invocation_execution_end_source": execution_end_cutpoint["source"],
        "invocation_execution_end_step_started_at": execution_end_cutpoint["step_started_at"],
        "invocation_execution_end_step_completed_at": execution_end_cutpoint["step_completed_at"],
        "invocation_execution_end_job_ordinal_in_run": execution_end_cutpoint["job_ordinal_in_run"],
        "invocation_execution_end_step_ordinal_in_job": execution_end_cutpoint["step_ordinal_in_job"],

        "invocation_execution_window_started_at": dt_to_iso_z(invocation_start_dt),
        "invocation_execution_window_ended_at": dt_to_iso_z(execution_end_dt),
        "pre_invocation_seconds": "" if pre_invocation_seconds is None else str(pre_invocation_seconds),
        "invocation_execution_window_seconds": "" if invocation_execution_window_seconds is None else str(invocation_execution_window_seconds),
        "post_invocation_seconds": "" if post_invocation_seconds is None else str(post_invocation_seconds),

        "setup_sum_seconds": str(duration_buckets["Setup"]),
        "provision_sum_seconds": str(duration_buckets["Provision"]),
        "test_sum_seconds": str(duration_buckets["Test"]),
        "artifact_report_sum_seconds": str(duration_buckets["Artifact/Report"]),
        "cleanup_teardown_sum_seconds": str(duration_buckets["Cleanup/Teardown"]),
        "other_sum_seconds": str(duration_buckets["Other"]),

        "execution_related_sum_seconds": str(duration_buckets["Execution-related"]),
        "non_execution_overhead_sum_seconds": str(duration_buckets["Non-execution overhead"]),

        "pre_test_overhead_sum_seconds": str(duration_buckets["Pre-test overhead"]),
        "active_test_sum_seconds": str(duration_buckets["Active test"]),
        "post_test_overhead_sum_seconds": str(duration_buckets["Post-test overhead"]),

        "setup_step_count": str(count_buckets["Setup"]),
        "provision_step_count": str(count_buckets["Provision"]),
        "test_step_count": str(count_buckets["Test"]),
        "artifact_report_step_count": str(count_buckets["Artifact/Report"]),
        "cleanup_teardown_step_count": str(count_buckets["Cleanup/Teardown"]),
        "other_step_count": str(count_buckets["Other"]),

        "execution_related_step_count": str(count_buckets["Execution-related"]),
        "non_execution_overhead_step_count": str(count_buckets["Non-execution overhead"]),

        "pre_test_overhead_step_count": str(count_buckets["Pre-test overhead"]),
        "active_test_step_count": str(count_buckets["Active test"]),
        "post_test_overhead_step_count": str(count_buckets["Post-test overhead"]),

        # V18 new Stage 3 auxiliary fields
        "invocation_candidate_count_total": str(candidate_summary["invocation_candidate_count_total"]),
        "stage1_anchor_candidate_count": str(candidate_summary["stage1_anchor_candidate_count"]),
        "explicit_instru_candidate_count": str(candidate_summary["explicit_instru_candidate_count"]),
        "custom_supported_candidate_count": str(candidate_summary["custom_supported_candidate_count"]),
        "distinct_invocation_candidate_step_name_count": str(candidate_summary["distinct_invocation_candidate_step_name_count"]),
        "distinct_invocation_candidate_job_count": str(candidate_summary["distinct_invocation_candidate_job_count"]),
        "invocation_candidate_step_names": str(candidate_summary["invocation_candidate_step_names"]),
        "invocation_candidate_job_names": str(candidate_summary["invocation_candidate_job_names"]),
        "selected_invocation_priority_source": selected_invocation_priority_source,
        "execution_window_candidate_count": str(execution_window_summary["execution_window_candidate_count"]),
        "execution_window_distinct_job_count": str(execution_window_summary["execution_window_distinct_job_count"]),
        "execution_window_candidate_job_names": str(execution_window_summary["execution_window_candidate_job_names"]),
        "cross_job_execution_window_flag": str(execution_window_summary["cross_job_execution_window_flag"]),

        # carried Stage 2 V18 style auxiliary fields
        "style_distinct_job_count": first_nonempty_value(style_row, ["style_distinct_job_count"]),
        "style_distinct_job_base_name_count": first_nonempty_value(style_row, ["style_distinct_job_base_name_count"]),
        "style_matrix_like_job_count": first_nonempty_value(style_row, ["style_matrix_like_job_count"]),
        "style_matrix_expanded_flag": first_nonempty_value(style_row, ["style_matrix_expanded_flag"]),
        "style_parallel_same_style_flag": first_nonempty_value(style_row, ["style_parallel_same_style_flag"]),
        "style_max_parallel_jobs": first_nonempty_value(style_row, ["style_max_parallel_jobs"]),
        "style_repeated_same_style_flag": first_nonempty_value(style_row, ["style_repeated_same_style_flag"]),
        "style_invocation_candidate_step_count_proxy": first_nonempty_value(style_row, ["style_invocation_candidate_step_count_proxy"]),
        "style_distinct_invocation_step_name_count_proxy": first_nonempty_value(style_row, ["style_distinct_invocation_step_name_count_proxy"]),
        "style_invocation_candidate_step_names_proxy": first_nonempty_value(style_row, ["style_invocation_candidate_step_names_proxy"]),
        "style_same_style_complexity_class": first_nonempty_value(style_row, ["style_same_style_complexity_class"]),
    }

    run_support_fields = stage4_compatible_run_support_fields(row=run_row, jobs=jobs)

    # Carry these as raw fields only, no semantic fallback.
    run_support_fields["run_attempt"] = run_attempt
    run_support_fields["status"] = run_status
    run_support_fields["run_conclusion"] = run_conclusion
    run_support_fields["event"] = event
    run_support_fields["trigger"] = trigger

    # Carry Stage 2 V18 run-level auxiliary fields into run-level Stage 3 output
    for k in [
        "instru_distinct_job_count",
        "instru_distinct_job_base_name_count",
        "instru_matrix_like_job_count",
        "instru_matrix_expanded_flag",
        "instru_parallel_jobs_flag",
        "instru_max_parallel_jobs",
    ]:
        run_support_fields[k] = first_nonempty_value(run_row, [k])

    return per_style_row, step_breakdown_rows, run_support_fields


# =========================
# CSV fields
# =========================
BASE_RUN_KEEP_FIELDS = [
    "full_name",
    "default_branch",
    "workflow_path",
    "workflow_id",
    "workflow_identifier",
    "run_id",
    "run_number",
    "run_attempt",
    "head_sha",
    "created_at",
    "run_started_at",
    "run_updated_at",
    "status",
    "run_conclusion",
    "event",
    "trigger",
    "head_branch",
    "html_url",
    "looks_like_instru",
    "styles",
    "invocation_types",
    "third_party_provider_name",
    "test_invocation_step_names",
    "anchor_job_ordinal",
    "anchor_step_ordinal_in_job",
    "called_instru_signal",
    "called_instru_file_paths",
    "called_instru_origin_refs",
    "called_instru_origin_step_names",
    "called_instru_file_types",
    "instru_job_count",
    "runner_os",
    "runs_on",
    "os",
    "runner_labels",
    "job_count_total",
    "jobs_total",
    "total_jobs",
    "jobs_count",

    # Stage 2 V18 run-level auxiliary fields
    "instru_distinct_job_count",
    "instru_distinct_job_base_name_count",
    "instru_matrix_like_job_count",
    "instru_matrix_expanded_flag",
    "instru_parallel_jobs_flag",
    "instru_max_parallel_jobs",
]

out_fieldnames_3a = BASE_RUN_KEEP_FIELDS + [
    "styles_count_stage3",
    "styles_stage3",
]
for sty, pref in [
    ("Community", "community"),
    ("Custom", "custom"),
    ("GMD", "gmd"),
    ("Third-Party", "third_party"),
]:
    for k in STYLE_METRIC_KEYS:
        out_fieldnames_3a.append(f"{pref}_{k}")

out_fieldnames_3a += [
    "stage3_extracted_at_utc",
    "stage3_error",
]

out_fieldnames_3b = [
    "full_name", "workflow_path", "workflow_ref", "run_id",
    "run_attempt", "status", "run_conclusion", "event", "trigger",
    "target_style",
    "job_id", "job_url", "job_html_url", "job_ordinal_in_run", "step_ordinal_in_job",
    "runner_os", "runner_labels", "runner_name",
    "job_name", "step_name", "status_step", "conclusion_step",
    "started_at", "completed_at", "duration_seconds",
    "uses", "run",
    "step_activity_group", "execution_role", "overhead_phase",
    "phase_guess", "phase_reason", "phase_score", "phase_group_consistency",
    "path_linked_to_selected_invocation",
    "inside_invocation_execution_window",
    "selected_invocation_cutpoint", "selected_execution_end_cutpoint",
    "stage1_anchor_match", "stage1_anchor_match_reason",
    "explicit_instru", "explicit_instru_reason",
    "custom_followed_file_instru", "custom_stage1_supported_exec",
    "setup_flag", "provision_flag", "artifact_report_flag", "cleanup_teardown_flag",
    "third_party_provider_flag", "third_party_provider_name",
    "stage3_extracted_at_utc",
]

per_style_fields = [
    "full_name",
    "workflow_path",
    "workflow_ref",
    "run_id",
    "run_attempt",
    "status",
    "run_conclusion",
    "event",
    "trigger",
    "target_style",
    "inferred_styles_all",
] + STYLE_METRIC_KEYS + ["stage3_extracted_at_utc"]


# =========================
# Main
# =========================
def main():
    tokens = read_env_tokens(TOKENS_ENV_PATH)
    gh = GitHubClient(tokens=tokens)

    run_rows = read_csv_rows(IN_STAGE2_RUN_CSV)
    style_rows = read_csv_rows(IN_STAGE2_PER_STYLE_CSV)

    total_rows_before_filter = len(style_rows)

    if PROCESS_ONLY_RELEVANT_ROWS:
        style_rows = [
            r for r in style_rows
            if norm(r.get("run_id"))
            and normalize_style_label(r.get("target_style")) in STYLE_CANONICAL
        ]

    total_rows_after_runid_filter = len(style_rows)

    style_rows = [r for r in style_rows if row_is_instru_executed(r)]

    total_rows_after_exec_filter = len(style_rows)

    if total_rows_after_exec_filter == 0:
        raise RuntimeError(
            "Stage 3 executed-run filter removed all rows. "
            "Check that run_inventory_per_style.csv contains style_instru_job_count and that it is populated correctly."
        )

    run_index: Dict[str, Dict[str, str]] = {}
    for r in run_rows:
        rid = norm(r.get("run_id"))
        if rid:
            run_index[rid] = r

    all_step_rows: List[Dict[str, object]] = []
    all_per_style_rows: List[Dict[str, object]] = []
    run_level_rows: List[Dict[str, object]] = []
    extracted_ts = now_utc_iso()

    style_to_pref = {
        "Community": "community",
        "Custom": "custom",
        "GMD": "gmd",
        "Third-Party": "third_party",
    }

    iterator = tqdm(style_rows, desc="Stage3 V18") if tqdm else style_rows

    current_run_id = None
    current_run_out: Optional[Dict[str, object]] = None
    current_seen_styles: List[str] = []

    def flush_current_run():
        nonlocal current_run_out, current_seen_styles
        if current_run_out is None:
            return
        current_run_out["styles_count_stage3"] = len(unique_preserve(current_seen_styles))
        current_run_out["styles_stage3"] = safe_join_names(current_seen_styles)
        current_run_out["stage3_extracted_at_utc"] = extracted_ts
        current_run_out["stage3_error"] = current_run_out.get("stage3_error", "")
        run_level_rows.append(current_run_out)
        current_run_out = None
        current_seen_styles = []

    for style_row in iterator:
        run_id = norm(style_row.get("run_id"))
        run_row = run_index.get(run_id, {})

        try:
            per_style_row, step_rows, run_support_fields = build_stage3_outputs_for_style(
                gh=gh,
                run_row=run_row,
                style_row=style_row,
            )
            per_style_row["stage3_extracted_at_utc"] = extracted_ts
            all_per_style_rows.append(per_style_row)

            for sr in step_rows:
                sr["stage3_extracted_at_utc"] = extracted_ts
                all_step_rows.append(sr)

            if current_run_id != run_id:
                flush_current_run()
                current_run_id = run_id

                base_run = dict(run_row)
                base_run.update(run_support_fields)

                for sty, pref in style_to_pref.items():
                    for k in STYLE_METRIC_KEYS:
                        base_run[f"{pref}_{k}"] = ""
                base_run["stage3_error"] = ""
                current_run_out = base_run

            current_seen_styles.append(str(per_style_row["target_style"]))

            pref = style_to_pref[str(per_style_row["target_style"])]
            for k in STYLE_METRIC_KEYS:
                current_run_out[f"{pref}_{k}"] = per_style_row.get(k, "")

        except Exception as e:
            if current_run_id != run_id:
                flush_current_run()
                current_run_id = run_id

                base_run = dict(run_row) if run_row else dict(style_row)
                base_run.update(stage4_compatible_run_support_fields(row=base_run, jobs=[]))

                # raw carry only, no semantic fallback
                base_run["run_attempt"] = resolve_run_attempt(style_row, run_row)
                base_run["status"] = resolve_run_status(style_row, run_row)
                base_run["run_conclusion"] = resolve_run_conclusion(style_row, run_row)
                base_run["event"] = resolve_event(style_row, run_row)
                base_run["trigger"] = resolve_trigger(style_row, run_row)

                for k in [
                    "instru_distinct_job_count",
                    "instru_distinct_job_base_name_count",
                    "instru_matrix_like_job_count",
                    "instru_matrix_expanded_flag",
                    "instru_parallel_jobs_flag",
                    "instru_max_parallel_jobs",
                ]:
                    base_run[k] = first_nonempty_value(run_row, [k]) or first_nonempty_value(style_row, [k])

                for sty, pref in style_to_pref.items():
                    for k in STYLE_METRIC_KEYS:
                        base_run[f"{pref}_{k}"] = ""
                base_run["stage3_error"] = str(e)
                current_run_out = base_run
                current_seen_styles = []

    flush_current_run()

    write_csv(OUT_STAGE3B_STEPS_CSV, out_fieldnames_3b, all_step_rows)
    write_csv(OUT_STAGE3A_RUNS_CSV, out_fieldnames_3a, run_level_rows)
    write_csv(OUT_STAGE3C_RUN_PER_STYLE_CSV, per_style_fields, all_per_style_rows)

    print("[info] Stage 3 input rows before any filter:", total_rows_before_filter)
    print("[info] Stage 3 rows after run_id/style filter:", total_rows_after_runid_filter)
    print("[info] Stage 3 rows after executed-run filter (style_instru_job_count > 0):", total_rows_after_exec_filter)

    print("[done] Run metrics:", OUT_STAGE3A_RUNS_CSV)
    print("[done] Step breakdown:", OUT_STAGE3B_STEPS_CSV)
    print("[done] Run x style:", OUT_STAGE3C_RUN_PER_STYLE_CSV)


if __name__ == "__main__":
    main()

Stage3 V18: 100%|██████████| 8910/8910 [3:07:53<00:00,  1.27s/it]  


[info] Stage 3 input rows before any filter: 22413
[info] Stage 3 rows after run_id/style filter: 22407
[info] Stage 3 rows after executed-run filter (style_instru_job_count > 0): 8910
[done] Run metrics: C:\Android Mobile App\ICST2026_Ext\run_metrics_v16_stage3_enhanced.csv
[done] Step breakdown: C:\Android Mobile App\ICST2026_Ext\run_steps_v16_stage3_breakdown.csv
[done] Run x style: C:\Android Mobile App\ICST2026_Ext\run_per_style_v1_stage3.csv


## Stage 4 — Build workload/test signature layer (artifact-first) and parse result/report artifacts

In [12]:
# -*- coding: utf-8 -*-
"""
Stage 4 — Current Study Plan
Style-agnostic workload signature for normalization

Current-plan alignment
1) Compatible with current Stage 3 outputs
   - run_metrics_v16_stage3_enhanced.csv
   - run_steps_v16_stage3_breakdown.csv
2) Fingerprints ONLY Stage 3 executed runs
   - Stage 3 executed condition is preserved upstream
3) Handles Stage 3 run×style step duplication by deduplicating executed steps
4) Prefers effective_ref_for_stage4 from Stage 3 run-level output
   - falls back to head_sha if needed
5) Skips Stage 3 rows with explicit stage3_error where possible
6) Emits BOTH base and full signature hashes
   - base: OS + jobs + steps
   - full: OS + jobs + steps + suite size
   - signature_hash remains the FULL hash for backward compatibility
7) Keeps stricter numeric parsing and safer source selection
8) Keeps YAML runs-on parsing fallback
9) Keeps optional bounded artifact parsing for junit_cases / test_suite_size_bucket

Interpretation
- Stage 4 remains style-agnnostic
- It fingerprints workload structure, not Layer 1 or Layer 2 timing definitions
- Therefore the signature logic remains based on:
    runner_os_bucket
    job_count_total_bucket
    step_count_total_bucket
    test_suite_size_bucket (full signature only)

Important note
- The current study changed Stage 3 Layer-2 timing outputs
- The study also dropped instrumentation-specific conclusion as a study field
- Stage 4 does NOT use those timing fields or any instrumentation-conclusion field
- Stage 4 continues to fingerprint the overall executed workload structure of the run

Inputs (ROOT_DIR):
- run_metrics_v16_stage3_enhanced.csv
- run_steps_v16_stage3_breakdown.csv

Output:
- run_workload_signature_v3.csv
"""

import base64
import csv
import hashlib
import random
import re
import time
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple, Union
import xml.etree.ElementTree as ET

import requests

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
ROOT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_RUN_METRICS_CSV = ROOT_DIR / "run_metrics_v16_stage3_enhanced.csv"
IN_RUN_STEPS_CSV = ROOT_DIR / "run_steps_v16_stage3_breakdown.csv"

OUT_STAGE4_SIGNATURE_CSV = ROOT_DIR / "run_workload_signature_v3.csv"

MAX_TOKENS_TO_USE = 7
CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 90
MAX_RETRIES_PER_REQUEST = 8
BACKOFF_BASE_S = 1.7
BACKOFF_CAP_S = 60
MAX_PAGES_PER_LIST = 2000

FETCH_WORKFLOW_YAML = True
WORKFLOW_YAML_CACHE_MAX = 5000

DOWNLOAD_AND_PARSE_ARTIFACTS = True
MAX_ARTIFACT_ZIP_BYTES = 15 * 1024 * 1024
MAX_ARTIFACTS_TO_PARSE = 3
MAX_XMLS_PER_ARTIFACT = 40

# =========================
# Helpers
# =========================
BOM = "\ufeff"
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}", re.MULTILINE)
FULL_NAME_RE = re.compile(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$")


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def safe_lower(s: str) -> str:
    return (s or "").strip().lower()


def _clean_key(k: str) -> str:
    return (k or "").replace(BOM, "").strip()


def read_csv_rows(path: Path) -> Tuple[List[Dict[str, str]], List[str]]:
    if not path.exists():
        raise FileNotFoundError(f"Missing input CSV: {path}")
    with path.open("r", encoding="utf-8-sig", errors="ignore", newline="") as f:
        rdr = csv.DictReader(f)
        raw_fields = rdr.fieldnames or []
        fields = [_clean_key(x) for x in raw_fields]
        rows: List[Dict[str, str]] = []
        for r in rdr:
            row = {}
            for k, v in r.items():
                row[_clean_key(k)] = (v or "")
            rows.append(row)
    return rows, fields


def write_csv(path: Path, fieldnames: List[str], rows: List[Dict[str, str]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow(r)


def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    """
    Accept both:
    - GITHUB_TOKEN=...
    - GITHUB_TOKEN_1=...
    - GITHUB_TOKEN_2=...
    etc.
    """
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")

    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if not v:
            continue
        if k == "GITHUB_TOKEN" or k.startswith("GITHUB_TOKEN_"):
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break

    if not tokens:
        raise ValueError(
            f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN=... or GITHUB_TOKEN_1=..."
        )
    return tokens


def first_nonempty(row: Dict[str, str], keys: List[str]) -> str:
    for k in keys:
        v = (row.get(k) or "").strip()
        if v:
            return v
    return ""


def to_int_loose(x: str, default: int = 0) -> int:
    s = (x or "").strip()
    if not s or s.lower() in {"nan", "none"}:
        return default
    try:
        return int(float(s))
    except Exception:
        return default


def parse_int_strict(s: str) -> Optional[int]:
    s = (s or "").strip()
    if not s:
        return None

    if re.search(r"\d{4}-\d{2}-\d{2}", s):
        return None
    if re.search(r"T\d{2}:\d{2}:\d{2}", s):
        return None
    if re.search(r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", s):
        return None

    s2 = s.replace(",", "").replace("_", "").strip()

    if re.fullmatch(r"\d+", s2):
        try:
            return int(s2)
        except Exception:
            return None

    if re.fullmatch(r"\d+\.0+", s2):
        try:
            return int(float(s2))
        except Exception:
            return None

    low = s2.lower()
    if any(k in low for k in ["job", "jobs", "total_jobs", "jobs_total", "job_count", "jobs_count"]):
        nums = re.findall(r"\d+", s2)
        if len(nums) == 1:
            try:
                return int(nums[0])
            except Exception:
                return None

    return None


def sanitize_gha_expr(text: str) -> str:
    if not text:
        return ""
    return GHA_EXPR_RE.sub("MATRIX", text)


def normalize_full_name(v: str) -> str:
    v = (v or "").strip().replace(BOM, "")
    if not v:
        return ""

    m = re.search(r"github\.com/([A-Za-z0-9_.-]+)/([A-Za-z0-9_.-]+)", v)
    if m:
        return f"{m.group(1)}/{m.group(2)}"

    if FULL_NAME_RE.match(v):
        return v

    v2 = v.replace(":", "/").replace("\\", "/").strip()
    if FULL_NAME_RE.match(v2):
        return v2

    parts = [p for p in re.split(r"[\s/]+", v2) if p]
    if len(parts) >= 2:
        cand = f"{parts[-2]}/{parts[-1]}"
        if FULL_NAME_RE.match(cand):
            return cand
    return ""


def _job_identity_from_step_row(s: Dict[str, str]) -> str:
    job_id = (s.get("job_id") or "").strip()
    if job_id:
        return f"id:{job_id}"

    job_url = (s.get("job_url") or s.get("job_html_url") or s.get("html_url") or s.get("url") or "").strip()
    if job_url:
        return f"url:{job_url}"

    job_ordinal = (
        s.get("job_ordinal_in_run")
        or s.get("job_ordinal")
        or s.get("job_index")
        or s.get("job_number")
        or s.get("job_position")
        or ""
    ).strip()
    if job_ordinal:
        return f"ord:{job_ordinal}"

    job_name = (s.get("job_name") or "").strip()
    if job_name:
        attempt = (s.get("job_attempt") or s.get("attempt") or "").strip()
        matrix = (s.get("matrix") or s.get("strategy_matrix") or s.get("matrix_id") or s.get("job_matrix_id") or "").strip()
        return f"name:{job_name}|attempt:{attempt}|matrix:{matrix}"

    return ""


def _step_identity_from_step_row(s: Dict[str, str]) -> str:
    """
    Deduplicate Stage 3 run×style step rows back to executed-step level.
    """
    job_ident = _job_identity_from_step_row(s)
    step_name = (s.get("step_name") or "").strip()
    started_at = (s.get("started_at") or "").strip()
    completed_at = (s.get("completed_at") or "").strip()
    duration_seconds = (s.get("duration_seconds") or "").strip()
    step_ordinal_in_job = (s.get("step_ordinal_in_job") or "").strip()

    if job_ident or step_name or started_at or completed_at or step_ordinal_in_job:
        return "||".join([
            job_ident,
            step_ordinal_in_job,
            step_name,
            started_at,
            completed_at,
            duration_seconds,
        ])

    return "||".join([
        (s.get("job_name") or "").strip(),
        step_name,
        started_at,
        completed_at,
        duration_seconds,
    ])


# -------------------------
# Bucketing
# -------------------------
def bucket_runner_os(os_raw: str) -> str:
    s = safe_lower(os_raw)
    if "ubuntu" in s or "linux" in s:
        return "ubuntu"
    if "macos" in s or "osx" in s or (s.startswith("mac") and "machine" not in s):
        return "macos"
    if "windows" in s or s.startswith("win"):
        return "windows"
    if not s:
        return "unknown"
    return "mixed_or_unknown"


def bucket_job_count(n: Optional[int]) -> str:
    if n is None:
        return "unknown"
    if n <= 1:
        return "1"
    if 2 <= n <= 3:
        return "2_3"
    if 4 <= n <= 6:
        return "4_6"
    return ">6"


def bucket_step_count(n: Optional[int]) -> str:
    if n is None:
        return "unknown"
    if n <= 20:
        return "<=20"
    if 21 <= n <= 40:
        return "21_40"
    if 41 <= n <= 80:
        return "41_80"
    return ">80"


def bucket_suite_size(n: Optional[int]) -> str:
    if n is None or n <= 0:
        return "unknown"
    if n <= 100:
        return "1_100"
    if n <= 500:
        return "101_500"
    if n <= 2000:
        return "501_2000"
    return ">2000"


# =========================
# GitHub API client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None


class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "stage4-signature-current-plan/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        now = int(time.time())
        candidates = []
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                candidates.append((0, i))
            else:
                if st.reset_epoch is not None and st.reset_epoch <= now:
                    candidates.append((0, i))
                else:
                    candidates.append((1, i))
        candidates.sort()
        return candidates[0][1]

    def _sleep_until_reset(self) -> None:
        now = int(time.time())
        resets = [st.reset_epoch for st in self.tokens if st.reset_epoch]
        if not resets:
            time.sleep(5)
            return
        soonest = min(resets)
        sleep_s = max(1, soonest - now + 2)
        print(f"[rate-limit] sleeping {sleep_s}s until reset...")
        time.sleep(sleep_s)

    def _backoff(self, attempt: int) -> None:
        sleep_s = min(BACKOFF_CAP_S, (BACKOFF_BASE_S ** attempt)) + random.random()
        time.sleep(sleep_s)

    def request(self, method: str, url: str, params: Optional[Dict] = None, stream: bool = False) -> Optional[requests.Response]:
        last_status = None
        for attempt in range(1, MAX_RETRIES_PER_REQUEST + 1):
            idx = self._pick_idx()
            st = self.tokens[idx]
            self.session.headers["Authorization"] = f"Bearer {st.token}"
            try:
                resp = self.session.request(
                    method,
                    url,
                    params=params,
                    timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                    stream=stream,
                )
            except requests.exceptions.RequestException:
                self._backoff(attempt)
                continue

            last_status = resp.status_code

            rem = resp.headers.get("X-RateLimit-Remaining")
            if rem is not None:
                try:
                    st.remaining = int(rem)
                except Exception:
                    pass
            rst = resp.headers.get("X-RateLimit-Reset")
            if rst is not None:
                try:
                    st.reset_epoch = int(rst)
                except Exception:
                    pass

            if resp.status_code == 404:
                return None

            text_l = (resp.text or "").lower()
            if resp.status_code in (403, 429) and (
                "rate limit" in text_l
                or "secondary rate limit" in text_l
                or "abuse detection" in text_l
                or "too many requests" in text_l
            ):
                if any((t.remaining is None) or (t.remaining > 0) for t in self.tokens):
                    self._backoff(attempt)
                    continue
                self._sleep_until_reset()
                continue

            if resp.status_code in (500, 502, 503, 504):
                self._backoff(attempt)
                continue

            if resp.status_code >= 400:
                return None

            return resp

        print(f"[giveup] {method} {url} after {MAX_RETRIES_PER_REQUEST} tries (last_status={last_status})")
        return None

    def request_json(self, method: str, url: str, params: Optional[Dict] = None) -> Optional[Union[Dict, List]]:
        resp = self.request(method, url, params=params, stream=False)
        if resp is None:
            return None
        try:
            return resp.json()
        except Exception:
            return None

    def paginate(self, url: str, params: Optional[Dict], item_key: str) -> Iterable[Dict]:
        page = 1
        while page <= MAX_PAGES_PER_LIST:
            p = dict(params or {})
            p.update({"per_page": 100, "page": page})
            data = self.request_json("GET", url, params=p)
            if data is None:
                return
            items = data if isinstance(data, list) else data.get(item_key, [])
            if not items:
                return
            for it in items:
                yield it
            if isinstance(items, list) and len(items) < 100:
                return
            page += 1


# =========================
# GitHub endpoints
# =========================
def fetch_workflow_yaml(gh: GitHubClient, full_name: str, workflow_path: str, ref: str) -> str:
    url = f"https://api.github.com/repos/{full_name}/contents/{workflow_path.lstrip('/')}"
    data = gh.request_json("GET", url, params={"ref": ref})
    if not data or not isinstance(data, dict):
        return ""
    if data.get("encoding") == "base64" and data.get("content"):
        try:
            return base64.b64decode(data["content"]).decode("utf-8", errors="ignore")
        except Exception:
            return ""
    dl = data.get("download_url")
    if dl:
        resp = gh.request("GET", dl, params=None, stream=False)
        if resp and resp.status_code == 200:
            return resp.text or ""
    return ""


def list_run_artifacts(gh: GitHubClient, full_name: str, run_id: int) -> List[Dict]:
    url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/artifacts"
    return list(gh.paginate(url, params={}, item_key="artifacts"))


def download_artifact_zip(gh: GitHubClient, full_name: str, artifact_id: int) -> Optional[bytes]:
    url = f"https://api.github.com/repos/{full_name}/actions/artifacts/{artifact_id}/zip"
    resp = gh.request("GET", url, params=None, stream=True)
    if resp is None or resp.status_code != 200:
        return None
    data = bytearray()
    try:
        for chunk in resp.iter_content(chunk_size=1024 * 128):
            if not chunk:
                continue
            data.extend(chunk)
            if len(data) > MAX_ARTIFACT_ZIP_BYTES:
                return None
    except Exception:
        return None
    return bytes(data)


# =========================
# Declared step counting
# =========================
def count_declared_steps_from_yaml(yaml_text: str) -> Optional[int]:
    if not yaml_text or not yaml_text.strip():
        return None
    y = sanitize_gha_expr(yaml_text)
    lines = y.splitlines()

    total_steps = 0
    i = 0
    while i < len(lines):
        line = lines[i]
        m = re.match(r"^(\s*)steps\s*:\s*$", line)
        if not m:
            i += 1
            continue
        base_indent = len(m.group(1))
        i += 1
        while i < len(lines):
            ln = lines[i]
            if not ln.strip():
                i += 1
                continue
            indent = len(ln) - len(ln.lstrip(" "))
            if indent <= base_indent:
                break
            if re.match(r"^\s*-\s+(name|uses|run)\s*:", ln):
                total_steps += 1
            i += 1
    return total_steps if total_steps > 0 else None


def parse_runs_on_from_yaml(yaml_text: str) -> str:
    if not yaml_text or not yaml_text.strip():
        return "unknown"
    y = sanitize_gha_expr(yaml_text)
    vals = re.findall(r"(?im)^\s*runs-on\s*:\s*([^\n#]+)", y)
    buckets: Set[str] = set()
    for v in vals:
        v = v.strip().strip('"').strip("'")
        if not v:
            continue
        buckets.add(bucket_runner_os(v))
    if not buckets:
        return "unknown"
    if len(buckets) == 1:
        return list(buckets)[0]
    return "mixed_or_unknown"


# =========================
# JUnit count parsing
# =========================
_JUNIT_XML_HINTS = (
    "junit", "test", "tests", "result", "results", "report", "reports",
    "androidtest", "instrumentation", "connected", "surefire", "TEST-"
)


def _try_parse_junit_xml_counts(xml_bytes: bytes) -> int:
    try:
        root = ET.fromstring(xml_bytes)
    except Exception:
        return 0
    tag = (root.tag or "").lower()
    if not (tag.endswith("testsuite") or tag.endswith("testsuites")):
        return 0

    if tag.endswith("testsuite"):
        nodes = [root]
    else:
        nodes = list(root.findall(".//testsuite"))

    total = 0
    for n in nodes:
        t = n.attrib.get("tests")
        if t and re.fullmatch(r"\d+", t.strip()):
            total += int(t.strip())
    if total > 0:
        return total

    tcs = root.findall(".//testcase")
    return len(tcs) if tcs else 0


def extract_junit_cases_from_artifacts(gh: GitHubClient, full_name: str, run_id: int) -> Tuple[Optional[int], str]:
    artifacts = list_run_artifacts(gh, full_name, run_id) or []
    if not artifacts:
        return None, "none"

    def score_name(name: str) -> int:
        n = safe_lower(name)
        score = 0
        for kw in [
            "junit", "test-results", "test_results", "test-result",
            "androidtest", "instrumentation", "connected",
            "reports", "report", "results", "surefire"
        ]:
            if kw in n:
                score += 3
        return score

    scored = []
    for a in artifacts:
        nm = a.get("name") or ""
        scored.append((score_name(nm), a))
    scored.sort(key=lambda x: x[0], reverse=True)

    parse_list = [(s, a) for (s, a) in scored if s > 0][:MAX_ARTIFACTS_TO_PARSE]
    if not parse_list:
        return None, "none"

    total_cases = 0
    parsed_any = False

    for _, a in parse_list:
        try:
            aid = int(a.get("id"))
        except Exception:
            continue
        zip_bytes = download_artifact_zip(gh, full_name, aid)
        if not zip_bytes:
            continue
        try:
            zf = zipfile.ZipFile(BytesIO(zip_bytes))
            names = zf.namelist()
        except Exception:
            continue

        xmls = []
        for n in names:
            nl = (n or "").lower()
            if not nl.endswith(".xml"):
                continue
            if any(h.lower() in nl for h in _JUNIT_XML_HINTS) or re.search(r"(?i)/TEST-[^/]+\.xml$", n):
                xmls.append(n)

        seen = set()
        xmls2 = []
        for x in xmls:
            if x in seen:
                continue
            seen.add(x)
            xmls2.append(x)

        for xn in xmls2[:MAX_XMLS_PER_ARTIFACT]:
            try:
                raw = zf.read(xn)
            except Exception:
                continue
            if not raw or len(raw) > 2_000_000:
                continue
            c = _try_parse_junit_xml_counts(raw)
            if c > 0:
                total_cases += c
                parsed_any = True

    if parsed_any and total_cases > 0:
        return total_cases, "artifacts"
    return None, "none"


# =========================
# MAIN
# =========================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    run_rows, _ = read_csv_rows(IN_RUN_METRICS_CSV)
    step_rows, _ = read_csv_rows(IN_RUN_STEPS_CSV)

    steps_by_run: Dict[Tuple[str, str], List[Dict[str, str]]] = {}
    unique_steps_by_run: Dict[Tuple[str, str], Set[str]] = {}
    jobs_by_run: Dict[Tuple[str, str], Set[str]] = {}
    runner_os_by_run: Dict[Tuple[str, str], Set[str]] = {}

    step_runner_keys = ["runner_os", "runs_on", "runner_labels", "runner_name", "os"]

    for s in step_rows:
        fn_raw = first_nonempty(s, ["full_name", "repo_full_name", "repository", "repo", "repo_name"])
        fn = normalize_full_name(fn_raw)
        rid = (s.get("run_id") or "").strip()
        if not fn or not rid:
            continue

        key = (fn, rid)
        steps_by_run.setdefault(key, []).append(s)

        step_ident = _step_identity_from_step_row(s)
        if step_ident:
            unique_steps_by_run.setdefault(key, set()).add(step_ident)

        jid = _job_identity_from_step_row(s)
        if jid:
            jobs_by_run.setdefault(key, set()).add(jid)

        ros_raw = first_nonempty(s, step_runner_keys)
        if ros_raw:
            runner_os_by_run.setdefault(key, set()).add(bucket_runner_os(ros_raw))

    yaml_cache: Dict[Tuple[str, str, str], str] = {}
    out_rows: List[Dict[str, str]] = []
    extracted_at_utc = now_utc_iso()

    for r in run_rows:
        stage3_error = (r.get("stage3_error") or "").strip()
        if stage3_error:
            continue

        instru_job_count = to_int_loose(r.get("instru_job_count", ""), 0)
        if instru_job_count <= 0:
            continue

        fn_raw = first_nonempty(
            r,
            ["full_name", "repo_full_name", "repository", "repo", "repo_name", "repo_url", "html_url"]
        )
        full_name = normalize_full_name(fn_raw)
        run_id_s = (r.get("run_id") or "").strip()

        workflow_path = (r.get("workflow_path") or "").strip()
        head_sha = (r.get("head_sha") or "").strip()
        effective_ref = first_nonempty(r, ["effective_ref_for_stage4", "head_sha"])

        if not full_name or not run_id_s:
            continue

        run_key = (full_name, run_id_s)
        sr_list = steps_by_run.get(run_key, [])
        has_steps_rows = bool(sr_list)

        yaml_text = ""
        has_yaml = False
        step_count_decl = None
        yaml_runner_bucket = "unknown"

        if FETCH_WORKFLOW_YAML and workflow_path and effective_ref:
            ck = (full_name, workflow_path, effective_ref)
            if ck in yaml_cache:
                yaml_text = yaml_cache[ck]
            else:
                yaml_text = fetch_workflow_yaml(gh, full_name, workflow_path, effective_ref)
                if len(yaml_cache) < WORKFLOW_YAML_CACHE_MAX:
                    yaml_cache[ck] = yaml_text

            has_yaml = bool((yaml_text or "").strip())
            if has_yaml:
                step_count_decl = count_declared_steps_from_yaml(yaml_text)
                yaml_runner_bucket = parse_runs_on_from_yaml(yaml_text)

        # runner OS selection
        run_os_raw = first_nonempty(r, ["runner_os", "runs_on", "os", "runner_labels"])
        if run_os_raw:
            runner_os_bucket = bucket_runner_os(run_os_raw)
            runner_os_source = "run_metrics"
        else:
            os_set = runner_os_by_run.get(run_key, set())
            if len(os_set) == 1:
                runner_os_bucket = list(os_set)[0]
                runner_os_source = "steps"
            elif len(os_set) > 1:
                runner_os_bucket = "mixed_or_unknown"
                runner_os_source = "steps"
            else:
                runner_os_bucket = yaml_runner_bucket
                runner_os_source = "yaml" if yaml_runner_bucket != "unknown" else "unknown"

        # job count source selection
        job_count_total = None
        job_count_raw = first_nonempty(r, ["job_count_total", "jobs_total", "total_jobs", "jobs_count"])
        job_count_total = parse_int_strict(job_count_raw)

        if job_count_total is None and jobs_by_run.get(run_key):
            job_count_total = len(jobs_by_run[run_key])

        job_count_total_bucket = bucket_job_count(job_count_total)

        # dedup executed steps from duplicated run×style step rows
        unique_step_count_exec = len(unique_steps_by_run.get(run_key, set())) if unique_steps_by_run.get(run_key) else None
        step_count_exec_bucket = bucket_step_count(unique_step_count_exec) if unique_step_count_exec is not None else "unknown"
        step_count_decl_bucket = bucket_step_count(step_count_decl) if step_count_decl is not None else "unknown"

        if unique_step_count_exec is not None:
            step_count_total_bucket = step_count_exec_bucket
            step_count_source = "executed_dedup"
        elif step_count_decl is not None:
            step_count_total_bucket = step_count_decl_bucket
            step_count_source = "declared"
        else:
            step_count_total_bucket = "unknown"
            step_count_source = "unknown"

        # optional bounded artifact parse
        junit_cases = None
        junit_source = "none"
        if DOWNLOAD_AND_PARSE_ARTIFACTS:
            try:
                junit_cases, junit_source = extract_junit_cases_from_artifacts(gh, full_name, int(run_id_s))
            except Exception:
                junit_cases, junit_source = (None, "none")

        test_suite_size_bucket = bucket_suite_size(junit_cases)

        signature_inputs_parts = []
        if has_steps_rows:
            signature_inputs_parts.append("steps")
        if has_yaml:
            signature_inputs_parts.append("yaml")
        if junit_source == "artifacts":
            signature_inputs_parts.append("artifacts")
        signature_inputs = "+".join(signature_inputs_parts) if signature_inputs_parts else ""

        sig_basis_base = "\n".join([
            f"runner_os_bucket={runner_os_bucket}",
            f"job_count_total_bucket={job_count_total_bucket}",
            f"step_count_total_bucket={step_count_total_bucket}",
        ])
        signature_hash_base = hashlib.sha256(sig_basis_base.encode("utf-8", errors="ignore")).hexdigest()[:16]

        sig_basis_full = "\n".join([
            f"runner_os_bucket={runner_os_bucket}",
            f"job_count_total_bucket={job_count_total_bucket}",
            f"step_count_total_bucket={step_count_total_bucket}",
            f"test_suite_size_bucket={test_suite_size_bucket}",
        ])
        signature_hash_full = hashlib.sha256(sig_basis_full.encode("utf-8", errors="ignore")).hexdigest()[:16]

        out_rows.append({
            "full_name": full_name,
            "run_id": run_id_s,
            "workflow_identifier": r.get("workflow_identifier", ""),
            "workflow_path": workflow_path,
            "head_sha": head_sha,
            "effective_ref_for_stage4": effective_ref,

            "signature_inputs": signature_inputs,

            "runner_os_bucket": runner_os_bucket,
            "runner_os_source": runner_os_source,

            "job_count_total": str(job_count_total) if job_count_total is not None else "",
            "job_count_total_bucket": job_count_total_bucket,

            "step_count_exec": str(unique_step_count_exec) if unique_step_count_exec is not None else "",
            "step_count_exec_bucket": step_count_exec_bucket,
            "step_count_decl": str(step_count_decl) if step_count_decl is not None else "",
            "step_count_decl_bucket": step_count_decl_bucket,
            "step_count_source": step_count_source,
            "step_count_total_bucket": step_count_total_bucket,

            "junit_cases": str(junit_cases) if junit_cases is not None else "",
            "junit_source": junit_source,
            "test_suite_size_bucket": test_suite_size_bucket,

            "sig_basis_base": sig_basis_base,
            "signature_hash_base": signature_hash_base,
            "sig_basis_full": sig_basis_full,
            "signature_hash_full": signature_hash_full,
            "signature_hash": signature_hash_full,

            "stage4_extracted_at_utc": extracted_at_utc,
        })

    out_fields = [
        "full_name", "run_id", "workflow_identifier", "workflow_path", "head_sha", "effective_ref_for_stage4",
        "signature_inputs",
        "runner_os_bucket", "runner_os_source",
        "job_count_total", "job_count_total_bucket",
        "step_count_exec", "step_count_exec_bucket",
        "step_count_decl", "step_count_decl_bucket",
        "step_count_source", "step_count_total_bucket",
        "junit_cases", "junit_source", "test_suite_size_bucket",
        "sig_basis_base", "signature_hash_base",
        "sig_basis_full", "signature_hash_full",
        "signature_hash",
        "stage4_extracted_at_utc",
    ]

    write_csv(OUT_STAGE4_SIGNATURE_CSV, out_fields, out_rows)
    print("[done] Stage 4 signature (current study plan, Stage3-only, base+full, dedup-adjusted):", OUT_STAGE4_SIGNATURE_CSV)


if __name__ == "__main__":
    main()

[done] Stage 4 signature (current study plan, Stage3-only, base+full, dedup-adjusted): C:\Android Mobile App\ICST2026_Ext\run_workload_signature_v3.csv


#merge the related datasets to form the Main Dataset which alingns with the Steps telemetry covers the need from RQ1 to RQ5.

In [13]:
# ============================================================
# One-cell Jupyter code to build MainDataset.csv
# V18-adjusted for latest uploaded schemas
#
# - Stage 2 input: run_inventory_per_style.csv
# - Stage 3 input: run_per_style_v1_stage3.csv
# - Stage 4 input: run_workload_signature_v3.csv
# - FIXED: style-aware merge on full_name + run_id + target_style
# - FIXED: preserves workflow_identifier and workflow_path
# - FIXED: avoids Stage 1 row explosion
# - Includes controller fields needed for controlled subset filtering
# - Controlled subset is defined only by:
#     run_attempt == 1
#     run verdict complete
# - Excludes Real-Device rows
#
# V18 additions
# - Carries Stage 2 V18 run-level auxiliary fields
# - Carries Stage 2 V18 style-level auxiliary fields
# - Carries Stage 3 V18 auxiliary invocation/window fields
#
# FIX IN THIS VERSION
# - Drops non-study residual timing diagnostics that were causing noise:
#       study_layer2_total_seconds
#       study_grouped_known_sum_seconds
#       study_other_raw_seconds
#       study_other_seconds
#       study_other_source
#       study_window_decomp_sum_seconds
#       study_window_decomp_diff_seconds
#       study_timing_consistency_flag
# - Keeps the actual study-facing Layer 1 / Layer 2 variables and cutpoint checks
# - Robust now requires complete Stage 3 Layer 2 triplet:
#       pre + execution window + post
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Paths
# ----------------------------
BASE_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext")

IN_STAGE1 = BASE_DIR / "verified_workflows_v16.csv"
IN_STAGE2 = BASE_DIR / "run_inventory_per_style.csv"
IN_STAGE3 = BASE_DIR / "run_per_style_v1_stage3.csv"
IN_STAGE4 = BASE_DIR / "run_workload_signature_v3.csv"

OUT_MAIN = BASE_DIR / "MainDataset.csv"

# ----------------------------
# Helpers
# ----------------------------
CANONICAL_STYLES = ["Community", "Custom", "Third-Party", "GMD", "Real-Device"]

def canon_style_token(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    mapping = {
        "community": "Community",
        "custom": "Custom",
        "third-party": "Third-Party",
        "third party": "Third-Party",
        "gmd": "GMD",
        "real-device": "Real-Device",
        "real device": "Real-Device",
    }
    return mapping.get(s.lower(), s)

def canon_style_list(x):
    if pd.isna(x):
        return np.nan
    parts = [canon_style_token(p) for p in str(x).split(",")]
    parts = [p for p in parts if pd.notna(p) and str(p).strip() != ""]
    return ",".join(parts) if parts else np.nan

def bool_from_series(s):
    return s.fillna(False).astype(bool)

def in_style_scope(styles_csv, target_style):
    if pd.isna(styles_csv) or pd.isna(target_style):
        return False
    parts = [p.strip() for p in str(styles_csv).split(",") if p.strip()]
    return str(target_style).strip() in parts

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def source_col(condition, label):
    return pd.Series(np.where(condition, label, None), index=condition.index, dtype="object")

def safe_series(df, col, dtype=None):
    if col in df.columns:
        return df[col]
    return pd.Series(index=df.index, dtype=dtype)

def coalesce_cols(df, cols, dtype=None):
    available = [c for c in cols if c in df.columns]
    if not available:
        return pd.Series(index=df.index, dtype=dtype)
    out = df[available[0]].copy()
    for c in available[1:]:
        out = out.combine_first(df[c])
    return out

# ----------------------------
# Read inputs
# ----------------------------
print("Reading input files...")
stage1 = pd.read_csv(IN_STAGE1, low_memory=False)
stage2 = pd.read_csv(IN_STAGE2, low_memory=False)
stage3 = pd.read_csv(IN_STAGE3, low_memory=False)
stage4 = pd.read_csv(IN_STAGE4, low_memory=False)

print("Stage 1 shape:", stage1.shape)
print("Stage 2 shape:", stage2.shape)
print("Stage 3 shape:", stage3.shape)
print("Stage 4 shape:", stage4.shape)

# ----------------------------
# Canonicalize style fields
# ----------------------------
for df_, cols in [
    (stage1, ["styles"]),
    (stage2, ["target_style", "styles_in_run_all"]),
    (stage3, ["target_style", "inferred_styles_all"]),
]:
    for c in cols:
        if c in df_.columns:
            if c == "target_style":
                df_[c] = df_[c].map(canon_style_token)
            else:
                df_[c] = df_[c].map(canon_style_list)

# ----------------------------
# Keep only relevant fields
# ----------------------------
stage2_keep = [
    "full_name",
    "run_id",
    "target_style",
    "workflow_identifier",
    "workflow_id",
    "workflow_path",
    "run_number",
    "run_attempt",
    "created_at",
    "run_started_at",
    "run_updated_at",
    "status",
    "run_conclusion",
    "event",
    "head_branch",
    "head_sha",
    "html_url",
    "styles_in_run_all",
    "multi_style_run_flag",

    "layer1_run_started_at_effective",
    "layer1_run_ended_at_effective",
    "layer1_run_duration_seconds_effective",
    "layer1_run_timing_source",

    "style_instru_job_count",
    "style_instru_job_names",
    "style_first_instru_job_name",
    "style_first_instru_job_started_at",
    "style_first_instru_job_source",
    "style_last_instru_job_name",
    "style_last_instru_job_completed_at",
    "style_last_instru_job_source",

    "style_time_to_instrumentation_envelope_seconds",
    "style_instrumentation_job_envelope_seconds",
    "style_post_instrumentation_tail_seconds",
    "style_layer1_model",

    # V18 Stage 2 style auxiliary fields
    "style_distinct_job_count",
    "style_distinct_job_base_name_count",
    "style_matrix_like_job_count",
    "style_matrix_expanded_flag",
    "style_parallel_same_style_flag",
    "style_max_parallel_jobs",
    "style_repeated_same_style_flag",
    "style_invocation_candidate_step_count_proxy",
    "style_distinct_invocation_step_name_count_proxy",
    "style_invocation_candidate_step_names_proxy",
    "style_same_style_complexity_class",
]

stage3_keep = [
    "full_name",
    "workflow_path",
    "workflow_ref",
    "run_id",
    "run_attempt",
    "status",
    "run_conclusion",
    "event",
    "trigger",
    "target_style",
    "inferred_styles_all",

    "layer2_measurement_mode",
    "layer2_measurement_quality",

    "run_boundary_start_at",
    "run_boundary_end_at",

    "matched_invocation_step_name",
    "matched_invocation_job_name",
    "matched_invocation_source",
    "matched_invocation_step_started_at",
    "matched_invocation_step_completed_at",
    "matched_invocation_job_ordinal_in_run",
    "matched_invocation_step_ordinal_in_job",

    "invocation_execution_end_step_name",
    "invocation_execution_end_job_name",
    "invocation_execution_end_source",
    "invocation_execution_end_step_started_at",
    "invocation_execution_end_step_completed_at",
    "invocation_execution_end_job_ordinal_in_run",
    "invocation_execution_end_step_ordinal_in_job",

    "invocation_execution_window_started_at",
    "invocation_execution_window_ended_at",

    "pre_invocation_seconds",
    "invocation_execution_window_seconds",
    "post_invocation_seconds",

    "setup_sum_seconds",
    "provision_sum_seconds",
    "test_sum_seconds",
    "artifact_report_sum_seconds",
    "cleanup_teardown_sum_seconds",
    "other_sum_seconds",
    "execution_related_sum_seconds",
    "non_execution_overhead_sum_seconds",
    "pre_test_overhead_sum_seconds",
    "active_test_sum_seconds",
    "post_test_overhead_sum_seconds",

    "setup_step_count",
    "provision_step_count",
    "test_step_count",
    "artifact_report_step_count",
    "cleanup_teardown_step_count",
    "other_step_count",
    "execution_related_step_count",
    "non_execution_overhead_step_count",
    "pre_test_overhead_step_count",
    "active_test_step_count",
    "post_test_overhead_step_count",

    # V18 Stage 3 auxiliary fields
    "invocation_candidate_count_total",
    "stage1_anchor_candidate_count",
    "explicit_instru_candidate_count",
    "custom_supported_candidate_count",
    "distinct_invocation_candidate_step_name_count",
    "distinct_invocation_candidate_job_count",
    "invocation_candidate_step_names",
    "invocation_candidate_job_names",
    "selected_invocation_priority_source",
    "execution_window_candidate_count",
    "execution_window_distinct_job_count",
    "execution_window_candidate_job_names",
    "cross_job_execution_window_flag",

    # carried Stage 2 style auxiliary fields in Stage 3 output
    "style_distinct_job_count",
    "style_distinct_job_base_name_count",
    "style_matrix_like_job_count",
    "style_matrix_expanded_flag",
    "style_parallel_same_style_flag",
    "style_max_parallel_jobs",
    "style_repeated_same_style_flag",
    "style_invocation_candidate_step_count_proxy",
    "style_distinct_invocation_step_name_count_proxy",
    "style_invocation_candidate_step_names_proxy",
    "style_same_style_complexity_class",
]

stage4_keep = [
    "full_name",
    "run_id",
    "workflow_identifier",
    "workflow_path",
    "head_sha",
    "effective_ref_for_stage4",
    "signature_inputs",
    "runner_os_bucket",
    "runner_os_source",
    "job_count_total",
    "job_count_total_bucket",
    "step_count_exec",
    "step_count_exec_bucket",
    "step_count_decl",
    "step_count_decl_bucket",
    "step_count_total_bucket",
    "step_count_source",
    "junit_cases",
    "junit_source",
    "test_suite_size_bucket",
    "sig_basis_base",
    "signature_hash_base",
    "sig_basis_full",
    "signature_hash_full",
    "signature_hash",
]

stage1_keep = [
    "full_name",
    "workflow_identifier",
    "workflow_id",
    "workflow_path",
    "styles",
    "invocation_types",
    "third_party_provider_name",
]

s2 = stage2[[c for c in stage2_keep if c in stage2.columns]].copy()
s3 = stage3[[c for c in stage3_keep if c in stage3.columns]].copy()
s4 = stage4[[c for c in stage4_keep if c in stage4.columns]].copy()
s1 = stage1[[c for c in stage1_keep if c in stage1.columns]].copy()

# ----------------------------
# Deduplicate Stage 1 safely to avoid row explosion
# ----------------------------
if not s1.empty:
    if "workflow_identifier" in s1.columns:
        s1["workflow_identifier"] = s1["workflow_identifier"].replace("", np.nan)
    if "workflow_id" in s1.columns:
        s1["workflow_id"] = s1["workflow_id"].replace("", np.nan)
    if "workflow_path" in s1.columns:
        s1["workflow_path"] = s1["workflow_path"].replace("", np.nan)

    s1_exact = s1.dropna(subset=["workflow_identifier", "workflow_id"]).copy()
    s1_exact = s1_exact.drop_duplicates(
        subset=["full_name", "workflow_identifier", "workflow_id"], keep="first"
    )

    s1_by_ident = s1.dropna(subset=["workflow_identifier"]).copy()
    s1_by_ident = s1_by_ident.drop_duplicates(
        subset=["full_name", "workflow_identifier"], keep="first"
    )

    s1_by_wfid = s1.dropna(subset=["workflow_id"]).copy()
    s1_by_wfid = s1_by_wfid.drop_duplicates(
        subset=["full_name", "workflow_id"], keep="first"
    )
else:
    s1_exact = s1.copy()
    s1_by_ident = s1.copy()
    s1_by_wfid = s1.copy()

# ----------------------------
# Rename overlapping columns before merge
# ----------------------------
s2 = s2.rename(columns={
    "workflow_identifier": "workflow_identifier_s2",
    "workflow_id": "workflow_id_s2",
    "workflow_path": "workflow_path_s2",
    "run_number": "run_number_s2",
    "run_attempt": "run_attempt_s2",
    "created_at": "created_at_s2",
    "run_started_at": "run_started_at_s2",
    "run_updated_at": "run_updated_at_s2",
    "status": "status_s2",
    "run_conclusion": "run_conclusion_s2",
    "event": "event_s2",
    "head_branch": "head_branch_s2",
    "head_sha": "head_sha_s2",
    "html_url": "html_url_s2",
    "styles_in_run_all": "styles_in_run_all_s2",
    "multi_style_run_flag": "multi_style_run_flag_s2",
    "layer1_run_started_at_effective": "layer1_run_started_at_effective_s2",
    "layer1_run_ended_at_effective": "layer1_run_ended_at_effective_s2",
    "layer1_run_duration_seconds_effective": "L1_run_duration_seconds_effective_s2",
    "layer1_run_timing_source": "L1_run_timing_source_s2",
    "style_instru_job_count": "style_instru_job_count_s2",
    "style_instru_job_names": "style_instru_job_names_s2",
    "style_first_instru_job_name": "style_first_instru_job_name_s2",
    "style_first_instru_job_started_at": "style_first_instru_job_started_at_s2",
    "style_first_instru_job_source": "style_first_instru_job_source_s2",
    "style_last_instru_job_name": "style_last_instru_job_name_s2",
    "style_last_instru_job_completed_at": "style_last_instru_job_completed_at_s2",
    "style_last_instru_job_source": "style_last_instru_job_source_s2",
    "style_time_to_instrumentation_envelope_seconds": "style_time_to_instrumentation_envelope_seconds_s2",
    "style_instrumentation_job_envelope_seconds": "style_instrumentation_job_envelope_seconds_s2",
    "style_post_instrumentation_tail_seconds": "style_post_instrumentation_tail_seconds_s2",
    "style_layer1_model": "style_layer1_model_s2",

    # V18 Stage 2 style auxiliaries
    "style_distinct_job_count": "style_distinct_job_count_s2",
    "style_distinct_job_base_name_count": "style_distinct_job_base_name_count_s2",
    "style_matrix_like_job_count": "style_matrix_like_job_count_s2",
    "style_matrix_expanded_flag": "style_matrix_expanded_flag_s2",
    "style_parallel_same_style_flag": "style_parallel_same_style_flag_s2",
    "style_max_parallel_jobs": "style_max_parallel_jobs_s2",
    "style_repeated_same_style_flag": "style_repeated_same_style_flag_s2",
    "style_invocation_candidate_step_count_proxy": "style_invocation_candidate_step_count_proxy_s2",
    "style_distinct_invocation_step_name_count_proxy": "style_distinct_invocation_step_name_count_proxy_s2",
    "style_invocation_candidate_step_names_proxy": "style_invocation_candidate_step_names_proxy_s2",
    "style_same_style_complexity_class": "style_same_style_complexity_class_s2",
})

s3 = s3.rename(columns={
    "workflow_path": "workflow_path_s3",
    "run_attempt": "run_attempt_s3",
    "status": "status_s3",
    "run_conclusion": "run_conclusion_s3",
    "event": "event_s3",
    "trigger": "trigger_s3",

    # carried Stage 2 style auxiliaries in Stage 3 output
    "style_distinct_job_count": "style_distinct_job_count_s3",
    "style_distinct_job_base_name_count": "style_distinct_job_base_name_count_s3",
    "style_matrix_like_job_count": "style_matrix_like_job_count_s3",
    "style_matrix_expanded_flag": "style_matrix_expanded_flag_s3",
    "style_parallel_same_style_flag": "style_parallel_same_style_flag_s3",
    "style_max_parallel_jobs": "style_max_parallel_jobs_s3",
    "style_repeated_same_style_flag": "style_repeated_same_style_flag_s3",
    "style_invocation_candidate_step_count_proxy": "style_invocation_candidate_step_count_proxy_s3",
    "style_distinct_invocation_step_name_count_proxy": "style_distinct_invocation_step_name_count_proxy_s3",
    "style_invocation_candidate_step_names_proxy": "style_invocation_candidate_step_names_proxy_s3",
    "style_same_style_complexity_class": "style_same_style_complexity_class_s3",
})

s4 = s4.rename(columns={
    "workflow_identifier": "workflow_identifier_s4",
    "workflow_path": "workflow_path_s4",
    "head_sha": "head_sha_s4",
})

s1_exact = s1_exact.rename(columns={
    "workflow_identifier": "workflow_identifier_s1_exact",
    "workflow_id": "workflow_id_s1_exact",
    "workflow_path": "workflow_path_s1_exact",
    "styles": "styles_s1_exact",
    "invocation_types": "invocation_types_s1_exact",
    "third_party_provider_name": "third_party_provider_name_s1_exact",
})

s1_by_ident = s1_by_ident.rename(columns={
    "workflow_identifier": "workflow_identifier_s1_ident",
    "workflow_id": "workflow_id_s1_ident",
    "workflow_path": "workflow_path_s1_ident",
    "styles": "styles_s1_ident",
    "invocation_types": "invocation_types_s1_ident",
    "third_party_provider_name": "third_party_provider_name_s1_ident",
})

s1_by_wfid = s1_by_wfid.rename(columns={
    "workflow_identifier": "workflow_identifier_s1_wfid",
    "workflow_id": "workflow_id_s1_wfid",
    "workflow_path": "workflow_path_s1_wfid",
    "styles": "styles_s1_wfid",
    "invocation_types": "invocation_types_s1_wfid",
    "third_party_provider_name": "third_party_provider_name_s1_wfid",
})

# ----------------------------
# Merge Stage 3 + Stage 2 (style-aware) + Stage 4
# ----------------------------
df = s3.merge(
    s2,
    on=["full_name", "run_id", "target_style"],
    how="left",
)

df = df.merge(
    s4,
    on=["full_name", "run_id"],
    how="left",
)

# ----------------------------
# Rebuild canonical identity fields BEFORE Stage 1 fallback merges
# ----------------------------
df["workflow_id"] = coalesce_cols(df, ["workflow_id_s2"], dtype="object")
df["workflow_identifier"] = coalesce_cols(df, ["workflow_identifier_s2", "workflow_identifier_s4"], dtype="object")
df["workflow_path"] = coalesce_cols(df, ["workflow_path_s3", "workflow_path_s2", "workflow_path_s4"], dtype="object")
df["head_sha"] = coalesce_cols(df, ["head_sha_s2", "head_sha_s4"], dtype="object")
df["html_url"] = coalesce_cols(df, ["html_url_s2"], dtype="object")

# ----------------------------
# Stage 1 fallback merges WITHOUT explosion
# ----------------------------
if not s1_exact.empty:
    df = df.merge(
        s1_exact,
        left_on=["full_name", "workflow_identifier", "workflow_id"],
        right_on=["full_name", "workflow_identifier_s1_exact", "workflow_id_s1_exact"],
        how="left",
    )

if not s1_by_ident.empty:
    df = df.merge(
        s1_by_ident,
        left_on=["full_name", "workflow_identifier"],
        right_on=["full_name", "workflow_identifier_s1_ident"],
        how="left",
    )

if not s1_by_wfid.empty:
    df = df.merge(
        s1_by_wfid,
        left_on=["full_name", "workflow_id"],
        right_on=["full_name", "workflow_id_s1_wfid"],
        how="left",
    )

df = df.copy()

# ----------------------------
# Metadata fallback
# ----------------------------
df["styles"] = coalesce_cols(
    df,
    ["inferred_styles_all", "styles_in_run_all_s2", "styles_s1_exact", "styles_s1_ident", "styles_s1_wfid"],
    dtype="object",
).map(canon_style_list)

df["invocation_types"] = coalesce_cols(
    df,
    ["invocation_types_s1_exact", "invocation_types_s1_ident", "invocation_types_s1_wfid"],
    dtype="object"
)

df["third_party_provider_name"] = coalesce_cols(
    df,
    [
        "third_party_provider_name_s1_exact",
        "third_party_provider_name_s1_ident",
        "third_party_provider_name_s1_wfid",
    ],
    dtype="object"
)

df["workflow_id"] = coalesce_cols(
    df,
    ["workflow_id", "workflow_id_s1_exact", "workflow_id_s1_ident", "workflow_id_s1_wfid"],
    dtype="object"
)

df["workflow_identifier"] = coalesce_cols(
    df,
    ["workflow_identifier", "workflow_identifier_s1_exact", "workflow_identifier_s1_ident", "workflow_identifier_s1_wfid"],
    dtype="object"
)

df["workflow_path"] = coalesce_cols(
    df,
    ["workflow_path", "workflow_path_s1_exact", "workflow_path_s1_ident", "workflow_path_s1_wfid"],
    dtype="object"
)

df["style"] = safe_series(df, "target_style", dtype="object")
if "target_style" in df.columns:
    df["target_style"] = df["target_style"].map(canon_style_token)

# ----------------------------
# Study-facing controller/event fields
# ----------------------------
df["run_number"] = safe_series(df, "run_number_s2")
df["run_attempt"] = coalesce_cols(df, ["run_attempt_s3", "run_attempt_s2"])
df["status"] = coalesce_cols(df, ["status_s3", "status_s2"], dtype="object")
df["run_conclusion"] = coalesce_cols(df, ["run_conclusion_s3", "run_conclusion_s2"], dtype="object")
df["event"] = coalesce_cols(df, ["event_s3", "event_s2"], dtype="object")
df["trigger"] = safe_series(df, "trigger_s3", dtype="object")

df["created_at"] = safe_series(df, "created_at_s2", dtype="object")
df["run_started_at"] = safe_series(df, "run_started_at_s2", dtype="object")
df["run_updated_at"] = safe_series(df, "run_updated_at_s2", dtype="object")
df["head_branch"] = safe_series(df, "head_branch_s2", dtype="object")
df["multi_style_run_flag"] = safe_series(df, "multi_style_run_flag_s2")

# ----------------------------
# Exclude Real-Device
# ----------------------------
exclude_mask = safe_series(df, "target_style", dtype="object").eq("Real-Device")
excluded_count = int(exclude_mask.fillna(False).sum())
df = df.loc[~exclude_mask.fillna(False)].copy()

print("Excluded Real-Device rows:", excluded_count)
print("Shape after Real-Device exclusion:", df.shape)

dup_count = int(df.duplicated(subset=["full_name", "run_id", "style"], keep=False).sum())
print("Duplicate rows on (full_name, run_id, style):", dup_count)

# ----------------------------
# Controller regime
# ----------------------------
df["controller_style_in_scope"] = [
    in_style_scope(styles_csv, tgt)
    for styles_csv, tgt in zip(
        safe_series(df, "styles", dtype="object"),
        safe_series(df, "target_style", dtype="object")
    )
]

df["instru_job_count"] = to_num(safe_series(df, "style_instru_job_count_s2"))
df["controller_instru_job_count_gt0"] = df["instru_job_count"].fillna(0).gt(0)
df["controller_attempt_eq_1"] = to_num(safe_series(df, "run_attempt")).eq(1)

terminal_conclusions = {
    "success", "failure", "cancelled", "timed_out",
    "neutral", "skipped", "startup_failure", "action_required"
}

df["controller_run_verdict_complete"] = safe_series(df, "run_conclusion", dtype="object").isin(terminal_conclusions)

df["Base"] = (
    bool_from_series(df["controller_attempt_eq_1"]) &
    bool_from_series(df["controller_run_verdict_complete"])
)

df = df.copy()

# ----------------------------
# Layer 1 (from Stage 2 per-style)
# ----------------------------
df["study_run_duration_seconds"] = safe_series(df, "L1_run_duration_seconds_effective_s2")
df["study_run_duration_source"] = safe_series(df, "L1_run_timing_source_s2", dtype="object")

df["study_layer1_time_to_instrumentation_envelope_seconds"] = safe_series(
    df, "style_time_to_instrumentation_envelope_seconds_s2"
)
df["study_layer1_instrumentation_job_envelope_seconds"] = safe_series(
    df, "style_instrumentation_job_envelope_seconds_s2"
)
df["study_layer1_post_instrumentation_tail_seconds"] = safe_series(
    df, "style_post_instrumentation_tail_seconds_s2"
)
df["study_layer1_model"] = safe_series(df, "style_layer1_model_s2", dtype="object")

# ----------------------------
# V18 Stage 2 run/style auxiliary fields
# ----------------------------
for src, dst in [
    ("instru_distinct_job_count", "study_instru_distinct_job_count"),
    ("instru_distinct_job_base_name_count", "study_instru_distinct_job_base_name_count"),
    ("instru_matrix_like_job_count", "study_instru_matrix_like_job_count"),
    ("instru_matrix_expanded_flag", "study_instru_matrix_expanded_flag"),
    ("instru_parallel_jobs_flag", "study_instru_parallel_jobs_flag"),
    ("instru_max_parallel_jobs", "study_instru_max_parallel_jobs"),
]:
    df[dst] = safe_series(df, src, dtype="object")

for src, dst in [
    ("style_distinct_job_count_s2", "study_style_distinct_job_count"),
    ("style_distinct_job_base_name_count_s2", "study_style_distinct_job_base_name_count"),
    ("style_matrix_like_job_count_s2", "study_style_matrix_like_job_count"),
    ("style_matrix_expanded_flag_s2", "study_style_matrix_expanded_flag"),
    ("style_parallel_same_style_flag_s2", "study_style_parallel_same_style_flag"),
    ("style_max_parallel_jobs_s2", "study_style_max_parallel_jobs"),
    ("style_repeated_same_style_flag_s2", "study_style_repeated_same_style_flag"),
    ("style_invocation_candidate_step_count_proxy_s2", "study_style_invocation_candidate_step_count_proxy"),
    ("style_distinct_invocation_step_name_count_proxy_s2", "study_style_distinct_invocation_step_name_count_proxy"),
    ("style_invocation_candidate_step_names_proxy_s2", "study_style_invocation_candidate_step_names_proxy"),
    ("style_same_style_complexity_class_s2", "study_style_same_style_complexity_class"),
]:
    df[dst] = safe_series(df, src, dtype="object")

# ----------------------------
# Layer 2 direct / selected
# ----------------------------
df["study_pre_invocation_direct_seconds"] = safe_series(df, "pre_invocation_seconds")
df["study_pre_invocation_direct_source"] = source_col(
    df["study_pre_invocation_direct_seconds"].notna(), "measured_step_telemetry"
)

df["study_invocation_execution_window_direct_seconds"] = safe_series(df, "invocation_execution_window_seconds")
df["study_invocation_execution_window_direct_source"] = source_col(
    df["study_invocation_execution_window_direct_seconds"].notna(), "measured_step_telemetry"
)

df["study_post_invocation_direct_seconds"] = safe_series(df, "post_invocation_seconds")
df["study_post_invocation_direct_source"] = source_col(
    df["study_post_invocation_direct_seconds"].notna(), "measured_step_telemetry"
)

df["study_pre_invocation_selected_stage3_seconds"] = df["study_pre_invocation_direct_seconds"]
df["study_pre_invocation_selected_stage3_source"] = df["study_pre_invocation_direct_source"]

df["study_invocation_execution_window_selected_stage3_seconds"] = df["study_invocation_execution_window_direct_seconds"]
df["study_invocation_execution_window_selected_stage3_source"] = df["study_invocation_execution_window_direct_source"]

df["study_post_invocation_selected_stage3_seconds"] = df["study_post_invocation_direct_seconds"]
df["study_post_invocation_selected_stage3_source"] = df["study_post_invocation_direct_source"]

df["study_layer2_measurement_mode"] = safe_series(df, "layer2_measurement_mode", dtype="object")
df["study_layer2_measurement_quality"] = safe_series(df, "layer2_measurement_quality", dtype="object")

# ----------------------------
# V18 Stage 3 auxiliary invocation/window fields
# ----------------------------
for src, dst in [
    ("invocation_candidate_count_total", "study_invocation_candidate_count_total"),
    ("stage1_anchor_candidate_count", "study_stage1_anchor_candidate_count"),
    ("explicit_instru_candidate_count", "study_explicit_instru_candidate_count"),
    ("custom_supported_candidate_count", "study_custom_supported_candidate_count"),
    ("distinct_invocation_candidate_step_name_count", "study_distinct_invocation_candidate_step_name_count"),
    ("distinct_invocation_candidate_job_count", "study_distinct_invocation_candidate_job_count"),
    ("invocation_candidate_step_names", "study_invocation_candidate_step_names"),
    ("invocation_candidate_job_names", "study_invocation_candidate_job_names"),
    ("selected_invocation_priority_source", "study_selected_invocation_priority_source"),
    ("execution_window_candidate_count", "study_execution_window_candidate_count"),
    ("execution_window_distinct_job_count", "study_execution_window_distinct_job_count"),
    ("execution_window_candidate_job_names", "study_execution_window_candidate_job_names"),
    ("cross_job_execution_window_flag", "study_cross_job_execution_window_flag"),
]:
    df[dst] = safe_series(df, src, dtype="object")

for dst, cols in [
    ("study_style_distinct_job_count", ["style_distinct_job_count_s3", "study_style_distinct_job_count"]),
    ("study_style_distinct_job_base_name_count", ["style_distinct_job_base_name_count_s3", "study_style_distinct_job_base_name_count"]),
    ("study_style_matrix_like_job_count", ["style_matrix_like_job_count_s3", "study_style_matrix_like_job_count"]),
    ("study_style_matrix_expanded_flag", ["style_matrix_expanded_flag_s3", "study_style_matrix_expanded_flag"]),
    ("study_style_parallel_same_style_flag", ["style_parallel_same_style_flag_s3", "study_style_parallel_same_style_flag"]),
    ("study_style_max_parallel_jobs", ["style_max_parallel_jobs_s3", "study_style_max_parallel_jobs"]),
    ("study_style_repeated_same_style_flag", ["style_repeated_same_style_flag_s3", "study_style_repeated_same_style_flag"]),
    ("study_style_invocation_candidate_step_count_proxy", ["style_invocation_candidate_step_count_proxy_s3", "study_style_invocation_candidate_step_count_proxy"]),
    ("study_style_distinct_invocation_step_name_count_proxy", ["style_distinct_invocation_step_name_count_proxy_s3", "study_style_distinct_invocation_step_name_count_proxy"]),
    ("study_style_invocation_candidate_step_names_proxy", ["style_invocation_candidate_step_names_proxy_s3", "study_style_invocation_candidate_step_names_proxy"]),
    ("study_style_same_style_complexity_class", ["style_same_style_complexity_class_s3", "study_style_same_style_complexity_class"]),
]:
    df[dst] = coalesce_cols(df, cols, dtype="object")

# ----------------------------
# Layer 2 cutpoints
# ----------------------------
df["study_run_boundary_start_at"] = safe_series(df, "run_boundary_start_at", dtype="object")
df["study_run_boundary_end_at"] = safe_series(df, "run_boundary_end_at", dtype="object")

df["study_matched_invocation_step_name"] = safe_series(df, "matched_invocation_step_name", dtype="object")
df["study_matched_invocation_job_name"] = safe_series(df, "matched_invocation_job_name", dtype="object")
df["study_matched_invocation_source"] = safe_series(df, "matched_invocation_source", dtype="object")
df["study_matched_invocation_step_started_at"] = safe_series(df, "matched_invocation_step_started_at", dtype="object")
df["study_matched_invocation_step_completed_at"] = safe_series(df, "matched_invocation_step_completed_at", dtype="object")
df["study_matched_invocation_job_ordinal_in_run"] = safe_series(df, "matched_invocation_job_ordinal_in_run", dtype="object")
df["study_matched_invocation_step_ordinal_in_job"] = safe_series(df, "matched_invocation_step_ordinal_in_job", dtype="object")

df["study_invocation_execution_end_step_name"] = safe_series(df, "invocation_execution_end_step_name", dtype="object")
df["study_invocation_execution_end_job_name"] = safe_series(df, "invocation_execution_end_job_name", dtype="object")
df["study_invocation_execution_end_source"] = safe_series(df, "invocation_execution_end_source", dtype="object")
df["study_invocation_execution_end_step_started_at"] = safe_series(df, "invocation_execution_end_step_started_at", dtype="object")
df["study_invocation_execution_end_step_completed_at"] = safe_series(df, "invocation_execution_end_step_completed_at", dtype="object")
df["study_invocation_execution_end_job_ordinal_in_run"] = safe_series(df, "invocation_execution_end_job_ordinal_in_run", dtype="object")
df["study_invocation_execution_end_step_ordinal_in_job"] = safe_series(df, "invocation_execution_end_step_ordinal_in_job", dtype="object")

df["study_invocation_execution_window_started_at"] = safe_series(df, "invocation_execution_window_started_at", dtype="object")
df["study_invocation_execution_window_ended_at"] = safe_series(df, "invocation_execution_window_ended_at", dtype="object")

df = df.copy()

# ----------------------------
# Grouping / decomposition outputs
# ----------------------------
for src, dst in [
    ("setup_sum_seconds", "study_setup_sum_seconds"),
    ("provision_sum_seconds", "study_provision_sum_seconds"),
    ("test_sum_seconds", "study_test_sum_seconds"),
    ("artifact_report_sum_seconds", "study_artifact_report_sum_seconds"),
    ("cleanup_teardown_sum_seconds", "study_cleanup_teardown_sum_seconds"),
    ("other_sum_seconds", "study_other_sum_seconds"),
    ("execution_related_sum_seconds", "study_execution_related_sum_seconds"),
    ("non_execution_overhead_sum_seconds", "study_non_execution_overhead_sum_seconds"),
    ("pre_test_overhead_sum_seconds", "study_pre_test_overhead_sum_seconds"),
    ("active_test_sum_seconds", "study_active_test_sum_seconds"),
    ("post_test_overhead_sum_seconds", "study_post_test_overhead_sum_seconds"),
    ("setup_step_count", "study_setup_step_count"),
    ("provision_step_count", "study_provision_step_count"),
    ("test_step_count", "study_test_step_count"),
    ("artifact_report_step_count", "study_artifact_report_step_count"),
    ("cleanup_teardown_step_count", "study_cleanup_teardown_step_count"),
    ("other_step_count", "study_other_step_count"),
    ("execution_related_step_count", "study_execution_related_step_count"),
    ("non_execution_overhead_step_count", "study_non_execution_overhead_step_count"),
    ("pre_test_overhead_step_count", "study_pre_test_overhead_step_count"),
    ("active_test_step_count", "study_active_test_step_count"),
    ("post_test_overhead_step_count", "study_post_test_overhead_step_count"),
]:
    if src in df.columns:
        df[dst] = safe_series(df, src)

# ----------------------------
# Automated validation diagnostics from cutpoints
# ----------------------------
run_start_dt = pd.to_datetime(df["study_run_boundary_start_at"], errors="coerce")
run_end_dt = pd.to_datetime(df["study_run_boundary_end_at"], errors="coerce")
inv_start_dt = pd.to_datetime(df["study_matched_invocation_step_started_at"], errors="coerce")
exec_end_dt = pd.to_datetime(df["study_invocation_execution_end_step_completed_at"], errors="coerce")

df["study_cutpoint_pre_invocation_diff_seconds"] = (
    (inv_start_dt - run_start_dt).dt.total_seconds()
    - to_num(df["study_pre_invocation_selected_stage3_seconds"])
)
df["study_cutpoint_execution_window_diff_seconds"] = (
    (exec_end_dt - inv_start_dt).dt.total_seconds()
    - to_num(df["study_invocation_execution_window_selected_stage3_seconds"])
)
df["study_cutpoint_post_invocation_diff_seconds"] = (
    (run_end_dt - exec_end_dt).dt.total_seconds()
    - to_num(df["study_post_invocation_selected_stage3_seconds"])
)

for c in [
    "study_cutpoint_pre_invocation_diff_seconds",
    "study_cutpoint_execution_window_diff_seconds",
    "study_cutpoint_post_invocation_diff_seconds",
]:
    df.loc[df[c].between(-1e-9, 1e-9, inclusive="both"), c] = 0.0

df["study_cutpoint_consistency_flag"] = "missing"
have_cutpoints = (
    run_start_dt.notna() &
    inv_start_dt.notna() &
    exec_end_dt.notna() &
    run_end_dt.notna()
)
cutpoints_ok = (
    df["study_cutpoint_pre_invocation_diff_seconds"].abs().fillna(np.inf).le(1e-9) &
    df["study_cutpoint_execution_window_diff_seconds"].abs().fillna(np.inf).le(1e-9) &
    df["study_cutpoint_post_invocation_diff_seconds"].abs().fillna(np.inf).le(1e-9)
)
df.loc[have_cutpoints, "study_cutpoint_consistency_flag"] = "mismatch"
df.loc[have_cutpoints & cutpoints_ok, "study_cutpoint_consistency_flag"] = "ok"

df["study_temporal_order_flag"] = "missing"
temporal_ok = (
    (run_start_dt <= inv_start_dt) &
    (inv_start_dt <= exec_end_dt) &
    (exec_end_dt <= run_end_dt)
)
df.loc[have_cutpoints, "study_temporal_order_flag"] = "mismatch"
df.loc[have_cutpoints & temporal_ok, "study_temporal_order_flag"] = "ok"

df = df.copy()

# ----------------------------
# Signature / workload fields
# ----------------------------
df["study_effective_ref_for_stage4"] = safe_series(df, "effective_ref_for_stage4", dtype="object")
df["study_signature_inputs"] = safe_series(df, "signature_inputs", dtype="object")
df["study_sig_basis_base"] = safe_series(df, "sig_basis_base", dtype="object")
df["study_signature_hash_base"] = safe_series(df, "signature_hash_base", dtype="object")
df["study_sig_basis_full"] = safe_series(df, "sig_basis_full", dtype="object")
df["study_signature_hash_full"] = safe_series(df, "signature_hash_full", dtype="object")
df["study_signature_hash"] = safe_series(df, "signature_hash", dtype="object")
df["study_runner_os_bucket"] = safe_series(df, "runner_os_bucket", dtype="object")
df["study_runner_os_source"] = safe_series(df, "runner_os_source", dtype="object")
df["study_job_count_total"] = safe_series(df, "job_count_total")
df["study_job_count_total_bucket"] = safe_series(df, "job_count_total_bucket", dtype="object")
df["study_step_count_exec"] = safe_series(df, "step_count_exec")
df["study_step_count_exec_bucket"] = safe_series(df, "step_count_exec_bucket", dtype="object")
df["study_step_count_decl"] = safe_series(df, "step_count_decl")
df["study_step_count_decl_bucket"] = safe_series(df, "step_count_decl_bucket", dtype="object")
df["study_step_count_total_bucket"] = safe_series(df, "step_count_total_bucket", dtype="object")
df["study_step_count_source"] = safe_series(df, "step_count_source", dtype="object")
df["study_junit_cases"] = safe_series(df, "junit_cases")
df["study_junit_source"] = safe_series(df, "junit_source", dtype="object")
df["study_test_suite_size_bucket"] = safe_series(df, "test_suite_size_bucket", dtype="object")

# ----------------------------
# Robust flag
# ----------------------------
df["Robust"] = (
    bool_from_series(df["Base"]) &
    df["study_pre_invocation_selected_stage3_seconds"].notna() &
    df["study_invocation_execution_window_selected_stage3_seconds"].notna() &
    df["study_post_invocation_selected_stage3_seconds"].notna()
)

df = df.copy()

# ----------------------------
# Final column order
# ----------------------------
final_cols = [
    "full_name",
    "run_id",
    "workflow_id",
    "workflow_identifier",
    "workflow_path",
    "workflow_ref",
    "html_url",

    "run_number",
    "run_attempt",
    "status",
    "run_conclusion",
    "event",
    "trigger",

    "head_branch",
    "head_sha",

    "style",
    "styles",
    "target_style",
    "multi_style_run_flag",
    "invocation_types",
    "third_party_provider_name",
    "instru_job_count",

    "controller_style_in_scope",
    "controller_instru_job_count_gt0",
    "controller_attempt_eq_1",
    "controller_run_verdict_complete",
    "Base",
    "Robust",

    "created_at",
    "run_started_at",
    "run_updated_at",

    "study_run_duration_seconds",
    "study_run_duration_source",
    "study_layer1_time_to_instrumentation_envelope_seconds",
    "study_layer1_instrumentation_job_envelope_seconds",
    "study_layer1_post_instrumentation_tail_seconds",
    "study_layer1_model",

    # V18 Stage 2 run/style auxiliaries
    "study_instru_distinct_job_count",
    "study_instru_distinct_job_base_name_count",
    "study_instru_matrix_like_job_count",
    "study_instru_matrix_expanded_flag",
    "study_instru_parallel_jobs_flag",
    "study_instru_max_parallel_jobs",

    "study_style_distinct_job_count",
    "study_style_distinct_job_base_name_count",
    "study_style_matrix_like_job_count",
    "study_style_matrix_expanded_flag",
    "study_style_parallel_same_style_flag",
    "study_style_max_parallel_jobs",
    "study_style_repeated_same_style_flag",
    "study_style_invocation_candidate_step_count_proxy",
    "study_style_distinct_invocation_step_name_count_proxy",
    "study_style_invocation_candidate_step_names_proxy",
    "study_style_same_style_complexity_class",

    "study_pre_invocation_direct_seconds",
    "study_pre_invocation_direct_source",
    "study_invocation_execution_window_direct_seconds",
    "study_invocation_execution_window_direct_source",
    "study_post_invocation_direct_seconds",
    "study_post_invocation_direct_source",
    "study_pre_invocation_selected_stage3_seconds",
    "study_pre_invocation_selected_stage3_source",
    "study_invocation_execution_window_selected_stage3_seconds",
    "study_invocation_execution_window_selected_stage3_source",
    "study_post_invocation_selected_stage3_seconds",
    "study_post_invocation_selected_stage3_source",
    "study_layer2_measurement_mode",
    "study_layer2_measurement_quality",

    # V18 Stage 3 auxiliaries
    "study_invocation_candidate_count_total",
    "study_stage1_anchor_candidate_count",
    "study_explicit_instru_candidate_count",
    "study_custom_supported_candidate_count",
    "study_distinct_invocation_candidate_step_name_count",
    "study_distinct_invocation_candidate_job_count",
    "study_invocation_candidate_step_names",
    "study_invocation_candidate_job_names",
    "study_selected_invocation_priority_source",
    "study_execution_window_candidate_count",
    "study_execution_window_distinct_job_count",
    "study_execution_window_candidate_job_names",
    "study_cross_job_execution_window_flag",

    "study_run_boundary_start_at",
    "study_run_boundary_end_at",
    "study_matched_invocation_step_name",
    "study_matched_invocation_job_name",
    "study_matched_invocation_source",
    "study_matched_invocation_step_started_at",
    "study_matched_invocation_step_completed_at",
    "study_matched_invocation_job_ordinal_in_run",
    "study_matched_invocation_step_ordinal_in_job",
    "study_invocation_execution_end_step_name",
    "study_invocation_execution_end_job_name",
    "study_invocation_execution_end_source",
    "study_invocation_execution_end_step_started_at",
    "study_invocation_execution_end_step_completed_at",
    "study_invocation_execution_end_job_ordinal_in_run",
    "study_invocation_execution_end_step_ordinal_in_job",
    "study_invocation_execution_window_started_at",
    "study_invocation_execution_window_ended_at",

    "study_cutpoint_pre_invocation_diff_seconds",
    "study_cutpoint_execution_window_diff_seconds",
    "study_cutpoint_post_invocation_diff_seconds",
    "study_cutpoint_consistency_flag",
    "study_temporal_order_flag",

    "study_setup_sum_seconds",
    "study_provision_sum_seconds",
    "study_test_sum_seconds",
    "study_artifact_report_sum_seconds",
    "study_cleanup_teardown_sum_seconds",
    "study_other_sum_seconds",
    "study_execution_related_sum_seconds",
    "study_non_execution_overhead_sum_seconds",
    "study_pre_test_overhead_sum_seconds",
    "study_active_test_sum_seconds",
    "study_post_test_overhead_sum_seconds",
    "study_setup_step_count",
    "study_provision_step_count",
    "study_test_step_count",
    "study_artifact_report_step_count",
    "study_cleanup_teardown_step_count",
    "study_other_step_count",
    "study_execution_related_step_count",
    "study_non_execution_overhead_step_count",
    "study_pre_test_overhead_step_count",
    "study_active_test_step_count",
    "study_post_test_overhead_step_count",

    "study_effective_ref_for_stage4",
    "study_signature_inputs",
    "study_sig_basis_base",
    "study_signature_hash_base",
    "study_sig_basis_full",
    "study_signature_hash_full",
    "study_signature_hash",
    "study_runner_os_bucket",
    "study_runner_os_source",
    "study_job_count_total",
    "study_job_count_total_bucket",
    "study_step_count_exec",
    "study_step_count_exec_bucket",
    "study_step_count_decl",
    "study_step_count_decl_bucket",
    "study_step_count_total_bucket",
    "study_step_count_source",
    "study_junit_cases",
    "study_junit_source",
    "study_test_suite_size_bucket",
]

main_df = df[[c for c in final_cols if c in df.columns]].copy()
main_df = main_df.drop_duplicates(subset=["full_name", "run_id", "style"], keep="first").copy()

main_df.to_csv(OUT_MAIN, index=False)

print("\nSaved:", OUT_MAIN)
print("MainDataset shape:", main_df.shape)
print("Unique run×style rows:", main_df[["full_name", "run_id", "style"]].drop_duplicates().shape[0])
print("workflow_identifier non-null:", int(main_df["workflow_identifier"].notna().sum()) if "workflow_identifier" in main_df.columns else 0)
print("workflow_path non-null:", int(main_df["workflow_path"].notna().sum()) if "workflow_path" in main_df.columns else 0)

print("\nStyle counts:")
print(main_df["style"].value_counts(dropna=False))

print("\nBase counts:")
print(main_df["Base"].value_counts(dropna=False))

print("\nRobust counts:")
print(main_df["Robust"].value_counts(dropna=False))

print("\nEvent counts:")
if "event" in main_df.columns:
    print(main_df["event"].value_counts(dropna=False).head(20))

print("\nTrigger counts:")
if "trigger" in main_df.columns:
    print(main_df["trigger"].value_counts(dropna=False).head(20))

v18_check_cols = [
    "study_style_distinct_job_count",
    "study_style_matrix_expanded_flag",
    "study_style_repeated_same_style_flag",
    "study_invocation_candidate_count_total",
    "study_selected_invocation_priority_source",
    "study_cross_job_execution_window_flag",
]
print("\nV18 field presence:")
for c in v18_check_cols:
    print(f"{c}: {'yes' if c in main_df.columns else 'no'}")

try:
    display(main_df.head(3))
except Exception:
    print(main_df.head(3))

Reading input files...
Stage 1 shape: (1637, 23)
Stage 2 shape: (22413, 64)
Stage 3 shape: (8910, 81)
Stage 4 shape: (8906, 26)
Excluded Real-Device rows: 0
Shape after Real-Device exclusion: (8910, 184)
Duplicate rows on (full_name, run_id, style): 0

Saved: C:\Android Mobile App\ICST2026_Ext\MainDataset.csv
MainDataset shape: (8910, 146)
Unique run×style rows: 8910
workflow_identifier non-null: 8910
workflow_path non-null: 8910

Style counts:
style
Community      8046
Third-Party     548
GMD             265
Custom           51
Name: count, dtype: int64

Base counts:
Base
True     8661
False     249
Name: count, dtype: int64

Robust counts:
Robust
True     5727
False    3183
Name: count, dtype: int64

Event counts:
event
push                 7142
schedule             1491
pull_request          234
workflow_dispatch      43
Name: count, dtype: int64

Trigger counts:
trigger
NaN    8910
Name: count, dtype: int64

V18 field presence:
study_style_distinct_job_count: yes
study_style_matrix

,full_name,run_id,workflow_id,workflow_identifier,workflow_path,workflow_ref,html_url,run_number,run_attempt,status,...,study_job_count_total_bucket,study_step_count_exec,study_step_count_exec_bucket,study_step_count_decl,study_step_count_decl_bucket,study_step_count_total_bucket,study_step_count_source,study_junit_cases,study_junit_source,study_test_suite_size_bucket
0,connectbot/connectbot,23171866011,6226310,Continuous Integration,.github/workflows/ci.yml,0944e1b264e71d4345c83066223b8ab9ebf7ce4d,https://github.com/connectbot/connectbot/actio...,2707,1,completed,...,2_3,27.0,21_40,17.0,<=20,21_40,executed_dedup,NaN,none,unknown
1,connectbot/connectbot,23171849685,6226310,Continuous Integration,.github/workflows/ci.yml,bac36988c6ad1d459cb3bdfc1afd6b7754c014c5,https://github.com/connectbot/connectbot/actio...,2706,1,completed,...,2_3,27.0,21_40,17.0,<=20,21_40,executed_dedup,NaN,none,unknown
2,connectbot/connectbot,23171829938,6226310,Continuous Integration,.github/workflows/ci.yml,84463432802f13dfa2266c6086da1995a12e0e80,https://github.com/connectbot/connectbot/actio...,2705,1,completed,...,2_3,27.0,21_40,17.0,<=20,21_40,executed_dedup,NaN,none,unknown
